# DuoRank — phenotype-driven compound-heterozygote discovery for a rare-disease genome

**Rare Disease, Real Kid: MVA Hackathon 2026** (Sage Bionetworks · MVA Society · Hugging Face · BEACON) — Track 1 (variant prediction) with a Track 2 (drug repurposing) evidence module.

This notebook is a fully automated, public-data-only pipeline. Nothing about the answer is hard-coded: the ranked
submission is derived from the proband's VCF and HPO terms, and the same code runs unchanged on any other single-sample
GRCh38 VCF plus a list of HPO terms.

| § | Stage | Evidence it adds |
|---|---|---|
| 1–3 | Environment, tools (micromamba / bioconda), public reference resources | — |
| 4 | Phenotype engine: HPO information-content similarity against every annotated Mendelian disease | gene-level phenotype fit |
| 5 | Case data (gated Hugging Face dataset via the Kaggle Secret `HF_TOKEN`; public demo case without it) | — |
| 6 | Genome QC, sex, autozygosity, chromosome-level aneuploidy screen with detection limit | data quality, cytogenetic context |
| 7 | Evidence model: ACMG/AMP Bayesian points, genotype hypotheses, EPCR | — |
| 8 | Normalise + annotate: gnomAD v4.1, ClinVar, MANE consequences, NMD rule, AlphaMissense | rarity, consequence, prior assertions |
| 9 | Ranking, second-hit rescue, robustness analysis, submission CSV checked with the organisers' scorer | ranked causal hypotheses |
| 10 | Deep annotation of the short list (Ensembl VEP, exact gnomAD, PanelApp, chance-bystander model) | refined ranking |
| 11 | GPU models: SpliceAI on the leading genes, ESM-2 saturation mutagenesis, AlphaFold context | orthogonal evidence |
| 12 | Phase of the leading pair | cis / trans |
| 13 | Spike-in benchmark on a public genome | generalisation |
| 14 | Track 2: mechanism → approved-drug hypotheses, citations resolved live | repurposing evidence |
| 14b | Track 2: allele-level therapeutic triage (read-through, NMD, exon skipping, cDNA phasing, allele-resolved peptides, ClinVar lever atlas) | repurposing evidence |
| 15 | Reports (Track 1 methods write-up, Track 2 report) generated from the results | deliverables |
| 16 | Read-level evidence from the raw FASTQ (unique-minimizer recruitment, re-alignment, re-calling, depth / SV screen) | independent confirmation |
| 17 | Final reports, deletion of all scratch data, run log | — |

**How to run.** Settings → Accelerator **GPU T4 x2**, Internet **on**. Add-ons → Secrets → add `HF_TOKEN` (a Hugging Face
token of an account that has been granted access to `SageBio/mva-hackathon-2026-data`) and tick it for this notebook.
Then **Save Version → Save & Run All** (runs in the background; ~1 h for everything except the FASTQ module, which adds
~2–5 h and stops itself after `READ_LEVEL_MAX_HOURS`). Without the secret the notebook runs end-to-end in **demo mode**
on public data. Outputs (download from the Output tab): `<team>_<model>.csv` (Track 1 submission),
`<team>_track1_report.md`, `<team>_track2_report.md`, `figures/`, `run_summary.json`.

**Data handling.** The genome is controlled-access. Everything that carries genotype-scale data is written to ephemeral
scratch storage (`/kaggle/tmp`) and deleted at the end of the run; only findings (ranked candidates, figures, reports)
are written to `/kaggle/working`. Genes outside the stated indication are not named in displayed tables or reports.
Keep this notebook **private**.

In [ ]:
CFG = dict(
    # ---- identity (used in output file names; see the submission instructions) -------------------------------------
    TEAM="rusheel86",
    MODEL_NAME="duorank-v1",
    # ---- case inputs ------------------------------------------------------------------------------------------------
    HF_DATASET="SageBio/mva-hackathon-2026-data",
    VCF_FILE="WGS_EX2312012_HGWCNDSX7.vcf.gz",
    PHENOTYPE_DOCX="Challenge_Clinical_Phenotype_1.docx",
    FASTQ_PATTERN="WGS_EX2312012_HGWCNDSX7_S16_L00{lane}_R{read}_001.fastq.gz",
    FASTQ_LANES=[1, 2, 3, 4],
    PROBAND_ID="PROBAND01",
    # Clinical indication as stated by the organisers. It is used as a transparent prior on top of the HPO terms
    # (both rankings are reported). Set to "" for an undiagnosed case.
    CLINICAL_DIAGNOSIS="mosaic variegated aneuploidy",
    HPO_TERMS=[],                 # leave empty to parse them from the phenotype document
    # ---- model parameters -------------------------------------------------------------------------------------------
    MAX_AF=0.01,                  # population allele-frequency ceiling for candidate variants
    FLANK_SECOND_HIT=5000,        # bp around a gene searched for non-coding second hits
    TOP_GENES_DEEP=30,            # genes taken forward to deep annotation / second-hit rescue
    # ---- optional modules -------------------------------------------------------------------------------------------
    RUN_GPU_MODELS=True,
    RUN_READ_LEVEL=True,          # FASTQ module (the long one)
    RUN_BENCHMARK=True,
    RUN_TRACK2=True,
    READ_LEVEL_MAX_HOURS=7.0,     # the FASTQ module stops recruiting new lanes after this many hours
    INCLUDE_SECONDARY_ROWS=False, # secondary / incidental rows in the CSV (off: organisers' IRB guidance is pending)
    MASK_INCIDENTAL=True,         # withhold gene names outside the indication panel / leading hypothesis from tables and reports
    REPO_URL="https://github.com/Rusheel86/MVA",   # public after the hackathon closes, as the rules require
    # ---- disclosure (required by the methods template) -------------------------------------------------------------
    LLM_DISCLOSURE=("Anthropic Claude Code (Claude Opus 5.5, Claude Opus 5 and Claude Sonnet 5.5), Claude Pro plan, was used to design and "
                    "write this pipeline, and to draft the text that it renders into this report. Every literature claim "
                    "was checked against the cited record before it was included. The model-training opt-out was enabled "
                    "and no output ratings or feedback were submitted."),
)

## 1 · Environment, helpers and platform probe

All genotype-scale files live under an ephemeral scratch directory (`WORK`), never under `/kaggle/working`
(which Kaggle persists with the notebook). Optional stages are wrapped so that a failure is logged and the run
continues; the core path (VCF → ranked CSV) fails loudly.

In [ ]:
import collections, gzip, io, itertools, json, math, os, platform, re, shutil, subprocess, sys, textwrap, threading
import time, traceback, zipfile
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

os.environ.setdefault("TORCH_DEVICE_BACKEND_AUTOLOAD", "0")    # TPU images: keep torch from auto-loading torch_xla
import numpy as np
import pandas as pd

T0 = time.time()
ON_KAGGLE = os.path.isdir("/kaggle/working")
OUT = Path("/kaggle/working") if ON_KAGGLE else Path("./mva_out").resolve()
WORK = Path("/kaggle/tmp/mva") if ON_KAGGLE else Path("./mva_work").resolve()
for _d in (OUT, OUT / "figures", WORK, WORK / "data", WORK / "ref", WORK / "tmp", WORK / "logs", WORK / "bin", WORK / "hf"):
    _d.mkdir(parents=True, exist_ok=True)

os.environ.update({
    "TMPDIR": str(WORK / "tmp"), "HF_HOME": str(WORK / "hf"), "MAMBA_ROOT_PREFIX": str(WORK / "mamba"),
    "HF_HUB_DISABLE_PROGRESS_BARS": "1", "HF_XET_FIXED_DOWNLOAD_CONCURRENCY": "4", "HF_HUB_DOWNLOAD_TIMEOUT": "60",
    "TF_FORCE_GPU_ALLOW_GROWTH": "true",
    "TF_CPP_MIN_LOG_LEVEL": "2", "TOKENIZERS_PARALLELISM": "false", "PYTHONUNBUFFERED": "1",
})
os.chdir(WORK)          # tools that drop temp files into the cwd must not leak into the saved notebook output
NCPU = os.cpu_count() or 4

RUN = collections.OrderedDict()     # stage -> dict(status, seconds, note)
S = {}                              # shared state between stages (results that later stages/report consume)


_LOG_SEQ = itertools.count()


def log(msg):
    print(f"[{time.time() - T0:7.0f}s] {msg}", flush=True)


def sh(script, name="cmd", check=True, capture=False, env=None, timeout=None):
    """Run a bash script with errexit+pipefail. stdout/stderr go to a scratch log (not to the saved notebook
    output). Returns captured stdout when capture=True. Raises RuntimeError with the log tail on failure."""
    logf = WORK / "logs" / f"{next(_LOG_SEQ):03d}_{re.sub(r'[^A-Za-z0-9_.-]+', '_', name)[:50]}.log"
    full_env = dict(os.environ)
    full_env.update(env or {})
    with open(logf, "w") as lf:
        p = subprocess.run(["bash", "-euo", "pipefail", "-c", script], stdout=subprocess.PIPE if capture else lf,
                           stderr=lf, text=True, env=full_env, timeout=timeout)
    if p.returncode != 0 and check:
        tail = "".join(open(logf, errors="replace").readlines()[-25:])
        raise RuntimeError(f"[{name}] exit {p.returncode}\n--- script ---\n{script[:600]}\n--- log tail ---\n{tail[-2500:]}")
    return p.stdout if capture else p.returncode


class stage:
    """`with stage('name'):` times a block and records the outcome. Optional stages swallow exceptions
    (logged with a short traceback) so that one failing evidence module never costs the core deliverables."""

    def __init__(self, name, required=False):
        self.name, self.required = name, required

    def __enter__(self):
        self.t = time.time()
        log(f"▶ {self.name}")
        return self

    def __exit__(self, et, ev, tb):
        dt = time.time() - self.t
        if et is None:
            RUN[self.name] = dict(status="ok", seconds=round(dt, 1), note="")
            log(f"✔ {self.name} ({dt:.0f}s)")
            return False
        note = f"{et.__name__}: {str(ev)[:300]}"
        RUN[self.name] = dict(status="FAILED" if self.required else "skipped", seconds=round(dt, 1), note=note)
        log(f"✘ {self.name} ({dt:.0f}s) — {note}")
        if self.required:
            return False
        traceback.print_exception(et, ev, tb, limit=3)
        return True


def fetch(url, dest, min_bytes=1000, tries=5):
    """Resumable HTTPS download with retries; skips files that are already complete."""
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size >= min_bytes and not Path(str(dest) + ".part").exists():
        return dest
    part = Path(str(dest) + ".part")
    sh(f"curl -fsSL --retry {tries} --retry-delay 5 --retry-all-errors --speed-limit 500000 --speed-time 180 -C - "
       f"-o '{part}' '{url}'", name=f"fetch_{dest.name}")
    if part.stat().st_size < min_bytes:
        raise RuntimeError(f"download too small: {url} ({part.stat().st_size} bytes)")
    part.rename(dest)
    return dest


def fetch_parallel(url, dest, n=8, min_bytes=1000, budget_s=None):
    """Large-file download over n parallel byte ranges (each resumable); falls back to `fetch` without range support.
    `budget_s` caps the wall time so a throttled mirror fails fast and the caller's fallback source is used instead."""
    dest = Path(dest)
    t_end = time.time() + budget_s if budget_s else None
    if dest.exists() and dest.stat().st_size >= min_bytes:
        return dest
    hdr = sh(f"curl -sIL --retry 3 '{url}'", name=f"head_{dest.name}", capture=True, check=False) or ""
    sizes = re.findall(r"(?im)^content-length:\s*(\d+)", hdr)
    size = int(sizes[-1]) if sizes else 0
    probe = sh(f"curl -sL -r 0-0 -o /dev/null -w '%{{http_code}}' '{url}'", name=f"probe_{dest.name}", capture=True, check=False)
    if size < 200_000_000 or (probe or "").strip() != "206":
        return fetch(url, dest, min_bytes=min_bytes)
    step = -(-size // n)
    parts = [Path(f"{dest}.r{i}") for i in range(n)]

    def get(i):
        a, b = i * step, min(size, (i + 1) * step) - 1
        expect = b - a + 1
        for _ in range(8):
            have = parts[i].stat().st_size if parts[i].exists() else 0
            if have >= expect:
                break
            left = int(t_end - time.time()) if t_end else None
            if left is not None and left <= 0:
                break
            sh(f"curl -fsSL --retry 5 --retry-all-errors --speed-limit 100000 --speed-time 120 "
               + (f"--max-time {left} " if left else "") + f"-r {a + have}-{b} '{url}' >> '{parts[i]}'",
               name=f"range{i}_{dest.name}", check=False)
        if (parts[i].stat().st_size if parts[i].exists() else 0) != expect:
            raise RuntimeError(f"range {i} of {dest.name} incomplete")

    with ThreadPoolExecutor(n) as ex:
        list(ex.map(get, range(n)))
    sh(f"cat {' '.join(repr(str(p)) for p in parts)} > '{dest}.part' && mv '{dest}.part' '{dest}'", name=f"join_{dest.name}")
    for p in parts:
        p.unlink(missing_ok=True)
    if dest.stat().st_size != size:
        raise RuntimeError(f"{dest.name}: size {dest.stat().st_size} != {size}")
    return dest


def fetch_first(urls, dest, **kw):
    """Try mirrors in order."""
    last = None
    for u in urls:
        try:
            return fetch(u, dest, **kw)
        except Exception as e:      # noqa: BLE001 - try the next mirror
            last = e
            Path(str(dest) + ".part").unlink(missing_ok=True)
    raise RuntimeError(f"all mirrors failed for {Path(dest).name}: {last}")


def free_gb(p):
    return shutil.disk_usage(p).free / 1e9


def elapsed_h():
    return (time.time() - T0) / 3600.0

In [ ]:
# ---- platform probe (also what a secret-less API push prints, so the environment is on record) ---------------------
def _ver(mod):
    try:
        return __import__(mod).__version__
    except Exception as e:      # noqa: BLE001
        return f"missing ({type(e).__name__})"


def _first_line(cmd):
    try:
        out = subprocess.run(["bash", "-c", cmd], capture_output=True, text=True, timeout=60)
        return (out.stdout + out.stderr).strip().splitlines()[0][:200]
    except Exception as e:      # noqa: BLE001
        return f"n/a ({type(e).__name__})"


PROBE = {
    "platform": platform.platform(),
    "python": sys.version.split()[0],
    "kaggle_image_build": os.environ.get("BUILD_DATE", "n/a"),
    "cpus": NCPU,
    "ram_gb": round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9, 1) if hasattr(os, "sysconf") else None,
    "scratch_free_gb": round(free_gb(WORK), 1),
    "output_free_gb": round(free_gb(OUT), 1),
    "gpus": subprocess.run(["bash", "-c", "nvidia-smi -L 2>/dev/null || true"], capture_output=True, text=True).stdout.strip().splitlines(),
    "java": _first_line("java -version 2>&1"),
}
N_GPU = len(PROBE["gpus"])
for k, v in PROBE.items():
    log(f"{k}: {v}")
if ON_KAGGLE and N_GPU < 2:
    log(f"NOTE: {N_GPU} GPU(s) visible — the deep-learning modules run on the {NCPU} CPUs instead (ESM-2 in a child process; "
        "fastest on GPU T4 x2 or a large CPU host such as the TPU v5e-8 VM). The core ranking does not need a GPU.")

In [ ]:
# ---- access token: Kaggle Secret `HF_TOKEN` (Add-ons → Secrets), or an environment variable ------------------------
def get_hf_token():
    for name in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN", "HUGGINGFACE_TOKEN"):
        if os.environ.get(name):
            return os.environ[name].strip()
    try:
        from kaggle_secrets import UserSecretsClient
        client = UserSecretsClient()
    except Exception:           # noqa: BLE001 - not on Kaggle
        return None
    for name in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN", "HUGGINGFACE_TOKEN", "hf_token"):
        try:
            tok = client.get_secret(name)
            if tok:
                return tok.strip()
        except Exception:       # noqa: BLE001 - secret not attached under this label
            continue
    return None


HF_TOKEN = get_hf_token()
DEMO = HF_TOKEN is None
if DEMO:
    print("\n" + "=" * 110)
    print(" No Hugging Face token found (attach the Kaggle Secret `HF_TOKEN`: Add-ons → Secrets → tick the box).")
    print(" Running in DEMO mode on a PUBLIC genome (GIAB HG002) with a spiked-in recessive case, to verify that every")
    print(" stage of the pipeline works on this machine. No patient data is touched in DEMO mode.")
    print("=" * 110 + "\n")
else:
    os.environ["HF_TOKEN"] = HF_TOKEN
    log("HF token found — running on the gated challenge dataset.")
S["mode"] = "DEMO (public HG002 + spike-in)" if DEMO else "CHALLENGE (gated dataset)"

## 2 · Command-line tools and plotting style

One pinned micromamba environment provides the bioinformatics binaries (bcftools / samtools / htslib /
minimap2 / BBTools), with an `apt` fallback. Its `bin/` is *appended* to `PATH` so the image's own Python is
never shadowed.

In [ ]:
ENV_DIR = WORK / "env"
MICROMAMBA = WORK / "bin" / "micromamba"
BIO_PKGS = "bcftools samtools htslib minimap2 bbmap"


def _have(tool):
    return shutil.which(tool) is not None


with stage("install tools", required=True):
    if not (_have("bcftools") and _have("samtools") and _have("tabix") and _have("minimap2")):
        try:
            fetch_first(["https://github.com/mamba-org/micromamba-releases/releases/latest/download/micromamba-linux-64"],
                        MICROMAMBA, min_bytes=1_000_000)
            MICROMAMBA.chmod(0o755)
            if not (ENV_DIR / "bin" / "bcftools").exists():
                sh(f"'{MICROMAMBA}' create -y -q -p '{ENV_DIR}' --strict-channel-priority -c conda-forge -c bioconda {BIO_PKGS}",
                   name="micromamba_create", timeout=1800)
            os.environ["PATH"] = os.environ["PATH"] + os.pathsep + str(ENV_DIR / "bin")
        except Exception as e:      # noqa: BLE001
            log(f"micromamba route failed ({str(e)[:200]}); falling back to apt")
            sh("apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq bcftools samtools tabix minimap2 bbmap",
               name="apt_install", timeout=1800)
    for tool in ("bcftools", "samtools", "tabix", "bgzip", "minimap2"):
        if not _have(tool):
            raise RuntimeError(f"required tool missing after installation: {tool}")
    S["tool_versions"] = {
        "bcftools": _first_line("bcftools --version"), "samtools": _first_line("samtools --version"),
        "minimap2": _first_line("minimap2 --version"), "bbduk": _first_line("bbduk.sh --version 2>&1 | grep -i 'version' || echo missing"),
    }
    for k, v in S["tool_versions"].items():
        log(f"{k}: {v}")
    # small pure-Python dependencies (wheels exist for the Kaggle image's Python)
    sh(f"'{sys.executable}' -m pip install -q --no-input pysam pyfaidx 2>&1 | tail -2", name="pip_install", check=False, timeout=900)

In [ ]:
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

# One chart style for every figure: recessive axes, thin marks, a colour-blind-validated categorical order
# (blue, orange, aqua, yellow); status colours are reserved for status.
VIZ = dict(surface="#fcfcfb", ink="#0b0b0b", ink2="#52514e", muted="#898781", grid="#e1e0d9", axis="#c3c2b7",
           s1="#2a78d6", s2="#eb6834", s3="#1baf7a", s4="#eda100", neutral="#c3c2b7", critical="#d03b3b")
plt.rcParams.update({
    "figure.facecolor": VIZ["surface"], "axes.facecolor": VIZ["surface"], "savefig.facecolor": VIZ["surface"],
    "font.family": "DejaVu Sans", "font.size": 9, "axes.edgecolor": VIZ["axis"], "axes.labelcolor": VIZ["ink2"],
    "axes.titlesize": 10.5, "axes.titleweight": "bold", "axes.titlecolor": VIZ["ink"], "axes.titlelocation": "left",
    "xtick.color": VIZ["muted"], "ytick.color": VIZ["muted"], "xtick.labelcolor": VIZ["ink2"], "ytick.labelcolor": VIZ["ink2"],
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": VIZ["grid"],
    "grid.linewidth": 0.6, "grid.linestyle": "-", "axes.axisbelow": True, "legend.frameon": False,
    "legend.fontsize": 8.5, "lines.linewidth": 2.0, "figure.dpi": 110, "savefig.dpi": 160, "savefig.bbox": "tight",
})


def save_fig(fig, name, show=True):
    """Save a figure under OUT/figures and display it."""
    path = OUT / "figures" / f"{name}.png"
    fig.savefig(path)
    plt.close(fig)
    S.setdefault("figures", {})[name] = str(path.relative_to(OUT))
    if show:
        display(Image(filename=str(path)))
    return path


def md(text):
    display(Markdown(text))


def df_md(df, max_rows=30, floatfmt="{:.3g}"):
    """Small DataFrame -> GitHub markdown table (no external dependency)."""
    df = df.head(max_rows)
    cols = [str(c) for c in df.columns]

    def fmt(v):
        if isinstance(v, float):
            return "" if math.isnan(v) else floatfmt.format(v)
        return str(v).replace("|", "\\|").replace("\n", " ")

    lines = ["| " + " | ".join(cols) + " |", "|" + "|".join("---" for _ in cols) + "|"]
    for row in df.itertuples(index=False):
        lines.append("| " + " | ".join(fmt(v) for v in row) + " |")
    return "\n".join(lines)

## 3 · Public reference resources

Genome-wide steps run offline against downloaded public files; only a short list of top candidates is later sent
to public annotation services.

| Resource | Use | Source |
|---|---|---|
| GRCh38 no-alt analysis set | normalisation, consequence calling, read alignment (same primary assembly and coordinates as the VCF's reference) | NCBI (GCS mirror) |
| MANE Select / Plus Clinical gene models and proteins | consequences on the clinically agreed transcript | NCBI RefSeq / Ensembl |
| gnomAD v4.1 joint exomes+genomes, compact `echtvar` archive (AF, grpmax FAF95, homozygotes) | first-pass rarity, genome-wide and offline | gnomAD via Zenodo 22297079 (fallbacks: slivar gnotate v3, GATK af-only) |
| gnomAD v4.1 joint sites (remote tabix, short list only) | exact AF, filtering AF, homozygotes | gnomAD |
| ClinVar GRCh38 | prior clinical assertions, loss-of-function mechanism per gene | NCBI |
| AlphaMissense hg38 | calibrated missense predictor (PP3/BP4) | Google DeepMind (CC BY-NC-SA) |
| HPO ontology and disease annotations | phenotype engine | HPO / Monarch |

In [ ]:
REF_GCS = "https://storage.googleapis.com/deepvariant/case-study-testdata/GCA_000001405.15_GRCh38_no_alt_analysis_set.fna"
REF_NCBI = ("https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/000/001/405/GCA_000001405.15_GRCh38/"
            "seqs_for_alignment_pipelines.ucsc_ids/GCA_000001405.15_GRCh38_no_alt_analysis_set.fna")
MANE_DIR = "https://ftp.ncbi.nlm.nih.gov/refseq/MANE/MANE_human/current/"
HPO_BASE = "https://github.com/obophenotype/human-phenotype-ontology/releases/latest/download/"
GNOMAD_V4_JOINT = "https://storage.googleapis.com/gcp-public-data--gnomad/release/4.1/vcf/joint/gnomad.joint.v4.1.sites.{chrom}.vcf.bgz"
# first-pass population-frequency sources, tried in this order
AF_SOURCES = [
    dict(name="echtvar", label="gnomAD v4.1 joint exomes+genomes (echtvar archive, Zenodo 22297079): AF, grpmax FAF95, homozygotes",
         tool="https://github.com/brentp/echtvar/releases/download/v0.2.4/echtvar-v0.2.4-x86_64-unknown-linux-musl",
         data="https://zenodo.org/records/22297079/files/gnomad_4.1_whole_genome.zip?download=1", min_bytes=3_500_000_000),
    dict(name="slivar", label="gnomAD v3.0 genomes (slivar gnotate): max population AF, homozygotes",
         tool="https://github.com/brentp/slivar/releases/download/v0.3.4/slivar",
         data="https://slivar.s3.amazonaws.com/gnomad.hg38.genomes.v3.fix.zip", min_bytes=3_500_000_000),
    dict(name="afonly", label="gnomAD v2 lifted to GRCh38 (GATK af-only resource): global AF", tool=None,
         data="https://storage.googleapis.com/gatk-best-practices/somatic-hg38/af-only-gnomad.hg38.vcf.gz", min_bytes=2_500_000_000),
]
R = WORK / "ref"
RES = {}
PRIMARY = [f"chr{i}" for i in range(1, 23)] + ["chrX", "chrY"]


def _mane_files():
    listing = sh(f"curl -fsSL --retry 5 '{MANE_DIR}'", name="mane_listing", capture=True)
    out = {}
    for key, pat in (("gff", r"MANE\.GRCh38\.v[\d.]+\.ensembl_genomic\.gff\.gz"),
                     ("summary", r"MANE\.GRCh38\.v[\d.]+\.summary\.txt\.gz"),
                     ("protein", r"MANE\.GRCh38\.v[\d.]+\.ensembl_protein\.faa\.gz")):
        m = re.search(pat, listing)
        if not m:
            raise RuntimeError(f"MANE file not found in listing: {key}")
        out[key] = m.group(0)
    return out


with stage("download public resources", required=True):
    mane = _mane_files()
    S["mane_version"] = re.search(r"v[\d.]+\d", mane["gff"]).group(0)
    jobs = [
        ("ref", [REF_GCS], R / "ref.fa", 3_000_000_000),
        ("ref_fai", [REF_GCS + ".fai"], R / "ref.fa.fai", 5_000),
        ("mane_gff", [MANE_DIR + mane["gff"]], R / "mane.gff.gz", 5_000_000),
        ("mane_summary", [MANE_DIR + mane["summary"]], R / "mane.summary.txt.gz", 500_000),
        ("mane_protein", [MANE_DIR + mane["protein"]], R / "mane.protein.faa.gz", 5_000_000),
        ("clinvar", ["https://ftp.ncbi.nlm.nih.gov/pub/clinvar/vcf_GRCh38/clinvar.vcf.gz"], R / "clinvar.vcf.gz", 50_000_000),
        ("alphamissense", ["https://storage.googleapis.com/dm_alphamissense/AlphaMissense_hg38.tsv.gz",
                           "https://zenodo.org/records/10813168/files/AlphaMissense_hg38.tsv.gz?download=1"],
         R / "AlphaMissense_hg38.tsv.gz", 300_000_000),
        ("hp_obo", [HPO_BASE + "hp.obo", "http://purl.obolibrary.org/obo/hp.obo"], R / "hp.obo", 1_000_000),
        ("hpoa", [HPO_BASE + "phenotype.hpoa", "http://purl.obolibrary.org/obo/hp/hpoa/phenotype.hpoa"], R / "phenotype.hpoa", 1_000_000),
        ("genes_to_disease", [HPO_BASE + "genes_to_disease.txt", "http://purl.obolibrary.org/obo/hp/hpoa/genes_to_disease.txt"],
         R / "genes_to_disease.txt", 100_000),
        ("af_tool", [AF_SOURCES[0]["tool"]], WORK / "bin" / "echtvar", 3_000_000),
        ("af_data", [AF_SOURCES[0]["data"]], R / "af_echtvar.zip", AF_SOURCES[0]["min_bytes"]),
    ]

    def _job(j):
        key, urls, dest, min_bytes = j
        t = time.time()
        try:
            if min_bytes >= 2_000_000_000 and len(urls) == 1:        # multi-GB archives: parallel byte ranges
                # the gnomAD archive has fallbacks (AF_SOURCES), so a throttled mirror is abandoned after 15 min
                fetch_parallel(urls[0], dest, n=8, min_bytes=min_bytes, budget_s=900 if key == "af_data" else None)
            else:
                fetch_first(urls, dest, min_bytes=min_bytes)
            return key, str(dest), None, time.time() - t
        except Exception as e:      # noqa: BLE001
            return key, None, str(e)[:300], time.time() - t

    with ThreadPoolExecutor(max_workers=5) as ex:
        for key, path, err, dt in ex.map(_job, jobs):
            if err:
                log(f"  {key}: FAILED ({err})")
            else:
                RES[key] = path
                log(f"  {key}: {Path(path).stat().st_size / 1e6:,.0f} MB in {dt:.0f}s")
    for must in ("mane_gff", "mane_protein", "clinvar", "hp_obo", "hpoa", "genes_to_disease"):
        if must not in RES:
            raise RuntimeError(f"required resource missing: {must}")
    if "af_tool" in RES:
        Path(RES["af_tool"]).chmod(0o755)
    S["resource_dates"] = {"downloaded_utc": time.strftime("%Y-%m-%d %H:%M", time.gmtime())}

In [ ]:
PRIMARY_SET = set(PRIMARY)


def load_mane(gff_gz, csq_gff_path):
    """MANE gene models as Python structures, plus a minimal Ensembl-style GFF3 that `bcftools csq` understands."""
    tx, gene_tx, gene_pos = {}, collections.defaultdict(list), {}
    with gzip.open(gff_gz, "rt") as fin, open(csq_gff_path, "w") as fout:
        fout.write("##gff-version 3\n")
        for line in fin:
            if line.startswith("#"):
                continue
            c = line.rstrip("\n").split("\t")
            if c[0] not in PRIMARY_SET:         # MANE models on fix / alt patch contigs: not in the reference or the VCF
                continue
            a = dict(kv.split("=", 1) for kv in c[8].split(";") if "=" in kv)
            ftype = c[2]
            gid = a.get("gene_id", "").split(".")[0]
            tid = a.get("transcript_id", "").split(".")[0]
            if ftype == "gene":
                gene_pos[a["gene_name"]] = (c[0], int(c[3]), int(c[4]), c[6], gid)
                fout.write("\t".join(c[:8] + [f"ID=gene:{gid};biotype=protein_coding;Name={a['gene_name']}"]) + "\n")
            elif ftype == "transcript":
                tx[tid] = dict(gene=a["gene_name"], chrom=c[0], strand=c[6], start=int(c[3]), end=int(c[4]), cds=[], exons=[],
                               protein=a.get("protein_id", ""), refseq=a.get("Dbxref", "").replace("RefSeq:", ""),
                               tag=a.get("tag", ""), version=a.get("transcript_id", ""))
                gene_tx[a["gene_name"]].append(tid)
                fout.write("\t".join(c[:8] + [f"ID=transcript:{tid};Parent=gene:{gid};biotype=protein_coding"]) + "\n")
            elif ftype in ("exon", "CDS", "five_prime_UTR", "three_prime_UTR") and tid in tx:
                if ftype == "CDS":
                    tx[tid]["cds"].append((int(c[3]), int(c[4])))
                elif ftype == "exon":
                    tx[tid]["exons"].append((int(c[3]), int(c[4])))
                fout.write("\t".join(c[:8] + [f"Parent=transcript:{tid}"]) + "\n")
    for t in tx.values():       # transcription order
        rev = t["strand"] == "-"
        t["cds"].sort(reverse=rev)
        t["exons"].sort(reverse=rev)
        t["cds_len"] = sum(e - s + 1 for s, e in t["cds"])
    # prefer the MANE Select transcript when a gene also has a MANE Plus Clinical one
    for g in gene_tx:
        gene_tx[g].sort(key=lambda t: 0 if "MANE_Select" in tx[t]["tag"] else 1)
    return tx, gene_tx, gene_pos


def load_proteins(faa_gz):
    prot, name = {}, None
    with gzip.open(faa_gz, "rt") as f:
        for line in f:
            if line.startswith(">"):
                name = line[1:].split()[0]
                prot[name] = []
            elif name:
                prot[name].append(line.strip())
    return {k: "".join(v) for k, v in prot.items()}


def load_clinvar_plp(tsv):
    """P/LP (non-conflicting) ClinVar records; loss of function counts as an established disease mechanism for a
    gene when ClinVar holds >= 2 P/LP truncating alleles in it."""
    cv = pd.read_csv(tsv, sep="\t", header=None, dtype=str, keep_default_na=False, quoting=3,
                     names=["chrom", "pos", "ref", "alt", "clnsig", "clnrevstat", "geneinfo", "mc", "clndn"])
    cv = cv[~cv.clnsig.str.startswith("Conflicting")].copy()
    cv["pos"] = cv["pos"].astype(int)
    cv["gene"] = cv.geneinfo.str.split(":").str[0]
    cv["is_lof"] = cv.mc.str.contains("nonsense|frameshift|splice_donor|splice_acceptor", regex=True)
    cv["is_missense"] = cv.mc.str.contains("missense")
    n_lof = cv[cv.is_lof].groupby("gene").size()
    return cv, set(n_lof[n_lof >= 2].index)


with stage("prepare reference, gene models, ClinVar", required=True):
    # ---- reference FASTA (GRCh38 no-alt analysis set; same primary sequence and coordinates as the VCF's reference) ----
    REF = R / "ref.fa"
    if "ref" not in RES or "ref_fai" not in RES:        # fall back to the NCBI copy (plain gzip)
        gz = fetch(REF_NCBI + ".gz", R / "ref.fa.gz", min_bytes=500_000_000)
        sh(f"(pigz -dc '{gz}' || gzip -dc '{gz}') > '{REF}' && samtools faidx '{REF}'", name="ref_unpack")
        Path(gz).unlink(missing_ok=True)
    RES["ref"] = str(REF)
    CHROM_LEN = {l.split("\t")[0]: int(l.split("\t")[1]) for l in open(str(REF) + ".fai")}
    assert all(c in CHROM_LEN for c in PRIMARY), "reference is missing primary chromosomes"

    # ---- MANE ----------------------------------------------------------------------------------------------------------
    RES["csq_gff"] = str(R / "mane.csq.gff3")
    TX, GENE_TX, GENE_POS = load_mane(RES["mane_gff"], RES["csq_gff"])
    PROT = load_proteins(RES["mane_protein"])
    log(f"MANE {S['mane_version']}: {len(GENE_POS):,} genes, {len(TX):,} transcripts, {len(PROT):,} proteins")

    # ---- ClinVar: chr-prefixed, primary chromosomes only, with contig lines; plus a P/LP table ----------------------
    cv = R / "clinvar.chr.vcf.gz"
    if not Path(str(cv) + ".tbi").exists():
        keep_chr = {c[3:] for c in PRIMARY}
        plain = R / "clinvar.chr.vcf"
        with gzip.open(RES["clinvar"], "rt") as fin, open(plain, "w") as fout:
            for line in fin:
                if line.startswith("##"):
                    fout.write(line)
                elif line.startswith("#CHROM"):
                    fout.write("".join(f"##contig=<ID={c},length={CHROM_LEN[c]}>\n" for c in PRIMARY))
                    fout.write(line)
                else:
                    c, rest = line.split("\t", 1)
                    if c in keep_chr:
                        fout.write("chr" + c + "\t" + rest)
        sh(f"bgzip -f -@ {NCPU} '{plain}' && tabix -f -p vcf '{cv}'", name="clinvar_chr")        # writes clinvar.chr.vcf.gz
    RES["clinvar_chr"] = str(cv)
    S["clinvar_date"] = (sh(f"(gzip -dc '{RES['clinvar']}' | head -40 | grep -m1 '^##fileDate') || true", name="clinvar_date",
                            capture=True).strip().replace("##fileDate=", "") or "n/a")
    plp_tsv = R / "clinvar.plp.tsv"
    sh(f"bcftools query -i 'INFO/CLNSIG~\"athogenic\"' -f '%CHROM\t%POS\t%REF\t%ALT\t%INFO/CLNSIG\t%INFO/CLNREVSTAT\t%INFO/GENEINFO\t%INFO/MC\t%INFO/CLNDN\n' '{cv}' > '{plp_tsv}'",
       name="clinvar_plp")
    CLINVAR_PLP, LOF_GENES = load_clinvar_plp(plp_tsv)
    log(f"ClinVar {S['clinvar_date']}: {len(CLINVAR_PLP):,} P/LP records; LoF mechanism established in {len(LOF_GENES):,} genes")

    # ---- AlphaMissense: the public file is BGZF, so it only needs a tabix index --------------------------------------
    if "alphamissense" in RES and not Path(RES["alphamissense"] + ".tbi").exists():
        try:
            sh(f"tabix -f -s 1 -b 2 -e 2 -c '#' '{RES['alphamissense']}'", name="am_index")
        except Exception as e:      # noqa: BLE001
            log(f"AlphaMissense index failed ({str(e)[:120]}); missense scores will come from the short-list annotation only")
            RES.pop("alphamissense", None)

## 4 · Phenotype engine

Every Mendelian disease in the HPO annotation corpus is scored against the proband's terms with an
information-content (Resnik) best-match similarity; diseases roll up to genes. Two rankings are kept:

* **HPO-only** — blind to the diagnosis; this is what would be available for an undiagnosed child.
* **HPO + stated clinical indication** — genes whose disease name matches `CFG["CLINICAL_DIAGNOSIS"]` form a
  virtual panel (the way a clinical laboratory applies a panel for the referral indication).

Terms that describe relatives are kept but down-weighted.

In [ ]:
class HPO:
    def __init__(self, obo_path):
        self.name, self.parents, self.alt, self.version = {}, collections.defaultdict(set), {}, "n/a"
        cur = None
        with open(obo_path, encoding="utf-8") as f:
            for line in f:
                line = line.rstrip("\n")
                if line.startswith("data-version:"):
                    self.version = line.split(":", 1)[1].strip()
                    continue
                if line == "[Term]":
                    cur = {}
                    continue
                if line.startswith("["):
                    cur = None
                    continue
                if cur is None:
                    continue
                if line.startswith("id: "):
                    cur["id"] = line[4:]
                elif line.startswith("name: "):
                    self.name[cur["id"]] = line[6:]
                elif line.startswith("is_a: "):
                    self.parents[cur["id"]].add(line[6:].split(" ! ")[0].strip())
                elif line.startswith("alt_id: "):
                    self.alt[line[8:].strip()] = cur["id"]
                elif line.startswith("replaced_by: "):
                    self.alt[cur["id"]] = line[13:].strip()
        self._anc = {}

    def canon(self, t):
        return self.alt.get(t, t)

    def ancestors(self, t):
        t = self.canon(t)
        if t in self._anc:
            return self._anc[t]
        out = {t}
        for p in self.parents.get(t, ()):
            out |= self.ancestors(p)
        self._anc[t] = out
        return out


INHERITANCE_TERMS = {
    "HP:0000007": "AR", "HP:0000006": "AD", "HP:0001417": "XL", "HP:0001419": "XLR", "HP:0001423": "XLD",
    "HP:0001427": "MT", "HP:0001450": "YL", "HP:0012275": "AD", "HP:0012274": "AD", "HP:0001466": "AD",
    "HP:0025352": "AD",
}


class PhenoEngine:
    def __init__(self, obo, hpoa, genes_to_disease):
        self.hpo = HPO(obo)
        self.d_name, self.d_terms, self.d_inh = {}, collections.defaultdict(set), collections.defaultdict(set)
        with open(hpoa, encoding="utf-8") as f:
            ix = None
            for line in f:
                if line.startswith("#"):
                    continue
                c = line.rstrip("\n").split("\t")
                if ix is None:
                    ix = {k: i for i, k in enumerate(c)}
                    continue
                did, hid, aspect = c[ix["database_id"]], self.hpo.canon(c[ix["hpo_id"]]), c[ix["aspect"]]
                self.d_name[did] = c[ix["disease_name"]]
                if c[ix["qualifier"]].upper() == "NOT":
                    continue
                if aspect == "P":
                    self.d_terms[did].add(hid)
                elif aspect == "I" and hid in INHERITANCE_TERMS:
                    self.d_inh[did].add(INHERITANCE_TERMS[hid])
        self.g2d, self.d2g = collections.defaultdict(set), collections.defaultdict(set)
        with open(genes_to_disease, encoding="utf-8") as f:
            ix = {k: i for i, k in enumerate(f.readline().rstrip("\n").split("\t"))}
            for line in f:
                c = line.rstrip("\n").split("\t")
                if c[ix["association_type"]] == "POLYGENIC":       # susceptibility loci are not Mendelian explanations
                    continue
                self.g2d[c[ix["gene_symbol"]]].add(c[ix["disease_id"]])
                self.d2g[c[ix["disease_id"]]].add(c[ix["gene_symbol"]])
        n = len(self.d_terms)
        cnt = collections.Counter()
        self.d_closure = {}
        for did, ts in self.d_terms.items():
            clo = set()
            for t in ts:
                clo |= self.hpo.ancestors(t)
            self.d_closure[did] = clo
            cnt.update(clo)
        self.ic = {t: -math.log(k / n) for t, k in cnt.items()}
        self.n_diseases = n

    def term_ic(self, q):
        q = self.hpo.canon(q)
        if q in self.ic:
            return self.ic[q]
        return max((self.ic.get(a, 0.0) for a in self.hpo.ancestors(q)), default=0.0)

    def _best(self, q, closure):
        best = 0.0
        for a in self.hpo.ancestors(q):
            if a in closure:
                v = self.ic.get(a, 0.0)
                if v > best:
                    best = v
        return best

    def score_diseases(self, terms, weights=None):
        """score = 0.8 * (share of the patient's information content recovered by the disease)
                 + 0.2 * (share of the disease's information content present in the patient)."""
        terms = [self.hpo.canon(t) for t in terms]
        w = {self.hpo.canon(k): v for k, v in (weights or {}).items()}
        denom = sum(w.get(q, 1.0) * self.term_ic(q) for q in terms) or 1.0
        q_closure = set()
        for q in terms:
            q_closure |= self.hpo.ancestors(q)
        out = {}
        for did, clo in self.d_closure.items():
            fwd = sum(w.get(q, 1.0) * self._best(q, clo) for q in terms) / denom
            if fwd <= 0:
                continue
            rev_den = sum(self.ic.get(d, 0.0) for d in self.d_terms[did])
            rev = sum(self._best(d, q_closure) for d in self.d_terms[did]) / rev_den if rev_den else 0.0
            out[did] = (0.8 * fwd + 0.2 * rev, fwd, rev)
        return out

    def score_genes(self, terms, weights=None, diagnosis=None):
        ds = self.score_diseases(terms, weights)
        ranked = sorted(ds, key=lambda d: -ds[d][0])
        pct = {did: 1.0 - i / max(len(ranked) - 1, 1) for i, did in enumerate(ranked)}
        pat = (diagnosis or "").strip().lower()
        genes = {}
        for g, dids in self.g2d.items():
            scored = sorted((d for d in dids if d in ds), key=lambda d: -ds[d][0])
            dx = [d for d in dids if pat and pat in self.d_name.get(d, "").lower()]
            best = scored[0] if scored else (dx[0] if dx else None)
            inh_all = set()
            for d in dids:
                inh_all |= self.d_inh.get(d, set())
            # mode of inheritance: of the best-matching disease (or of the diagnosis-matched disease); gene-level union as fallback
            inh = set()
            for d in (dx or scored[:1]):
                inh |= self.d_inh.get(d, set())
            genes[g] = dict(pheno=ds[best][0] if best in ds else 0.0, pct=pct.get(best, 0.0), disease=best,
                            disease_name=self.d_name.get(best), inheritance=sorted(inh or inh_all),
                            inheritance_all=sorted(inh_all), dx_match=bool(dx),
                            dx_disease=self.d_name.get(dx[0]) if dx else None)
        return genes, ds


with stage("phenotype engine (load HPO)", required=True):
    PE = PhenoEngine(RES["hp_obo"], RES["hpoa"], RES["genes_to_disease"])
    S["hpo_version"] = PE.hpo.version
    log(f"HPO release {PE.hpo.version}: {PE.n_diseases:,} diseases with phenotype annotations, {len(PE.g2d):,} genes")

## 5 · Case data

* **Challenge mode** – the proband VCF and phenotype document come from the gated Hugging Face dataset; HPO terms
  are parsed from the document's table. Terms that describe relatives are listed
  but excluded from the proband similarity.
* **Demo mode** (no token) – the public GIAB HG002 genome plus a spiked-in recessive case built from ClinVar, so
  that every stage can be exercised without patient data. The same spike-in generator powers the benchmark in
  section 13.

In [ ]:
HG002_VCF = ("https://ftp-trace.ncbi.nlm.nih.gov/ReferenceSamples/giab/release/AshkenazimTrio/HG002_NA24385_son/"
             "NISTv4.2.1/GRCh38/HG002_GRCh38_1_22_v4.2.1_benchmark.vcf.gz")
RELATIVE_RE = re.compile(r"\b(parent|parental|maternal|paternal|mother|father|sibling|family history|relative)", re.I)


def parse_phenotype_docx(path):
    """HPO ids from the phenotype document (table rows first, any 'HP:nnnnnnn' in the text as fallback)."""
    xml = zipfile.ZipFile(path).read("word/document.xml").decode("utf-8", "replace")

    def text_of(fragment):
        paras = re.findall(r"<w:p[ >].*?</w:p>", fragment, flags=re.S)
        return " ".join("".join(re.findall(r"<w:t(?: [^>]*)?>([^<]*)</w:t>", p)) for p in paras).strip()

    out, seen = [], set()
    for row in re.findall(r"<w:tr[ >].*?</w:tr>", xml, flags=re.S):
        text = " | ".join(text_of(c) for c in re.findall(r"<w:tc[ >].*?</w:tc>", row, flags=re.S))
        for hp in re.findall(r"HP:\d{7}", text):
            if hp not in seen:
                seen.add(hp)
                rel = bool(RELATIVE_RE.search(text))
                out.append(dict(term=hp, relative=rel, weight=0.0 if rel else 1.0))
    if not out:
        for hp in re.findall(r"HP:\d{7}", text_of(xml)):
            if hp not in seen:
                seen.add(hp)
                out.append(dict(term=hp, relative=False, weight=1.0))
    return out


def hf_get(filename):
    from huggingface_hub import hf_hub_download
    try:
        return hf_hub_download(repo_id=CFG["HF_DATASET"], filename=filename, repo_type="dataset",
                               local_dir=str(WORK / "data"), token=HF_TOKEN)
    except Exception as e:      # noqa: BLE001
        raise RuntimeError(f"could not download '{filename}' from {CFG['HF_DATASET']} — check that the token belongs to an "
                           f"account that has been granted access to the gated dataset ({type(e).__name__}: {str(e)[:200]})") from e


def make_spike_cases(n, seed=7):
    """Synthetic recessive cases for the demo/benchmark: one ClinVar P/LP truncating allele + one ClinVar P/LP
    missense allele in an autosomal-recessive single-gene disease, with HPO terms sampled from that disease
    (imprecision and one unrelated noise term added, as in published phenotype-simulation benchmarks)."""
    rng = np.random.default_rng(seed)
    rs = CLINVAR_PLP.clnrevstat
    cv = CLINVAR_PLP[rs.str.contains("criteria_provided|expert_panel|practice_guideline") & ~rs.str.contains("no_assertion")
                     & CLINVAR_PLP.ref.str.fullmatch(r"[ACGT]{1,30}") & CLINVAR_PLP.alt.str.fullmatch(r"[ACGT]{1,30}")
                     & CLINVAR_PLP.chrom.isin(PRIMARY[:22])]
    lof = cv[cv.is_lof & ~cv.is_missense].groupby("gene")
    mis = cv[cv.is_missense & ~cv.is_lof & (cv.ref.str.len() == 1) & (cv.alt.str.len() == 1)].groupby("gene")
    lof_genes, mis_genes = set(lof.groups), set(mis.groups)
    eligible = []
    for did, terms in PE.d_terms.items():
        genes = PE.d2g.get(did, set())
        if len(genes) != 1 or PE.d_inh.get(did) != {"AR"} or len(terms) < 8:
            continue
        g = next(iter(genes))
        if g in GENE_POS and g in lof_genes and g in mis_genes and GENE_POS[g][0] in PRIMARY[:22]:
            eligible.append((did, g))
    eligible.sort()
    all_terms = sorted(PE.ic)
    picks = rng.choice(len(eligible), size=min(n, len(eligible)), replace=False)
    cases, used_genes = [], set()
    for ci in picks:
        did, g = eligible[int(ci)]
        if g in used_genes:
            continue
        used_genes.add(g)
        a = lof.get_group(g).iloc[int(rng.integers(len(lof.get_group(g))))]
        b = mis.get_group(g).iloc[int(rng.integers(len(mis.get_group(g))))]
        terms = sorted(PE.d_terms[did])
        ic = np.array([PE.ic.get(t, 0.0) for t in terms]) + 0.1
        chosen = list(rng.choice(terms, size=min(6, len(terms)), replace=False, p=ic / ic.sum()))
        noisy = []
        for t in chosen:
            parents = sorted(PE.hpo.parents.get(t, ()))
            if parents and rng.random() < 0.3 and PE.ic.get(parents[0], 0) > 1.0:
                noisy.append(parents[0])        # imprecise term
            else:
                noisy.append(t)
        noisy.append(all_terms[int(rng.integers(len(all_terms)))])      # unrelated noise term
        noisy = list(dict.fromkeys(noisy))
        cases.append(dict(case=len(cases), gene=g, disease=did, disease_name=PE.d_name[did], hpo=noisy,
                          variants=[(a.chrom, int(a.pos), a.ref, a.alt), (b.chrom, int(b.pos), b.ref, b.alt)]))
    return cases


with stage("case data", required=True):
    if not DEMO:
        vcf = hf_get(CFG["VCF_FILE"])
        # own index: the supplied .tbi (written by htsjdk) has no per-contig record counts, which bcftools index -s/-n need
        sh(f"bcftools index -f -t --threads {NCPU} '{vcf}'", name="index_vcf")
        if CFG["HPO_TERMS"]:
            S["hpo"] = [dict(term=t, relative=False, weight=1.0) for t in CFG["HPO_TERMS"]]
        else:
            S["hpo"] = parse_phenotype_docx(hf_get(CFG["PHENOTYPE_DOCX"]))
        if not S["hpo"]:
            raise RuntimeError("no HPO terms found — set CFG['HPO_TERMS']")
        S["diagnosis"] = CFG["CLINICAL_DIAGNOSIS"].strip()
        S["vcf_raw"], S["case_label"] = vcf, CFG["PROBAND_ID"]
        SPIKE_CASES = []
    else:
        vcf = fetch(HG002_VCF, WORK / "data" / "HG002.vcf.gz", min_bytes=100_000_000)
        fetch(HG002_VCF + ".tbi", WORK / "data" / "HG002.vcf.gz.tbi", min_bytes=100_000)
        SPIKE_CASES = make_spike_cases(60)
        demo = SPIKE_CASES[0]
        S["hpo"] = [dict(term=t, relative=False, weight=1.0) for t in demo["hpo"]]
        S["diagnosis"] = demo["disease_name"]
        S["vcf_raw"], S["case_label"] = str(vcf), "PROBAND01"
        S["demo_truth"] = dict(gene=demo["gene"], disease=demo["disease_name"],
                               variants=[f"{c}:{p} {r}>{a}" for c, p, r, a in demo["variants"]])
        log(f"DEMO case: {demo['disease_name']} ({demo['gene']}) spiked into HG002; {len(SPIKE_CASES)} benchmark cases prepared")
    for h in S["hpo"]:
        h["label"] = PE.hpo.name.get(PE.hpo.canon(h["term"]), "?")
        h["ic"] = round(PE.term_ic(h["term"]), 2)
    md("**HPO terms used**\n\n" + df_md(pd.DataFrame(S["hpo"])[["term", "label", "relative", "weight", "ic"]]))
    log(f"clinical indication: {S['diagnosis'] or '(none — undiagnosed mode)'}")

## 6 · Genome QC, sex, autozygosity and a chromosome-level aneuploidy screen

The disorder in question is defined cytogenetically (mosaic aneuploidies that differ from cell to cell), so before
ranking variants we ask what the bulk genome itself can and cannot show:

* **QC** – Ti/Tv, het/hom ratio, depth: is this a clean germline call set?
* **Sex and autozygosity** – X heterozygosity / depth, and long runs of homozygosity (parental relatedness changes the
  prior between homozygous and compound-heterozygous models).
* **Aneuploidy screen** – per-chromosome depth ratio and B-allele-frequency dispersion at heterozygous SNVs. A *clonal*
  gain or loss shifts both; *variegated* aneuploidy (a different chromosome in every cell) averages out in bulk DNA and
  is expected to be invisible. The screen therefore reports a detection limit rather than a diagnosis.

In [ ]:
def load_site_table(vcf_path):
    """Compact per-site numeric table for QC (biallelic records only). Stays on scratch."""
    tsv = WORK / "tmp" / "qc_sites.tsv"
    awk = r'''BEGIN{OFS="\t"} {
        n=split($7,a,","); gt=$6; gsub(/\|/,"/",gt);
        z=(gt=="0/1"||gt=="1/0")?1:((gt=="1/1")?2:0);
        snv=(length($3)==1&&length($4)==1)?1:0; ti=0;
        if(snv){p=$3 $4; if(p=="AG"||p=="GA"||p=="CT"||p=="TC") ti=1}
        print $1,$2,snv,ti,z,a[1]+0,a[2]+0,$8+0,$9+0,(($5=="PASS"||$5==".")?1:0),((snv&&($3=="G"||$3=="C"))?1:0) }'''
    sh(f"bcftools query -i 'N_ALT=1' -f '%CHROM\\t%POS\\t%REF\\t%ALT\\t%FILTER[\\t%GT\\t%AD\\t%DP\\t%GQ]\\n' '{vcf_path}' "
       f"| awk -F'\\t' '{awk}' > '{tsv}'", name="qc_query")
    t = pd.read_csv(tsv, sep="\t", header=None,
                    names=["chrom", "pos", "snv", "ti", "zyg", "ad_ref", "ad_alt", "dp", "gq", "is_pass", "gc"],
                    dtype={"chrom": "category", "pos": np.int32, "snv": np.int8, "ti": np.int8, "zyg": np.int8,
                           "ad_ref": np.int32, "ad_alt": np.int32, "dp": np.int32, "gq": np.int32, "is_pass": np.int8,
                           "gc": np.int8})
    tsv.unlink(missing_ok=True)
    t["chrom"] = t["chrom"].astype(str).str.replace("^chr", "", regex=True)
    return t


AUTOSOMES = [str(i) for i in range(1, 23)]
PAR_X = [(10_001, 2_781_479), (155_701_383, 156_030_895)]        # GRCh38 pseudo-autosomal regions


def qc_summary(t):
    q = {}
    p = t[t.is_pass == 1]
    q["biallelic_records"] = int(len(t))
    q["pass_fraction"] = round(float(len(p) / max(len(t), 1)), 4)
    snv = p[p.snv == 1]
    q["pass_snvs"], q["pass_indels"] = int(len(snv)), int(len(p) - len(snv))
    q["titv"] = round(float(snv.ti.sum() / max((snv.ti == 0).sum(), 1)), 3)
    auto = snv[snv.chrom.isin(AUTOSOMES)]
    q["het_hom_ratio"] = round(float((auto.zyg == 1).sum() / max((auto.zyg == 2).sum(), 1)), 3)
    hq = auto[(auto.zyg == 1) & (auto.gq >= 60)]
    q["median_depth_het_sites"] = float((hq.ad_ref + hq.ad_alt).median()) if len(hq) else float("nan")
    # sex from chrX: heterozygosity outside the PARs and depth relative to autosomes
    x = snv[(snv.chrom == "X") & (snv.gq >= 60)]
    x = x[~(((x.pos >= PAR_X[0][0]) & (x.pos <= PAR_X[0][1])) | ((x.pos >= PAR_X[1][0]) & (x.pos <= PAR_X[1][1])))]
    if len(x) > 500:
        q["x_het_fraction"] = round(float((x.zyg == 1).mean()), 4)
        xd = float((x[x.zyg == 2].ad_alt).median())
        ad = float((auto[auto.zyg == 2].ad_alt).median())
        q["x_to_autosome_depth"] = round(xd / ad, 3) if ad else float("nan")
        q["inferred_sex"] = "male (XY)" if (q["x_het_fraction"] < 0.15 and q["x_to_autosome_depth"] < 0.75) else "female (XX)"
    else:
        q["inferred_sex"] = "not determined (no chrX records)"
    return q


def runs_of_homozygosity(t, min_len=1_000_000, min_hom=100):
    """Het-gap method: a stretch >= min_len between consecutive confident het SNVs that still contains >= min_hom
    hom-alt SNVs is called a run of homozygosity (centromeres/gaps have no variants at all and are not counted)."""
    rows = []
    g = t[(t.is_pass == 1) & (t.snv == 1) & (t.gq >= 30) & t.chrom.isin(AUTOSOMES)]
    for ch, sub in g.groupby("chrom", observed=True):
        het = np.sort(sub.pos[sub.zyg == 1].to_numpy())
        hom = np.sort(sub.pos[sub.zyg == 2].to_numpy())
        if len(het) < 2:
            continue
        gaps = np.diff(het)
        for i in np.where(gaps >= min_len)[0]:
            a, b = het[i], het[i + 1]
            n_hom = int(np.searchsorted(hom, b) - np.searchsorted(hom, a))
            if n_hom >= min_hom and n_hom / ((b - a) / 1e6) >= 50:
                rows.append(dict(chrom=ch, start=int(a), end=int(b), mb=round((b - a) / 1e6, 2), n_hom=n_hom))
    roh = pd.DataFrame(rows, columns=["chrom", "start", "end", "mb", "n_hom"])
    return roh, float(roh.mb.sum() / 2875.0) if len(roh) else 0.0        # ~2,875 Mb of autosome


def _trimmed_mean(x):
    lo, hi = x.quantile(0.05), x.quantile(0.95)
    return float(x[(x >= lo) & (x <= hi)].mean())


def aneuploidy_screen(t, bin_mb=5, min_sites=300):
    """Depth and allele balance at confident heterozygous SNVs, per 5-Mb bin and per autosome.
    Depth is corrected for base composition with a within-sample proxy (GC fraction of the reference bases at
    SNV sites in the bin); a chromosome is flagged only if depth AND allele-balance dispersion deviate together."""
    p = t[(t.is_pass == 1) & (t.snv == 1) & t.chrom.isin(AUTOSOMES)].copy()
    p["bin"] = (p.pos // (bin_mb * 1_000_000)).astype(int)
    gc = p.groupby(["chrom", "bin"], observed=True)["gc"].mean().rename("gc_proxy")
    h = p[(p.zyg == 1) & (p.gq >= 60) & ((p.ad_ref + p.ad_alt) >= 15)].copy()
    h["depth"] = h.ad_ref + h.ad_alt
    h["ab"] = h.ad_alt / h.depth
    bins = h.groupby(["chrom", "bin"], observed=True).agg(
        n=("ab", "size"), depth=("depth", _trimmed_mean), baf_dev=("ab", lambda s: float(np.mean(np.abs(s - 0.5)))),
        skew_frac=("ab", lambda s: float(((s < 0.35) | (s > 0.65)).mean()))).join(gc).reset_index()
    bins = bins[bins["n"] >= min_sites].copy()
    bins["raw_ratio"] = bins["depth"] / float(bins["depth"].median())
    # collapsed-repeat artefact bins: excess of strongly skewed "hets" and/or inflated depth
    bins["artefact"] = (bins["skew_frac"] > 0.15) | (bins["raw_ratio"] > 1.3)
    ok = ~bins["artefact"]
    coef = np.polyfit(bins.loc[ok, "gc_proxy"], bins.loc[ok, "raw_ratio"], 2) if ok.sum() >= 5 else np.array([0.0, 0.0, 1.0])
    bins["ratio"] = bins["raw_ratio"] - np.polyval(coef, bins["gc_proxy"]) + 1.0
    order = {c: i for i, c in enumerate(AUTOSOMES)}
    bins["ord"] = bins["chrom"].map(order)
    bins = bins.sort_values(["ord", "bin"]).reset_index(drop=True)
    ok = ~bins["artefact"]
    good = bins[ok]
    chrom = good.groupby("chrom", observed=True).agg(bins=("ratio", "size"), depth_ratio=("ratio", "median"),
                                                     raw_depth_ratio=("raw_ratio", "median"),
                                                     baf_dev=("baf_dev", "median")).reset_index()
    chrom["ord"] = chrom["chrom"].map(order)
    chrom = chrom.sort_values("ord").drop(columns="ord").reset_index(drop=True)
    sd_ratio = max(float(chrom["depth_ratio"].std()), 0.01)        # includes residual systematic chromosome effects
    sd_baf = max(float(chrom["baf_dev"].std()), 0.0006)
    chrom["z_depth"] = (chrom["depth_ratio"] - chrom["depth_ratio"].median()) / sd_ratio
    chrom["z_baf"] = (chrom["baf_dev"] - chrom["baf_dev"].median()) / sd_baf
    chrom["flag"] = (chrom["z_depth"].abs() > 4) & (chrom["z_baf"] > 4)
    # a clonal trisomy (monosomy) in a fraction f of cells changes the depth ratio by +f/2 (-f/2)
    limit = dict(sd_depth_ratio=round(sd_ratio, 4), sd_baf_dev=round(sd_baf, 5),
                 gc_correlation=round(float(np.corrcoef(bins.loc[ok, "gc_proxy"], bins.loc[ok, "raw_ratio"])[0, 1]), 3),
                 min_detectable_clone_fraction=round(2 * 4 * sd_ratio, 3))
    return bins, chrom, limit


with stage("QC / sex / autozygosity / aneuploidy screen"):
    sites = load_site_table(S["vcf_raw"])
    S["qc"] = qc_summary(sites)
    roh, f_roh = runs_of_homozygosity(sites)
    S["qc"]["roh_segments_ge_1Mb"] = int(len(roh))
    S["qc"]["f_roh"] = round(f_roh, 4)
    S["qc"]["autozygosity_note"] = ("elevated — parental relatedness likely; homozygous models up-weighted" if f_roh > 0.02
                                    else "no evidence of parental relatedness")
    bins, chrom_tab, limit = aneuploidy_screen(sites)
    S["aneuploidy"] = dict(limit=limit, flagged=chrom_tab[chrom_tab.flag].chrom.tolist(),
                           table=chrom_tab.round(4).to_dict("records"),
                           artefact_bins=int(bins.artefact.sum()), bins=int(len(bins)))
    del sites
    for k, v in S["qc"].items():
        log(f"QC {k}: {v}")
    log(f"aneuploidy screen: flagged chromosomes = {S['aneuploidy']['flagged'] or 'none'}; detection limit = {limit}")

    # figure: two small multiples sharing the genome axis (one measure per panel, never a dual axis)
    off, ticks, x0 = {}, [], 0.0
    for c in AUTOSOMES:
        sub = bins[bins.chrom == c]
        if not len(sub):
            continue
        off[c] = x0
        width = float(sub.bin.max() + 1)
        ticks.append((x0 + width / 2, c))
        x0 += width + 1
    bins["x"] = bins.apply(lambda r: off[r.chrom] + r.bin + 0.5, axis=1)
    fig, axes = plt.subplots(2, 1, figsize=(11, 5.2), sharex=True, gridspec_kw=dict(hspace=0.32))
    for ax, col, title, base in ((axes[0], "ratio", "Depth ratio per 5-Mb bin (heterozygous SNV sites, GC-corrected; 1.0 = autosomal median)", 1.0),
                                 (axes[1], "baf_dev", "Allele-balance dispersion per 5-Mb bin (mean |alt fraction − 0.5|)", None)):
        ok, bad = bins[~bins.artefact], bins[bins.artefact]
        ax.scatter(ok.x, ok[col], s=9, color=VIZ["s1"], linewidths=0, label="bins analysed")
        ax.scatter(bad.x, bad[col].clip(upper=ok[col].max() * 1.6), s=9, color=VIZ["neutral"], linewidths=0,
                   label="excluded: collapsed-repeat artefact (clipped)")
        if base is not None:
            ax.axhline(base, color=VIZ["axis"], linewidth=0.8)
        for c, o in off.items():
            ax.axvline(o - 0.5, color=VIZ["grid"], linewidth=0.6)
        ax.set_title(title)
        ax.grid(axis="x", visible=False)
    axes[0].legend(loc="upper left", ncol=2, handletextpad=0.2, columnspacing=1.2)
    axes[1].set_xticks([t for t, _ in ticks])
    axes[1].set_xticklabels([c for _, c in ticks])
    axes[1].set_xlabel("chromosome")
    save_fig(fig, "aneuploidy_screen")
    md("**Chromosome-level table (artefact bins excluded)**\n\n" + df_md(chrom_tab.round(4), max_rows=25))

## 7 · Evidence model and genotype hypotheses

**Variant level.** Each rare variant collects ACMG/AMP evidence in Bayesian points (Tavtigian et al. 2018, 2020:
supporting = 1, moderate = 2, strong = 4, very strong = 8; prior 0.10; odds of pathogenicity 350 for very strong), so
`P = O·0.1 / ((O−1)·0.1 + 1)` with `O = 350^(points/8)`. All thresholds are taken from the literature and were
fixed before the pipeline was run; none is specific to a gene or a case.

| Evidence | Automated rule | Points |
|---|---|---|
| PVS1 | stop-gained / frameshift / canonical ±1,2 splice on the MANE transcript, in a gene with an established loss-of-function mechanism; nonsense-mediated decay predicted (Abou Tayoun 2018) | +8; NMD-escaping: +4 if > 10 % of the protein is lost, else +2; mechanism not established: +2 |
| PM2 | absent from gnomAD, or filtering AF ≤ 1e-4 with no homozygotes (ClinGen SVI PM2 v1.0: supporting) | +1 |
| BS1 / BA1 / BS2 | AF > 0.1 % (possible hypomorph: −1), ≥ 1 % (−4), ≥ 5 % (−8); ≥ 2 homozygotes in gnomAD (−4). Waived for ClinVar P/LP alleles below BA1 | −1 … −8 |
| PP3 / BP4 missense | AlphaMissense in ClinGen-SVI calibrated bands (Pejaver 2022; Bergquist 2025): ≥ 0.990 / ≥ 0.972 / ≥ 0.906 / ≥ 0.792; ≤ 0.169 / ≤ 0.099 / ≤ 0.070. REVEL bands are used only when AlphaMissense is missing | +4 / +3 / +2 / +1; −1 / −2 / −3 |
| PP3 / BP4 splice | SpliceAI max delta ≥ 0.2 (Walker 2023; graded ≥ 0.5, ≥ 0.8); ≤ 0.1 | +1 / +2 / +4; −1 (synonymous: −2, BP7) |
| PM4 | in-frame indel, stop-loss | +2 |
| ClinVar | used as a documented prior, never as a filter: ≥ 2-star Pathogenic sets P ≥ 0.99, P/LP ≥ 0.95, LP ≥ 0.90; 1-star P/LP +2; benign assertions mirrored | floor / cap |
| PM3 | applied once, to the weaker allele of a pair (ClinGen SVI PM3 v1.0, single proband): partner Pathogenic — *trans* +2, phase unknown +1; partner Likely pathogenic — *trans* +1, unknown 0; homozygous +1 | genotype level |

**Genotype level.** Hypotheses are enumerated per gene — homozygous / hemizygous, compound-heterozygous pairs, single
heterozygous — and scored as

`EPCR = P(genotype pathogenic) × phenotype fit × inheritance-mode fit × call quality`

A compound-het pair multiplies the two variant posteriors and a phase factor (read-backed *trans* 1.0; unknown 0.9 —
if the gene is causal and both alleles are pathogenic they must be in trans, bar a cis pair plus a hidden third
allele; read-backed *cis* 0.01). The phenotype enters once, through the gene-level phenotype fit (PP4 is therefore
not added to variant points). The heuristic weights live in one dictionary and are stress-tested in section 9.

In [ ]:
PRIOR_P, ODDS_VS = 0.10, 350.0
LOF_CSQ = {"stop_gained", "frameshift", "splice_donor", "splice_acceptor"}
NONCODING_CSQ = {"synonymous", "intron", "splice_region", "utr5", "utr3", "upstream", "downstream", "other"}

W = dict(
    phase_trans=1.0, phase_unknown=0.9, phase_cis=0.01,
    mode_biallelic_ok=1.0, mode_biallelic_unknown=0.6, mode_biallelic_dominant_gene=0.4,
    mode_het_dominant=0.5, mode_het_unknown=0.1, mode_het_single_hit=0.1, mode_het_recessive_gene=0.02,
    q_nonpass=0.3, q_allele_balance=0.5, q_low_gq=0.5,
    pheno_floor=0.02, pheno_power=3.0, dx_floor=0.95,
)
AM_BANDS = [(0.990, 4), (0.972, 3), (0.906, 2), (0.792, 1)]
AM_BENIGN = [(0.070, -3), (0.099, -2), (0.169, -1)]
REVEL_BANDS = [(0.932, 4), (0.879, 3), (0.773, 2), (0.644, 1)]
REVEL_BENIGN = [(0.016, -4), (0.052, -3), (0.183, -2), (0.290, -1)]


def points_to_posterior(points):
    odds = ODDS_VS ** (points / 8.0)
    return odds * PRIOR_P / ((odds - 1.0) * PRIOR_P + 1.0)


def clinvar_stars(revstat):
    r = (revstat or "").lower()
    if "practice_guideline" in r:
        return 4
    if "expert_panel" in r:
        return 3
    if r.startswith("no_") or "no_assertion" in r or "no_classification" in r:
        return 0
    if "multiple_submitters" in r and "no_conflict" in r:
        return 2
    return 1 if "criteria_provided" in r else 0


def clinvar_class(sig):
    s = (sig or "").lower()
    if not s or s == ".":
        return None
    if s.startswith("conflicting"):
        return "conflicting"
    if "pathogenic" in s and "benign" not in s:
        return "P/LP"
    if "benign" in s and "pathogenic" not in s:
        return "B/LB"
    if "uncertain" in s:
        return "VUS"
    return "other"


def _num(x):
    """None for missing values (NaN / None / '.' / '')."""
    if x is None:
        return None
    try:
        x = float(x)
    except (TypeError, ValueError):
        return None
    return None if math.isnan(x) else x


def _band(value, pathogenic, benign):
    for thr, pts in pathogenic:
        if value >= thr:
            return pts
    for thr, pts in benign:
        if value <= thr:
            return pts
    return 0


def variant_evidence(v, lof_ok=True):
    """(points, floor, cap, [(code, points, explanation)]) for one annotated variant (dict)."""
    ev, floor, cap = [], 0.0, 1.0
    af, nhom = _num(v.get("af")), _num(v.get("nhomalt"))
    cls, stars = clinvar_class(v.get("clnsig")), clinvar_stars(v.get("clnrevstat"))
    known_path = cls == "P/LP" and stars >= 1
    # ---- population data --------------------------------------------------------------------------------------------
    if af is None:
        ev.append(("PM2_supporting", 1, "absent from gnomAD"))
    elif af >= 0.05:
        ev.append(("BA1", -8, f"gnomAD AF {af:.2g} >= 5%"))
    elif af >= 0.01:
        if not known_path:
            ev.append(("BS1", -4, f"gnomAD AF {af:.2g} >= 1%"))
    elif af > 1e-3:
        if not known_path:
            ev.append(("BS1_supporting", -1, f"gnomAD AF {af:.2g} > 0.1% (too common for a fully penetrant allele; possible hypomorph)"))
    elif af <= 1e-4 and not nhom:
        ev.append(("PM2_supporting", 1, f"gnomAD AF or FAF95 (the higher) {af:.2g} <= 1e-4, no homozygotes"))
    if nhom is not None and nhom >= 2 and not known_path:
        ev.append(("BS2", -4, f"{int(nhom)} homozygotes in gnomAD"))
    ov = _num(v.get("common_overlap"))
    if ov is not None and ov >= 0.01 and not known_path:
        ev.append(("BS1_overlap", -4, f"overlaps a common gnomAD indel (AF {ov:.2g}): representation artefact likely"))
    # ---- ClinVar: documented prior, never a requirement ------------------------------------------------------------
    sig = (v.get("clnsig") or "").lower()
    if cls == "P/LP":
        if stars >= 2:
            floor = 0.90 if sig.startswith("likely_pathogenic") else (0.95 if "likely_pathogenic" in sig else 0.99)
            ev.append((f"ClinVar_{stars}star", 0, f"ClinVar {v.get('clnsig')} ({stars}-star): posterior >= {floor}"))
        else:
            ev.append((f"ClinVar_{stars}star", 2 if stars == 1 else 1, f"ClinVar {v.get('clnsig')} ({stars}-star)"))
    elif cls == "B/LB":
        if stars >= 2:
            cap = 0.01
            ev.append((f"ClinVar_benign_{stars}star", 0, f"ClinVar {v.get('clnsig')} ({stars}-star): posterior <= 0.01"))
        else:
            ev.append((f"ClinVar_benign_{stars}star", -2 if stars == 1 else -1, f"ClinVar {v.get('clnsig')}"))
    # ---- consequence -------------------------------------------------------------------------------------------------
    c = v.get("csq")
    sp = _num(v.get("spliceai"))
    if c in LOF_CSQ:
        frac = _num(v.get("frac_lost"))
        if not lof_ok:
            ev.append(("PVS1_moderate", 2, f"{c}; loss of function is not an established mechanism for this gene"))
        elif v.get("nmd_escape") is True:
            if frac is not None and frac > 0.10:
                ev.append(("PVS1_strong", 4, f"{c}; escapes NMD, {frac:.0%} of the protein lost"))
            else:
                ev.append(("PVS1_moderate", 2, f"{c}; escapes NMD, < 10% of the protein lost"))
        else:
            ev.append(("PVS1", 8, f"{c}; nonsense-mediated decay predicted"))
    elif c == "start_lost":
        ev.append(("PVS1_moderate", 2, "start codon lost"))
    elif c in ("stop_lost", "inframe_indel"):
        ev.append(("PM4", 2, c.replace("_", " ")))
    elif c == "missense":
        am, revel = _num(v.get("am")), _num(v.get("revel"))
        tool, val, pts = (("AlphaMissense", am, _band(am, AM_BANDS, AM_BENIGN)) if am is not None else
                          (("REVEL", revel, _band(revel, REVEL_BANDS, REVEL_BENIGN)) if revel is not None else (None, None, 0)))
        if pts:
            name = {4: "strong", 3: "moderate+", 2: "moderate", 1: "supporting"}[abs(pts)]
            ev.append((f"{'PP3' if pts > 0 else 'BP4'}_{name}", pts, f"{tool} {val:.3f}"))
    if c not in LOF_CSQ:
        if sp is not None and sp >= 0.2:
            code, pts = ("PP3_splice_strong", 4) if sp >= 0.8 else (("PP3_splice_moderate", 2) if sp >= 0.5 else ("PP3_splice_supporting", 1))
            ev.append((code, pts, f"SpliceAI max delta {sp:.2f}"))
        elif c in NONCODING_CSQ and not (cls == "P/LP"):
            if sp is not None and sp <= 0.1:
                ev.append(("BP4_BP7" if c == "synonymous" else "BP4", -2 if c == "synonymous" else -1, f"{c}; SpliceAI {sp:.2f}"))
            elif sp is None:
                ev.append(("BP4_default", -1, f"{c}; no predicted functional effect (splice model not run)"))
    return sum(p for _, p, _ in ev), floor, cap, ev


def posterior(v, extra=0.0):
    """Variant posterior with optional genotype-level points (PM3), bounded by the ClinVar floor / cap."""
    return float(min(v["cap"], max(v["floor"], points_to_posterior(v["points"] + extra))))


def quality_factor(v, w=W):
    q = 1.0
    if str(v.get("filt", "PASS")) not in ("PASS", "."):
        q *= w["q_nonpass"]
    a, b = _num(v.get("ad_ref")), _num(v.get("ad_alt"))
    if a is not None and b is not None and (a + b) > 0 and v.get("zyg") == "het":
        if not 0.25 <= b / (a + b) <= 0.75:
            q *= w["q_allele_balance"]
    gq = _num(v.get("gq"))
    if gq is not None and gq < 20:
        q *= w["q_low_gq"]
    return q


def phase_of(a, b):
    """Read-backed phase from GATK/Sentieon PID/PGT tags (same PID: same PGT = cis, opposite = trans)."""
    pa, pb = str(a.get("pid") or "."), str(b.get("pid") or ".")
    ga, gb = str(a.get("pgt") or "."), str(b.get("pgt") or ".")
    if pa not in (".", "", "nan") and pa == pb and ga in ("0|1", "1|0") and gb in ("0|1", "1|0"):
        return "cis" if ga == gb else "trans"
    return "unknown"


def pheno_factor(info, top, w=W):
    """Gene-level phenotype fit in (0, 1]: cubic in the HPO similarity relative to the best-matching gene; genes
    without a Mendelian phenotype annotation get a small floor; genes on the indication's panel are lifted."""
    if not info:
        return w["pheno_floor"]
    f = max(w["pheno_floor"], (info["pheno"] / top) ** w["pheno_power"]) if top > 0 else w["pheno_floor"]
    if info.get("dx_match"):
        f = max(f, w["dx_floor"])
    return min(f, 1.0)


def vkey(v):
    return (v["chrom"], int(v["pos"]), v["ref"], v["alt"])


def score_variants(cand_df, gene_table, lof_genes):
    recs = cand_df.to_dict("records")
    for v in recs:
        inh = set((gene_table.get(v["gene"]) or {}).get("inheritance_all") or [])
        lof_ok = v["gene"] in lof_genes or bool(inh & {"AR", "XLR", "XL"})
        v["points"], v["floor"], v["cap"], v["evidence"] = variant_evidence(v, lof_ok=lof_ok)
        v["p_path"] = posterior(v)
    return recs


def build_hypotheses(recs, gene_table, pheno_top, w=W, max_pair_variants=14):
    by_gene = collections.defaultdict(list)
    for v in recs:
        by_gene[v["gene"]].append(v)
    hyps = []
    for g, vs in by_gene.items():
        info = gene_table.get(g) or {}
        inh = set(info.get("inheritance") or [])
        pf = pheno_factor(info, pheno_top, w)
        recessive, dominant = bool(inh & {"AR", "XLR", "XL"}), bool(inh & {"AD", "XLD", "XL"})
        bi_mode = w["mode_biallelic_ok"] if recessive else (w["mode_biallelic_dominant_gene"] if dominant else w["mode_biallelic_unknown"])
        for v in vs:
            q = quality_factor(v, w)
            af = _num(v.get("af"))
            if v["zyg"] in ("hom", "hemi"):
                extra = 1.0 if (af is None or af <= 1e-4) else (-3.0 if af > 1e-3 else 0.0)     # PM3 (homozygous) / BS1 in a homozygous model
                hyps.append(dict(gene=g, model="homozygous" if v["zyg"] == "hom" else "hemizygous", variants=[v], phase="n/a",
                                 p_geno=posterior(v, extra), pheno=pf, mode=bi_mode, quality=q))
            else:
                if dominant:
                    mode = w["mode_het_dominant"] * (1.0 if (af is None or af <= 1e-4) else 0.1)        # dominant rarity gate
                elif recessive:
                    mode = w["mode_het_single_hit"] if v["p_path"] >= 0.9 else w["mode_het_recessive_gene"]
                else:
                    mode = w["mode_het_unknown"]
                hyps.append(dict(gene=g, model="heterozygous", variants=[v], phase="n/a", p_geno=v["p_path"], pheno=pf,
                                 mode=mode, quality=q))
        hets = sorted((v for v in vs if v["zyg"] == "het"), key=lambda v: -v["p_path"])[:max_pair_variants]
        for a, b in itertools.combinations(hets, 2):       # a is the stronger allele
            if vkey(a) == vkey(b):
                continue
            ph = phase_of(a, b)
            b_af = _num(b.get("af"))
            pm3 = 0.0
            if ph != "cis" and (b_af is None or b_af <= 1e-4):         # PM3 once, on the weaker allele, if it meets PM2
                if a["p_path"] >= 0.99:                                 # partner Pathogenic: trans moderate, unknown supporting
                    pm3 = 2.0 if ph == "trans" else 1.0
                elif a["p_path"] >= 0.90:                               # partner Likely pathogenic: trans supporting, unknown none
                    pm3 = 1.0 if ph == "trans" else 0.0
            pa, pb = a["p_path"], posterior(b, pm3)
            hyps.append(dict(gene=g, model="compound_het", variants=[a, b], phase=ph, p_geno=pa * pb * w[f"phase_{ph}"],
                             pheno=pf, mode=bi_mode, quality=quality_factor(a, w) * quality_factor(b, w),
                             p_each=(round(pa, 4), round(pb, 4)), pm3=pm3))
    for h in hyps:
        h["epcr"] = float(min(0.99, max(1e-4, h["p_geno"] * h["pheno"] * h["mode"] * h["quality"])))
        h["key"] = frozenset(vkey(v) for v in h["variants"])
    hyps.sort(key=lambda h: (-h["epcr"], -len(h["variants"])))
    return hyps


def describe_variant(v):
    hg = v.get("hgvsp") or ""
    return f"{v['chrom']}:{int(v['pos'])} {v['ref']}>{v['alt']} {v['csq']}" + (f" {hg}" if hg else "")


def hyp_table(hyps, n=15):
    rows = []
    for i, h in enumerate(hyps[:n], 1):
        rows.append(dict(rank=i, EPCR=round(h["epcr"], 4), gene=h["gene"], model=h["model"], phase=h["phase"],
                         variants=" + ".join(describe_variant(v) for v in h["variants"]),
                         evidence=" + ".join("[" + ", ".join(c for c, _, _ in v["evidence"]) + "]" for v in h["variants"]),
                         P_genotype=round(h["p_geno"], 3), phenotype_fit=round(h["pheno"], 2), mode_fit=h["mode"],
                         quality=round(h["quality"], 2)))
    return pd.DataFrame(rows)


def reportable_gene(g):
    """Gene names outside the stated indication's panel and the leading hypothesis are withheld from displayed tables
    and reports (organisers' guidance on incidental findings is pending); set CFG['MASK_INCIDENTAL']=False to show them."""
    if DEMO or not CFG.get("MASK_INCIDENTAL", True):
        return True
    return g == S.get("top_gene") or g in set((S.get("pheno") or {}).get("dx_panel", []))


def public_table(df):
    if "gene" not in df.columns or not len(df):
        return df
    df = df.copy()
    hide = ~df["gene"].map(reportable_gene)
    for c in ("variants", "evidence"):
        if c in df.columns:
            df.loc[hide, c] = "(withheld)"
    df.loc[hide, "gene"] = "other gene (withheld)"
    return df

## 8 · Normalise and annotate

1. primary chromosomes, UCSC contig names, multi-allelic sites split, indels left-aligned against the reference
   (`bcftools norm`); spanning-deletion `*` alleles dropped;
2. population frequency (gnomAD v4.1 joint exomes+genomes from a compact `echtvar` archive: AF, grpmax filtering AF,
   homozygote count; two independent fallbacks);
3. ClinVar assertions — looked up genome-wide and *before* any frequency filter, so that known pathogenic
   regulatory or founder alleles are not lost;
4. consequences on MANE Select / MANE Plus Clinical transcripts (`bcftools csq`), an automated PVS1 decision
   (nonsense-mediated-decay rule) and AlphaMissense for every rare missense variant.

The VCF's own `INFO/AF` is the within-sample allele fraction and is never used as a population frequency.

In [ ]:
CSQ_CLASS = {
    "stop_gained": "stop_gained", "frameshift": "frameshift", "splice_acceptor": "splice_acceptor",
    "splice_donor": "splice_donor", "start_lost": "start_lost", "stop_lost": "stop_lost",
    "inframe_deletion": "inframe_indel", "inframe_insertion": "inframe_indel", "inframe_altering": "inframe_indel",
    "missense": "missense", "splice_region": "splice_region", "synonymous": "synonymous",
    "stop_retained": "synonymous", "start_retained": "synonymous", "coding_sequence": "inframe_indel", "feature_truncation": "frameshift",
    "5_prime_utr": "utr5", "3_prime_utr": "utr3", "intron": "intron", "non_coding": "other",
    "intergenic": "other", "feature_elongation": "inframe_indel",
}
CSQ_RANK = ["stop_gained", "frameshift", "splice_acceptor", "splice_donor", "start_lost", "stop_lost", "inframe_indel",
            "missense", "splice_region", "synonymous", "utr5", "utr3", "intron", "upstream", "downstream", "other"]
CODING_CLASSES = set(CSQ_RANK[:10])
AA3 = dict(A="Ala", R="Arg", N="Asn", D="Asp", C="Cys", Q="Gln", E="Glu", G="Gly", H="His", I="Ile", L="Leu", K="Lys",
           M="Met", F="Phe", P="Pro", S="Ser", T="Thr", W="Trp", Y="Tyr", V="Val")
AA3["*"] = "Ter"


def parse_bcsq(bcsq):
    """INFO/BCSQ -> {gene: (class, transcript, aa_change)} keeping the most severe consequence per gene."""
    out = {}
    if not bcsq or bcsq == ".":
        return out
    for entry in bcsq.split(","):
        f = entry.split("|")
        if len(f) < 3 or entry.startswith("@"):
            continue
        terms = [CSQ_CLASS.get(t.lstrip("*"), "other") for t in f[0].split("&")]
        cls = min(terms, key=CSQ_RANK.index)
        gene, tid = f[1], f[2]
        aa = f[5] if len(f) > 5 else ""
        if gene not in out or CSQ_RANK.index(cls) < CSQ_RANK.index(out[gene][0]):
            out[gene] = (cls, tid, aa)
    return out


def protein_hgvs(aa):
    """bcftools csq amino-acid string ('737L>737*', '1002N>1002K') -> ('p.Leu737Ter', 737)."""
    m = re.match(r"^(\d+)([A-Z*]+)>(\d+)([A-Z*]+)$", aa or "")
    if not m:
        m2 = re.match(r"^(\d+)", aa or "")
        return "", (int(m2.group(1)) if m2 else None)
    pos, ref, alt = int(m.group(1)), m.group(2), m.group(4)
    if len(ref) == 1 and len(alt) == 1:
        return f"p.{AA3.get(ref, ref)}{pos}{AA3.get(alt, alt)}", pos
    return f"p.{AA3.get(ref[0], ref[0])}{pos}delins", pos


def nmd_status(gene, pos):
    """(escapes_nmd, fraction_of_protein_lost) for a truncating variant, from the MANE CDS structure:
    NMD is not predicted when the stop falls in the last coding exon or within 50 nt of the end of the penultimate one."""
    tids = GENE_TX.get(gene) or []
    t = TX.get(tids[0]) if tids else None
    if not t or not t["cds"]:
        return None, None
    plus, cum = t["strand"] == "+", 0
    for i, (s, e) in enumerate(t["cds"]):
        if s - 3 <= pos <= e + 3:
            p = min(max(pos, s), e)
            off, dist3 = (p - s, e - p) if plus else (e - p, p - s)
            last = len(t["cds"]) - 1
            escape = last == 0 or i == last or (i == last - 1 and dist3 < 50)
            return escape, round(1.0 - (cum + off) / max(t["cds_len"], 1), 3)
        cum += e - s + 1
    return None, None


def _has(hdr, kind, tag):
    return f"##{kind}=<ID={tag}," in hdr


def annotate_vcf(vcf_in, tag, spikes=None, sex="unknown"):
    """Raw single-sample VCF -> (candidate table, non-coding pool, annotated BCF path)."""
    d = WORK / "data"
    hdr = sh(f"bcftools view -h '{vcf_in}'", name=f"{tag}_hdr", capture=True)
    idx_names = [l.split("\t")[0] for l in sh(f"bcftools index -s '{vcf_in}'", name=f"{tag}_idx", capture=True).splitlines() if l.strip()]
    idx_set = set(idx_names)
    if "chr1" in idx_set or "chrX" in idx_set:
        chr_style = True
    elif "1" in idx_set or "X" in idx_set:
        chr_style = False
    else:
        raise RuntimeError(f"[{tag}] unrecognised contig naming: {idx_names[:8]}")
    primary_in = [c for c in (PRIMARY if chr_style else [p[3:] for p in PRIMARY]) if c in idx_set]
    if not primary_in:
        raise RuntimeError(f"[{tag}] no primary chromosomes with records")
    rename = ""
    if not chr_style:
        names = set(re.findall(r"##contig=<ID=([^,>]+)", hdr)) | set(idx_names)
        mapf = d / f"{tag}.chrmap.txt"
        mapf.write_text("".join(f"{c}\tchr{'M' if c == 'MT' else c}\n" for c in sorted(names)))
        rename = f"| bcftools annotate --rename-chrs '{mapf}' -Ou"
    norm = d / f"{tag}.norm.bcf"
    sh(f"bcftools view -r {','.join(primary_in)} -Ou '{vcf_in}' {rename} "
       f"| bcftools norm -m -any -f '{RES['ref']}' -c w -Ou "
       f"| bcftools view -e 'ALT=\"*\"' --threads {NCPU} -Ob -o '{norm}' && bcftools index -f '{norm}'", name=f"{tag}_norm")
    n_norm = int(sh(f"bcftools index -n '{norm}'", name=f"{tag}_n", capture=True).strip())
    log(f"[{tag}] normalised biallelic records on primary chromosomes: {n_norm:,}")

    if spikes:      # synthetic benchmark alleles (public ClinVar variants), tagged in the ID column
        order = {c: i for i, c in enumerate(PRIMARY)}
        rows = sorted(((c, p, r, a, f"SPIKE_{cs['case']}_{k}") for cs in spikes for k, (c, p, r, a) in enumerate(cs["variants"])),
                      key=lambda x: (order[x[0]], x[1]))
        spk = d / f"{tag}.spikes.vcf"
        with open(spk, "w") as f:
            f.write(sh(f"bcftools view -h '{norm}'", name=f"{tag}_hdr2", capture=True))
            for c, p, r, a, vid in rows:
                f.write(f"{c}\t{p}\t{vid}\t{r}\t{a}\t500\tPASS\t.\tGT:AD:DP:GQ\t0/1:20,20:40:99\n")
        merged = d / f"{tag}.spiked.bcf"
        sh(f"bgzip -f '{spk}' && bcftools index -f '{spk}.gz' && bcftools concat -a -Ou '{norm}' '{spk}.gz' "
           f"| bcftools sort -T '{WORK / 'tmp'}' -m 2G -Ob -o '{merged}' && bcftools index -f '{merged}'", name=f"{tag}_spike")
        norm.unlink()
        norm = merged
        n_norm += len(rows)

    # ---- population frequency: first source that works (fallback archives are downloaded lazily) --------------------
    gn, af_source, rare_expr = d / f"{tag}.gn.bcf", None, None
    for src in AF_SOURCES:
        try:
            if src["name"] == "echtvar":
                if "af_tool" not in RES or "af_data" not in RES:
                    raise RuntimeError("echtvar archive not downloaded")
                raw_gn = d / f"{tag}.gn.raw.bcf"
                sh(f"'{RES['af_tool']}' anno -e '{RES['af_data']}' '{norm}' '{raw_gn}' && bcftools view --threads {NCPU} -Ob -o '{gn}' '{raw_gn}' "
                   f"&& rm -f '{raw_gn}' && bcftools index -f '{gn}'", name=f"{tag}_echtvar")
                need, tagname = "gnomad_faf95max", "gnomad_faf95max"
            elif src["name"] == "slivar":
                tool = fetch(src["tool"], WORK / "bin" / "slivar", min_bytes=3_000_000)
                Path(tool).chmod(0o755)
                data = fetch(src["data"], WORK / "ref" / "af_slivar.zip", min_bytes=src["min_bytes"])
                sh(f"'{tool}' expr --vcf '{norm}' -g '{data}' --info 'true' | bcftools view --threads {NCPU} -Ob -o '{gn}' "
                   f"&& bcftools index -f '{gn}'", name=f"{tag}_slivar", env={"SLIVAR_QUIET": "1"})
                need, tagname = "gnomad_popmax_af", "gnomad_popmax_af"
            else:
                data = fetch(src["data"], WORK / "ref" / "af-only-gnomad.hg38.vcf.gz", min_bytes=src["min_bytes"])
                fetch(src["data"] + ".tbi", WORK / "ref" / "af-only-gnomad.hg38.vcf.gz.tbi", min_bytes=1_000_000)
                sh(f"bcftools annotate -a '{data}' -c 'INFO/gnomad_popmax_af:=INFO/AF' --threads {NCPU} -Ob -o '{gn}' '{norm}' "
                   f"&& bcftools index -f '{gn}'", name=f"{tag}_afonly")
                need, tagname = "gnomad_popmax_af", "gnomad_popmax_af"
            n_gn = int(sh(f"bcftools index -n '{gn}'", name=f"{tag}_n2", capture=True).strip())
            if n_gn < 0.98 * n_norm or not _has(sh(f"bcftools view -h '{gn}'", name=f"{tag}_h3", capture=True), "INFO", need):
                raise RuntimeError(f"annotation incomplete ({n_gn:,} of {n_norm:,} records)")
            rare_expr = f'(INFO/{tagname}<{2 * CFG["MAX_AF"]} || INFO/{tagname}=".")'
            af_source = src["label"]
            break
        except Exception as e:      # noqa: BLE001
            log(f"[{tag}] population-frequency source '{src['name']}' failed: {str(e)[:220]}")
    if af_source is None:
        raise RuntimeError("no population-frequency resource could be applied")
    norm.unlink(missing_ok=True)

    # ---- ClinVar (before any frequency filter), rarity filter, consequences ---------------------------------------
    keep = rare_expr + ' || INFO/CLNSIG~"athogenic" || ID~"^SPIKE"'
    ann = d / f"{tag}.ann.bcf"
    sh(f"bcftools annotate -a '{RES['clinvar_chr']}' -c INFO/CLNSIG,INFO/CLNREVSTAT,INFO/CLNDN,INFO/GENEINFO -Ou '{gn}' "
       f"| bcftools view -i '{keep}' -Ou "
       f"| bcftools csq -f '{RES['ref']}' -g '{RES['csq_gff']}' --local-csq --force --ncsq 64 --phase a -Ob -o '{ann}' "
       f"&& bcftools index -f '{ann}'", name=f"{tag}_annot")
    gn.unlink(missing_ok=True)
    Path(str(gn) + ".csi").unlink(missing_ok=True)

    h = sh(f"bcftools view -h '{ann}'", name=f"{tag}_h4", capture=True)
    info_tags = [t for t in ("gnomad_faf95max", "gnomad_AF", "gnomad_AC", "gnomad_HomAlt", "gnomad_popmax_af", "gnomad_nhomalt",
                             "CLNSIG", "CLNREVSTAT", "CLNDN", "GENEINFO", "BCSQ") if _has(h, "INFO", t)]
    fmt_tags = [t for t in ("GT", "AD", "DP", "GQ", "PGT", "PID") if _has(h, "FORMAT", t)]
    fmt = ("%CHROM\\t%POS\\t%REF\\t%ALT\\t%ID\\t%QUAL\\t%FILTER" + "".join(f"\\t%INFO/{t}" for t in info_tags)
           + "[" + "".join(f"\\t%{t}" for t in fmt_tags) + "]\\n")
    tsv = d / f"{tag}.ann.tsv"
    sh(f"bcftools query -f '{fmt}' '{ann}' > '{tsv}'", name=f"{tag}_query")
    cols = ["chrom", "pos", "ref", "alt", "rsid", "qual", "filt"] + [t.lower() for t in info_tags] + [t.lower() for t in fmt_tags]
    raw = pd.read_csv(tsv, sep="\t", header=None, names=cols, dtype=str, keep_default_na=False, quoting=3)
    tsv.unlink(missing_ok=True)
    for c in ("gnomad_faf95max", "gnomad_af", "gnomad_ac", "gnomad_homalt", "gnomad_popmax_af", "gnomad_nhomalt", "clnsig",
              "clnrevstat", "clndn", "geneinfo", "bcsq", "gt", "ad", "dp", "gq", "pgt", "pid"):
        if c not in raw.columns:
            raw[c] = "."
    log(f"[{tag}] rare or ClinVar-pathogenic records: {len(raw):,} ({af_source})")

    # ---- one row per (variant, gene) ---------------------------------------------------------------------------------
    recs = []
    for r in raw.itertuples(index=False):
        alleles = re.split(r"[/|]", r.gt)
        n_alt = sum(1 for x in alleles if x == "1")
        if n_alt == 0:
            continue
        zyg = "hom" if n_alt == len(alleles) else "het"
        male = sex.startswith("male")
        x_nonpar = r.chrom == "chrY" or (r.chrom == "chrX" and not any(a <= int(r.pos) <= b for a, b in PAR_X))
        if zyg == "hom" and (len(alleles) == 1 or (male and x_nonpar)):
            zyg = "hemi"
        elif zyg == "het" and male and x_nonpar:
            zyg = "het_xy"          # implausible in a male: single-allele model only, never paired
        ad = (r.ad.split(",") + [".", "."])[:2]
        # unified population frequency: the larger of the global AF and the grpmax filtering AF (echtvar), or the
        # max population AF (slivar; "absent" is encoded as -1), or the global AF (af-only). None = not in gnomAD.
        f95, gaf, pmx = _num(r.gnomad_faf95max), _num(r.gnomad_af), _num(r.gnomad_popmax_af)
        af = max([x for x in (f95, gaf, pmx) if x is not None and x > 0], default=None)
        nhom = _num(r.gnomad_homalt) if _num(r.gnomad_homalt) is not None else _num(r.gnomad_nhomalt)
        base = dict(chrom=r.chrom, pos=int(r.pos), ref=r.ref, alt=r.alt, rsid=r.rsid, qual=_num(r.qual), filt=r.filt,
                    zyg=zyg, ad_ref=_num(ad[0]), ad_alt=_num(ad[1]), dp=_num(r.dp), gq=_num(r.gq),
                    pgt=r.pgt, pid=r.pid, af=af, nhomalt=nhom, gnomad_ac=_num(r.gnomad_ac),
                    clnsig=r.clnsig if r.clnsig != "." else "", clnrevstat=r.clnrevstat if r.clnrevstat != "." else "",
                    clndn=(r.clndn if r.clndn != "." else "")[:120])
        genes = parse_bcsq(r.bcsq)
        if not genes and base["clnsig"] and r.geneinfo != ".":      # e.g. a known pathogenic regulatory allele
            genes = {r.geneinfo.split(":")[0]: ("other", "", "")}
        for g, (cls, tid, aa) in genes.items():
            hg, aa_pos = protein_hgvs(aa)
            recs.append(dict(base, gene=g, transcript=tid, csq=cls, aa_change=aa, hgvsp=hg, aa_pos=aa_pos))
        if not genes:
            recs.append(dict(base, gene="", transcript="", csq="other", aa_change="", hgvsp="", aa_pos=None))
    allv = pd.DataFrame(recs)
    allv["nmd_escape"], allv["frac_lost"], allv["am"], allv["spliceai"], allv["revel"], allv["cadd"] = None, np.nan, np.nan, np.nan, np.nan, np.nan
    lof_mask = allv.csq.isin(LOF_CSQ)
    for i in allv.index[lof_mask]:
        esc, frac = nmd_status(allv.at[i, "gene"], allv.at[i, "pos"])
        allv.at[i, "nmd_escape"], allv.at[i, "frac_lost"] = esc, (np.nan if frac is None else frac)

    # ---- AlphaMissense for every rare missense variant (tabix lookups in the public BGZF table) ------------------
    mis = allv[allv.csq == "missense"]
    if len(mis) and "alphamissense" in RES:
        try:
            reg = d / f"{tag}.am.regions.tsv"
            mis[["chrom", "pos"]].drop_duplicates().assign(p2=lambda x: x.pos).to_csv(reg, sep="\t", header=False, index=False)
            out = sh(f"tabix -R '{reg}' '{RES['alphamissense']}'", name=f"{tag}_am", capture=True)
            am = {}
            for line in out.splitlines():
                c = line.split("\t")
                if len(c) >= 9:
                    am[(c[0], int(c[1]), c[2], c[3])] = float(c[8])
            keys = list(zip(allv.chrom, allv.pos, allv.ref, allv.alt))
            allv.loc[mis.index, "am"] = [am.get(keys[i], np.nan) for i in mis.index]
            log(f"[{tag}] AlphaMissense scores attached to {int(allv.loc[mis.index, 'am'].notna().sum()):,} of {len(mis):,} rare missense variants")
        except Exception as e:      # noqa: BLE001
            log(f"[{tag}] AlphaMissense lookup failed: {str(e)[:200]}")

    is_plp = allv.clnsig.map(lambda s: clinvar_class(s) == "P/LP")
    rare = allv.af.isna() | (allv.af < CFG["MAX_AF"]) | is_plp
    cand = allv[rare & (allv.gene != "") & (allv.csq.isin(CODING_CLASSES) | is_plp)].reset_index(drop=True)
    pool = allv[rare & ~(allv.csq.isin(CODING_CLASSES) | is_plp)].reset_index(drop=True)      # non-coding, for second-hit rescue
    log(f"[{tag}] candidate table: {len(cand):,} coding/splice or ClinVar-P/LP rows in {cand.gene.nunique():,} genes; "
        f"non-coding pool: {len(pool):,} rows")
    return cand, pool, str(ann), af_source


with stage("normalise + annotate", required=True):
    _sex = (S.get("qc") or {}).get("inferred_sex", "unknown")
    CAND_ALL, POOL_ALL, ANN_BCF, S["af_source"] = annotate_vcf(S["vcf_raw"], "case", spikes=SPIKE_CASES if DEMO else None, sex=_sex)
    _is_spike = CAND_ALL.rsid.str.startswith("SPIKE_")
    if DEMO:        # the demo proband = HG002 background + the two alleles of spike case 0
        CAND = CAND_ALL[~_is_spike | CAND_ALL.rsid.isin(["SPIKE_0_0", "SPIKE_0_1"])].reset_index(drop=True)
    else:
        CAND = CAND_ALL[~_is_spike].reset_index(drop=True)
        assert not _is_spike.any(), "synthetic alleles must never enter the challenge candidate table"
    POOL = POOL_ALL[~POOL_ALL.rsid.str.startswith("SPIKE_")].reset_index(drop=True)
    S["n_candidates"], S["n_candidate_genes"] = int(len(CAND)), int(CAND.gene.nunique())
    md("**Candidate rows by consequence and zygosity**\n\n" + df_md(CAND.groupby(["csq", "zyg"]).size().unstack(fill_value=0).reset_index()))

## 9 · Ranking

1. **Phenotype scoring** — HPO-only ("blind") and HPO + stated indication.
2. **First-pass ranking** of every homozygous, compound-heterozygous and heterozygous hypothesis genome-wide.
3. **Second-hit rescue** — for the genes that already carry one credible allele, every rare non-coding variant in
   the gene and ±50 kb is added as a potential second allele (deep-intronic, UTR, regulatory), so a hypomorphic
   non-coding partner is not missed. They start with a small penalty until the splice model has scored them.
4. **Robustness** — the ranking is recomputed under random perturbation of every heuristic weight, with each HPO
   term left out in turn, and with ClinVar or the missense predictor switched off.
5. **Submission file** — written immediately (and rewritten after each later evidence stage), parsed with the
   organisers' own scoring code, and every submitted allele is checked against the source VCF.

In [ ]:
with stage("phenotype scoring", required=True):
    HPO_TERMS = [h["term"] for h in S["hpo"] if h.get("weight", 1.0) > 0]
    HPO_WEIGHTS = {h["term"]: h.get("weight", 1.0) for h in S["hpo"]}
    GENES_BLIND, DIS_BLIND = PE.score_genes(HPO_TERMS, HPO_WEIGHTS, diagnosis=None)
    GENES_DX, _ = PE.score_genes(HPO_TERMS, HPO_WEIGHTS, diagnosis=S.get("diagnosis") or None)
    PHENO_TOP = max(v["pheno"] for v in GENES_BLIND.values()) or 1.0
    _order = sorted(GENES_BLIND, key=lambda g: -GENES_BLIND[g]["pheno"])
    GENE_RANK_BLIND = {g: i + 1 for i, g in enumerate(_order)}
    S["pheno"] = dict(
        n_diseases=PE.n_diseases, n_genes=len(GENES_BLIND),
        dx_panel=sorted(g for g, v in GENES_DX.items() if v["dx_match"]),
        top_diseases=[dict(rank=i + 1, disease=d, name=PE.d_name[d], score=round(DIS_BLIND[d][0], 3),
                           genes=",".join(sorted(PE.d2g.get(d, []))[:5]), inheritance="/".join(sorted(PE.d_inh.get(d, []))))
                      for i, d in enumerate(sorted(DIS_BLIND, key=lambda d: -DIS_BLIND[d][0])[:12])])
    md("**Top diseases by HPO similarity (blind to the diagnosis)**\n\n" + df_md(pd.DataFrame(S["pheno"]["top_diseases"])))
    if S.get("diagnosis"):
        log(f"virtual panel from the stated indication '{S['diagnosis']}': {S['pheno']['dx_panel']}")

In [ ]:
def second_hit_rows(recs, pool_df, gene_table, pheno_top, w=W):
    """Rare non-coding variants in (and around) genes that already carry one credible heterozygous allele."""
    best = {}
    for v in recs:
        if v["zyg"] != "het" or v["p_path"] < 0.5:
            continue
        s = v["p_path"] * pheno_factor(gene_table.get(v["gene"]) or {}, pheno_top, w)
        if s > best.get(v["gene"], 0):
            best[v["gene"]] = s
    genes = [g for g in sorted(best, key=lambda g: -best[g]) if best[g] >= 0.02][:CFG["TOP_GENES_DEEP"]]
    out = []
    for g in genes:
        if g not in GENE_POS:
            continue
        chrom, start, end, strand, _ = GENE_POS[g]
        sub = pool_df[(pool_df.gene == g) & (pool_df.zyg == "het")]
        flank = pool_df[(pool_df.gene == "") & (pool_df.chrom == chrom) & (pool_df.zyg == "het")
                        & (pool_df.pos >= start - CFG["FLANK_SECOND_HIT"]) & (pool_df.pos <= end + CFG["FLANK_SECOND_HIT"])].copy()
        if len(flank):
            flank["gene"] = g
            up = (flank.pos < start) if strand == "+" else (flank.pos > end)
            flank["csq"] = np.where(up, "upstream", "downstream")
        out.append(pd.concat([sub, flank]))
    return (pd.concat(out).reset_index(drop=True) if out else pool_df.iloc[0:0]), genes


def rank_case(cand_df, pool_df, gene_table, pheno_top, w=W, rescue=True):
    recs = score_variants(cand_df, gene_table, LOF_GENES)
    rescue_genes = []
    if rescue and pool_df is not None and len(pool_df):
        extra, rescue_genes = second_hit_rows(recs, pool_df, gene_table, pheno_top, w)
        if len(extra):
            have = {(vkey(v), v["gene"]) for v in recs}
            recs += [v for v in score_variants(extra, gene_table, LOF_GENES) if (vkey(v), v["gene"]) not in have]
    return recs, build_hypotheses(recs, gene_table, pheno_top, w), rescue_genes


with stage("first-pass ranking", required=True):
    RECS, HYPS, RESCUE_GENES = rank_case(CAND, POOL, GENES_DX, PHENO_TOP)
    _, HYPS_BLIND, _ = rank_case(CAND, POOL, GENES_BLIND, PHENO_TOP)
    S["ranking_stage"] = "first pass (offline annotations)"
    S["n_hypotheses"] = len(HYPS)
    top = HYPS[0]
    S["top_gene"] = top["gene"]
    md(f"**{len(HYPS):,} genotype hypotheses** from {len(RECS):,} scored variant rows. Top hypotheses (with the stated indication):\n\n"
       + df_md(public_table(hyp_table(HYPS, 12))))
    md("**Blind to the diagnosis (HPO terms only):**\n\n"
       + df_md(public_table(hyp_table(HYPS_BLIND, 5))[["rank", "EPCR", "gene", "model", "variants", "phenotype_fit"]]))
    S["top_gene_pheno_rank_blind"] = GENE_RANK_BLIND.get(top["gene"])
    S["top_same_blind"] = bool(HYPS_BLIND and HYPS_BLIND[0]["key"] == top["key"])
    log(f"top hypothesis: {top['gene']} {top['model']} EPCR={top['epcr']:.3f}; phenotype rank of the gene (HPO only): "
        f"{S['top_gene_pheno_rank_blind']} of {len(GENES_BLIND):,}; same top hypothesis without the diagnosis: {S['top_same_blind']}")

In [ ]:
def robustness(recs, hyps, n_perturb=300, seed=11):
    """How stable is the top hypothesis? (a) every heuristic weight multiplied by a random factor in [0.5, 2];
    (b) each HPO term left out; (c) ClinVar ignored; (d) missense predictor ignored; (e) diagnosis ignored."""
    rng = np.random.default_rng(seed)
    top_key = hyps[0]["key"]
    out = {}
    same, ranks = 0, []
    for _ in range(n_perturb):
        w = {}
        for k, v in W.items():
            f = float(np.exp(rng.uniform(-math.log(2), math.log(2))))
            w[k] = v * f if k == "pheno_power" else min(1.0, v * f)
        h2 = build_hypotheses(recs, GENES_DX, PHENO_TOP, w)
        r = next((i + 1 for i, h in enumerate(h2) if h["key"] == top_key), None)
        ranks.append(r or 999)
        same += int(r == 1)
    out["weight_perturbation"] = dict(n=n_perturb, top1_unchanged=round(same / n_perturb, 3), worst_rank=int(max(ranks)))
    loo = []
    for h in S["hpo"]:
        if h.get("weight", 1.0) <= 0:
            continue
        terms = [t for t in HPO_TERMS if t != h["term"]]
        gt, _ = PE.score_genes(terms, HPO_WEIGHTS, diagnosis=None)        # blind, so that the term itself matters
        ptop = max(v["pheno"] for v in gt.values()) or 1.0
        _, h2, _ = rank_case(CAND, POOL, gt, ptop)
        loo.append(dict(term_left_out=h["term"], label=h.get("label", ""), rank_of_top=next((i + 1 for i, x in enumerate(h2) if x["key"] == top_key), None)))
    out["leave_one_term_out_blind"] = loo

    def ablate(mutator, table):
        c2 = CAND.copy()
        mutator(c2)
        _, h2, _ = rank_case(c2, POOL, table, PHENO_TOP)
        return next((i + 1 for i, x in enumerate(h2) if x["key"] == top_key), None)

    out["rank_without_clinvar"] = ablate(lambda c: c.__setitem__("clnsig", ""), GENES_DX)
    out["rank_without_missense_predictor"] = ablate(lambda c: c.__setitem__("am", np.nan), GENES_DX)
    out["rank_without_diagnosis"] = ablate(lambda c: None, GENES_BLIND)
    out["rank_without_clinvar_and_diagnosis"] = ablate(lambda c: c.__setitem__("clnsig", ""), GENES_BLIND)
    return out


with stage("robustness analysis"):
    S["robustness"] = robustness(RECS, HYPS)
    rb = S["robustness"]
    log(f"weight perturbation: top-1 unchanged in {rb['weight_perturbation']['top1_unchanged']:.1%} of "
        f"{rb['weight_perturbation']['n']} runs (worst rank {rb['weight_perturbation']['worst_rank']})")
    log(f"rank of the top hypothesis without ClinVar: {rb['rank_without_clinvar']}; without the missense predictor: "
        f"{rb['rank_without_missense_predictor']}; without the diagnosis: {rb['rank_without_diagnosis']}; "
        f"without both ClinVar and the diagnosis: {rb['rank_without_clinvar_and_diagnosis']}")
    md("**Leave-one-HPO-term-out (blind mode)**\n\n" + df_md(pd.DataFrame(rb["leave_one_term_out_blind"])))

In [ ]:
# ---- the organisers' public Track 1 scorer (evaluation.py of the challenge Space), embedded verbatim in logic so the
# ---- file we write is parsed and scored exactly as it will be on submission
RANK_POINT_TIERS = [(1, 100), (3, 50), (5, 25), (10, 10)]


def official_load(csv_path):
    import csv as _csv
    rows = []
    with open(csv_path, newline="") as f:
        for row in _csv.DictReader(f):
            def pv(c, p, r, a):
                return None if (not c or not p) else (c.strip(), int(p), r.strip().upper(), a.strip().upper())
            v1 = pv(row["chrom_1"], row["pos_1"], row["ref_1"], row["alt_1"])
            v2 = pv(row.get("chrom_2", ""), row.get("pos_2", ""), row.get("ref_2", ""), row.get("alt_2", ""))
            if v1 is None:
                raise ValueError("row missing primary variant")
            epcr = float(row["epcr"])
            if not (0 < epcr <= 1):
                raise ValueError(f"EPCR {epcr} out of range (0,1]")
            ft = (row.get("finding_type") or "primary").strip().lower()
            if ft not in ("primary", "secondary"):
                raise ValueError(f"bad finding_type {ft}")
            rows.append((row["proband_id"].strip(), frozenset([v1, v2]) if v2 else frozenset([v1]), epcr, ft))
    if len(rows) > 10:
        raise ValueError(f"{len(rows)} rows; max is 10")
    order = sorted(range(len(rows)), key=lambda i: (-rows[i][2], i))
    return [(rows[i][0], rows[i][1], rows[i][2], rank + 1) for rank, i in enumerate(order)]


def official_score(sub_rows, true_variants):
    full = next((r for _, vs, _, r in sub_rows if vs == true_variants), None)
    partial = None
    if len(true_variants) == 2 and full is None:
        partial = next((r for _, vs, _, r in sub_rows if vs & true_variants), None)
    pts = lambda r: next((p for m, p in RANK_POINT_TIERS if r <= m), 0)
    rank_points = float(pts(full)) if full else (0.5 * pts(partial) if partial else 0.0)
    best_f = 0.0
    for t in sorted({e for _, _, e, _ in sub_rows}, reverse=True):
        pred = set()
        for _, vs, e, _ in sub_rows:
            if e >= t:
                pred |= vs
        tp, fp, fn = len(pred & true_variants), len(pred - true_variants), len(true_variants - pred)
        pr = tp / (tp + fp) if tp + fp else 0.0
        rc = tp / (tp + fn) if tp + fn else 0.0
        best_f = max(best_f, 2 * pr * rc / (pr + rc) if pr + rc else 0.0)
    return dict(rank_points=rank_points, f_max=round(best_f, 3), full_match_rank=full, partial_match_rank=partial)


SUBMISSION_CSV = OUT / f"{CFG['TEAM']}_{CFG['MODEL_NAME']}.csv"


def select_rows(hyps, max_rows=10, per_gene=5, min_epcr=0.01, min_epcr_other_gene=0.25):
    """Rows for the CSV: the top hypothesis, alternatives within its gene (other pairings, single alleles: partial
    credit if only one allele is right), and hypotheses in other genes only when they are serious competitors."""
    rows, seen, n_gene = [], set(), collections.Counter()
    for h in hyps:
        if rows and h["epcr"] < min_epcr:
            break
        if rows and h["gene"] != rows[0]["gene"] and h["epcr"] < min_epcr_other_gene:
            continue
        if h["key"] in seen or n_gene[h["gene"]] >= per_gene:
            continue
        if rows and len(h["key"]) == 1 and h["key"] <= frozenset().union(*(r["key"] for r in rows)):
            continue        # a single allele already listed in a higher pair adds nothing to rank points or F-max
        if rows and any(v.get("points", 0) < 0 for v in h["variants"]):
            continue        # a variant with net benign ACMG evidence is not put forward as a causal candidate
        seen.add(h["key"])
        n_gene[h["gene"]] += 1
        rows.append(h)
        if len(rows) >= max_rows:
            break
    return rows


def write_submission(hyps, secondary=None):
    """Top hypotheses -> the organisers' CSV format (chr-prefixed GRCh38, integer positions, strictly decreasing EPCR)."""
    import csv as _csv
    secondary = (secondary or [])[:2] if CFG["INCLUDE_SECONDARY_ROWS"] else []
    primary = select_rows(hyps, max_rows=10 - len(secondary))
    out_rows, prev = [], None
    for h, ftype in [(h, "primary") for h in primary] + [(h, "secondary") for h in secondary]:
        e = round(min(h["epcr"], 0.99), 4)
        if ftype == "secondary":
            e = min(e, 0.009)
        if prev is not None and e >= prev:
            e = round(prev - 0.0005, 4)
        e = max(e, 0.0001)
        prev = e
        vs = sorted(h["variants"], key=lambda v: (PRIMARY.index(v["chrom"]) if v["chrom"] in PRIMARY else 99, int(v["pos"])))
        v1, v2 = vs[0], (vs[1] if len(vs) > 1 else None)
        note = h.get("note") or f"{h['gene']} {h['model'].replace('_', '-')}" + (f", phase {h['phase']}" if v2 else "")
        out_rows.append([CFG["PROBAND_ID"], v1["chrom"], int(v1["pos"]), v1["ref"], v1["alt"],
                         v2["chrom"] if v2 else "", int(v2["pos"]) if v2 else "", v2["ref"] if v2 else "", v2["alt"] if v2 else "",
                         f"{e:.4f}", ftype, note.replace(",", ";")])
    with open(SUBMISSION_CSV, "w", newline="") as f:
        wr = _csv.writer(f)
        wr.writerow(["proband_id", "chrom_1", "pos_1", "ref_1", "alt_1", "chrom_2", "pos_2", "ref_2", "alt_2", "epcr", "finding_type", "notes"])
        wr.writerows(out_rows)
    parsed = official_load(SUBMISSION_CSV)             # raises if the organisers' parser would reject the file
    assert parsed[0][1] == primary[0]["key"], "top row of the CSV is not the top hypothesis"
    assert len({e for _, _, e, _ in parsed}) == len(parsed), "EPCR values must be strictly decreasing"
    return primary, parsed


def verify_in_source(variants):
    """Every submitted allele must be present in the annotated copy of the source VCF (and must not be synthetic)."""
    res = []
    for v in variants:
        out = sh(f"bcftools view -H -r {v['chrom']}:{int(v['pos'])}-{int(v['pos'])} '{ANN_BCF}' | cut -f1-5", name="verify", capture=True)
        hit = [l.split("\t") for l in out.splitlines() if l.strip()]
        ok = [h for h in hit if int(h[1]) == int(v["pos"]) and h[3] == v["ref"] and h[4] == v["alt"]]
        res.append(dict(variant=f"{v['chrom']}:{int(v['pos'])} {v['ref']}>{v['alt']}", found=bool(ok),
                        synthetic=bool(ok and ok[0][2].startswith("SPIKE_"))))
    return res


with stage("write + validate submission (first pass)", required=True):
    SUB_ROWS, parsed = write_submission(HYPS)
    chk = verify_in_source([v for h in SUB_ROWS for v in h["variants"]])
    S["source_check"] = chk
    if not DEMO:
        assert all(c["found"] and not c["synthetic"] for c in chk), f"submitted allele not found in the source VCF: {chk}"
    if DEMO:
        truth = frozenset(SPIKE_CASES[0]["variants"])
        S["demo_score"] = official_score(parsed, truth)
        log(f"DEMO self-score against the spiked truth (organisers' scorer): {S['demo_score']}")
    log(f"submission written: {SUBMISSION_CSV.name} ({len(parsed)} rows)")
    md("**Submission rows (first pass)**\n\n" + df_md(pd.read_csv(SUBMISSION_CSV, dtype=str, keep_default_na=False)))

## 10 · Deep annotation of the short list and final ranking

Only the variants that appear in the leading hypotheses (a few dozen alleles) are sent to public annotation services:

* **Ensembl VEP REST** — HGVS on the MANE transcript, LOFTEE, pre-computed SpliceAI, CADD, REVEL, literature links;
* **gnomAD v4.1 joint** (remote tabix) — exact allele counts, group-maximum filtering AF, homozygotes, and a check
  for common overlapping indels (the classic way a mis-represented common allele masquerades as a novel one);
* **Genomics England PanelApp** — is the leading gene a green gene on NHS virtual panels, and with which mode of inheritance;
* **chance-bystander analysis** (gnomAD gene-level counts) — how often would an unrelated person carry, by coincidence,
  a rare variant at least as damaging as the weaker allele of the leading pair?

The evidence model is then re-run with the richer annotations and the submission file is rewritten.

In [ ]:
import requests

HTTP = requests.Session()
HTTP.headers.update({"User-Agent": "duorank-notebook/1.0 (research; rare-disease hackathon)"})


def http_json(method, url, tries=4, timeout=120, **kw):
    last = None
    for i in range(tries):
        try:
            r = HTTP.request(method, url, timeout=timeout, **kw)
            if r.status_code == 200:
                return r.json()
            last = f"HTTP {r.status_code}: {r.text[:150]}"
            if r.status_code in (400, 404):
                break
        except Exception as e:      # noqa: BLE001
            last = f"{type(e).__name__}: {str(e)[:150]}"
        time.sleep(3 * (i + 1))
    raise RuntimeError(f"{url.split('?')[0]} failed ({last})")


def shortlist_variants(hyps, recs, n_hyp=40, n_genes=3):
    keep = {}
    for h in hyps[:n_hyp]:
        for v in h["variants"]:
            keep[(vkey(v), v["gene"])] = v
    top_genes = list(dict.fromkeys(h["gene"] for h in hyps[:20]))[:n_genes]
    for v in recs:
        if v["gene"] in top_genes:
            keep[(vkey(v), v["gene"])] = v
    return list(keep.values()), top_genes


def vep_annotate(variants):
    """Ensembl VEP REST (GRCh38) for a short list; returns {vkey: raw VEP record}."""
    uniq = {vkey(v): v for v in variants}
    strings = {f"{k[0][3:] if k[0] != 'chrM' else 'MT'} {k[1]} . {k[2]} {k[3]} . . .": k for k in uniq}
    out, items = {}, list(strings)
    for i in range(0, len(items), 150):
        res = http_json("POST", "https://rest.ensembl.org/vep/homo_sapiens/region",
                        headers={"Content-Type": "application/json", "Accept": "application/json"},
                        params={"hgvs": 1, "mane": 1, "canonical": 1, "variant_class": 1, "SpliceAI": 1, "CADD": 1, "AlphaMissense": 1,
                                "REVEL": 1, "LoF": 1, "numbers": 1}, json={"variants": items[i:i + 150]}, timeout=600)
        for o in res:
            k = strings.get(o.get("input"))
            if k:
                out[k] = o
    return out


def apply_vep(recs, vep):
    n = 0
    for v in recs:
        o = vep.get(vkey(v))
        if not o:
            continue
        tcs = o.get("transcript_consequences") or []
        mane = [t for t in tcs if t.get("mane_select") and t.get("gene_symbol") == v["gene"]] or [t for t in tcs if t.get("mane_select")]
        t = (mane or [t for t in tcs if t.get("canonical")] or [None])[0]
        v["rsids"] = [c.get("id") for c in o.get("colocated_variants", []) if str(c.get("id", "")).startswith("rs")]
        v["pubmed"] = sorted({p for c in o.get("colocated_variants", []) for p in (c.get("pubmed") or [])})[:12]
        if not t:
            continue
        sp = t.get("spliceai") or {}
        if sp:
            v["spliceai"] = max(float(sp.get(x) or 0) for x in ("DS_AG", "DS_AL", "DS_DG", "DS_DL"))
            v["spliceai_detail"] = {x: sp.get(x) for x in ("DS_AG", "DS_AL", "DS_DG", "DS_DL", "DP_AG", "DP_AL", "DP_DG", "DP_DL")}
        for src, dst in (("revel", "revel"), ("cadd_phred", "cadd")):
            if t.get(src) is not None:
                v[dst] = float(t[src])
        am = (t.get("alphamissense") or {}).get("am_pathogenicity")
        if am is not None and _num(v.get("am")) is None:
            v["am"] = float(am)
        v["hgvsc"] = t.get("hgvsc", "")
        if t.get("hgvsp"):
            v["hgvsp"] = t["hgvsp"].split(":")[-1]
        v["mane_refseq"] = t.get("mane_select", "")
        v["exon"], v["intron"] = t.get("exon", ""), t.get("intron", "")
        v["loftee"] = t.get("lof", "")
        v["sift"], v["polyphen"] = t.get("sift_prediction", ""), t.get("polyphen_prediction", "")
        v["vep_consequence"] = ",".join(t.get("consequence_terms") or [])
        n += 1
    return n


GNOMAD_FMT = ("%CHROM\\t%POS\\t%REF\\t%ALT\\t%FILTER\\t%INFO/AC_joint\\t%INFO/AN_joint\\t%INFO/AF_joint\\t%INFO/nhomalt_joint"
              "\\t%INFO/AF_grpmax_joint\\t%INFO/grpmax_joint\\t%INFO/fafmax_faf95_max_joint\\n")


def gnomad_exact(variants):
    """gnomAD v4.1 joint: exact records for the short list (remote tabix) + common overlapping indels within 30 bp."""
    out, by_chr = {}, collections.defaultdict(dict)
    for v in variants:
        by_chr[v["chrom"]][vkey(v)] = v
    for chrom, vs in by_chr.items():
        if chrom not in PRIMARY:
            continue
        regions = ",".join(f"{chrom}:{max(1, k[1] - 30)}-{k[1] + len(k[2]) + 30}" for k in vs)
        txt = sh(f"bcftools query -r '{regions}' -f '{GNOMAD_FMT}' '{GNOMAD_V4_JOINT.format(chrom=chrom)}'", name=f"gnomad_{chrom}",
                 capture=True, timeout=900)
        rows = {}
        for line in txt.splitlines():
            c = line.split("\t")
            if len(c) >= 12:
                rows[(c[0], int(c[1]), c[2], c[3])] = c
        for k in vs:
            c = rows.get(k)
            rec = dict(present=c is not None, source="gnomAD v4.1 joint (remote tabix)")
            if c is not None:
                rec.update(filter=c[4], ac=_num(c[5]), an=_num(c[6]), af=_num(c[7]), nhomalt=_num(c[8]), af_grpmax=_num(c[9]),
                           grpmax=c[10] if c[10] != "." else "", faf95=_num(c[11]))
            if len(k[2]) != len(k[3]):       # indel: a common indel overlapping the same span suggests a representation artefact
                lo, hi = k[1], k[1] + len(k[2])
                ov = [(_num(r[7]) or 0.0) for kk, r in rows.items() if kk != k and len(kk[2]) != len(kk[3])
                      and kk[1] <= hi and kk[1] + len(kk[2]) >= lo]
                rec["common_overlap"] = max(ov, default=0.0)
            out[k] = rec
    return out


def gnomad_graphql(variants):
    """Fallback when remote tabix is unavailable: the public gnomAD GraphQL API (exact matches only)."""
    q = ("query($v: String!) { variant(variantId: $v, dataset: gnomad_r4) { exome { ac an homozygote_count } "
         "genome { ac an homozygote_count } joint { ac an homozygote_count faf95 { popmax popmax_population } } } }")
    out = {}
    for k in {vkey(v) for v in variants}:
        try:
            d = http_json("POST", "https://gnomad.broadinstitute.org/api", json={"query": q, "variables": {"v": f"{k[0][3:]}-{k[1]}-{k[2]}-{k[3]}"}})
            g = (d.get("data") or {}).get("variant")
            rec = dict(present=g is not None, source="gnomAD v4.1 (GraphQL API)")
            if g:
                j = g.get("joint") or {}
                ac = j.get("ac") if j else sum((g.get(x) or {}).get("ac", 0) for x in ("exome", "genome"))
                an = j.get("an") if j else sum((g.get(x) or {}).get("an", 0) for x in ("exome", "genome"))
                rec.update(ac=ac, an=an, af=(ac / an if an else None), nhomalt=j.get("homozygote_count"),
                           faf95=(j.get("faf95") or {}).get("popmax"), grpmax=(j.get("faf95") or {}).get("popmax_population") or "")
            out[k] = rec
            time.sleep(0.4)
        except Exception:       # noqa: BLE001
            continue
    return out


def apply_gnomad(recs, gn):
    for v in recs:
        g = gn.get(vkey(v))
        if not g:
            continue
        v["gnomad"] = g
        if g["present"]:
            vals = [x for x in (g.get("faf95"), g.get("af")) if x is not None and x > 0]
            v["af"] = max(vals) if vals else None
            v["nhomalt"] = g.get("nhomalt")
        else:
            v["af"], v["nhomalt"] = None, None
        if g.get("common_overlap"):
            v["common_overlap"] = g["common_overlap"]


def am_region(chrom, start, end):
    """All AlphaMissense rows in a region -> {(pos, ref, alt): (score, protein_variant, transcript)}."""
    if "alphamissense" not in RES:
        return {}
    txt = sh(f"tabix '{RES['alphamissense']}' {chrom}:{start}-{end}", name="am_region", capture=True)
    out = {}
    for line in txt.splitlines():
        c = line.split("\t")
        if len(c) >= 9:
            out[(int(c[1]), c[2], c[3])] = (float(c[8]), c[7], c[6])
    return out


def bystander_analysis(gene, weaker):
    """Chance-bystander model for the weaker allele of a pair, from gnomAD v4.1 gene-level allele counts."""
    q = ("query($s: String!) { gene(gene_symbol: $s, reference_genome: GRCh38) { variants(dataset: gnomad_r4) { "
         "variant_id consequence exome { ac an } genome { ac an } } } }")
    d = http_json("POST", "https://gnomad.broadinstitute.org/api", json={"query": q, "variables": {"s": gene}}, timeout=180)
    variants = ((d.get("data") or {}).get("gene") or {}).get("variants") or []
    if not variants:
        raise RuntimeError("no gnomAD gene-level data")
    chrom, start, end = GENE_POS[gene][0], GENE_POS[gene][1], GENE_POS[gene][2]
    am = am_region(chrom, start, end)
    cls_map = {"missense": {"missense_variant"}, "stop_gained": {"stop_gained", "frameshift_variant", "splice_donor_variant", "splice_acceptor_variant"},
               "frameshift": {"stop_gained", "frameshift_variant", "splice_donor_variant", "splice_acceptor_variant"},
               "inframe_indel": {"inframe_deletion", "inframe_insertion"}}
    wanted = cls_map.get(weaker["csq"], None)
    q_all = q_strict = 0.0
    n_all = n_strict = 0
    am_w = _num(weaker.get("am"))
    band_lo = max((thr for thr, _ in AM_BANDS if am_w is not None and am_w >= thr), default=None)
    for g in variants:
        if wanted is not None and g["consequence"] not in wanted:
            continue
        ac = sum((g.get(x) or {}).get("ac") or 0 for x in ("exome", "genome"))
        an = sum((g.get(x) or {}).get("an") or 0 for x in ("exome", "genome"))
        af = ac / an if an else 0.0
        if not 0 < af < 1e-4:
            continue
        q_all += af
        n_all += 1
        if weaker["csq"] == "missense" and band_lo is not None:
            _, pos, ref, alt = g["variant_id"].split("-")
            s = am.get((int(pos), ref, alt))
            if s and s[0] >= band_lo:
                q_strict += af
                n_strict += 1
    res = dict(gene=gene, variant_class=weaker["csq"], n_rare_variants=n_all, cumulative_af=q_all,
               carrier_probability=1 - (1 - q_all) ** 2)
    if weaker["csq"] == "missense" and band_lo is not None:
        res.update(am_band_threshold=band_lo, n_rare_at_least_as_damaging=n_strict, cumulative_af_strict=q_strict,
                   carrier_probability_strict=1 - (1 - q_strict) ** 2)
    # P(an unrelated person carries such an allele by chance): the conservative estimate uses every rare variant of the
    # same class; the strict one (only alleles at least as damaging) is reported as a sensitivity bound.
    pb = res["carrier_probability"]
    res["posterior_second_allele"] = {f"prior_{p}": round(p / (p + (1 - p) * pb), 4) for p in (0.2, 0.5, 0.8)}
    return res


def acmg_table(v, extra_points=0.0, extra_label=None):
    rows = [dict(criterion=c, points=p, basis=txt) for c, p, txt in v["evidence"]]
    if extra_points:
        rows.append(dict(criterion=extra_label or "PM3", points=extra_points, basis="partner allele (likely) pathogenic; phase unknown"))
    total = v["points"] + extra_points
    post = posterior(v, extra_points)
    by_post = ("Pathogenic" if post >= 0.99 else "Likely pathogenic" if post >= 0.90 else "Uncertain significance" if post > 0.10
               else "Likely benign" if post > 0.001 else "Benign")
    # the stronger of the posterior class (which carries ClinVar floors / caps) and the point-threshold class
    label = min(by_post, points_class(total), key=ACMG_CLASSES.index) if post > 0.10 else by_post
    return pd.DataFrame(rows), total, post, label


ACMG_CLASSES = ["Pathogenic", "Likely pathogenic", "Uncertain significance", "Likely benign", "Benign"]


def points_class(points):
    """ACMG/AMP class from the naturally scaled point system (Tavtigian 2020): P >= 10, LP 6-9, VUS 0-5, LB -1..-6, B <= -7."""
    return ("Pathogenic" if points >= 10 else "Likely pathogenic" if points >= 6 else "Uncertain significance" if points >= 0
            else "Likely benign" if points >= -6 else "Benign")


def rescore(recs):
    for v in recs:
        inh = set((GENES_DX.get(v["gene"]) or {}).get("inheritance_all") or [])
        lof_ok = v["gene"] in LOF_GENES or bool(inh & {"AR", "XLR", "XL"})
        v["points"], v["floor"], v["cap"], v["evidence"] = variant_evidence(v, lof_ok=lof_ok)
        v["p_path"] = posterior(v)


with stage("deep annotation of the short list"):
    SHORT, TOP_GENES = shortlist_variants(HYPS, RECS)
    log(f"short list: {len({vkey(v) for v in SHORT})} alleles; leading genes: {TOP_GENES}")
    try:
        VEP = vep_annotate(SHORT)
        log(f"Ensembl VEP REST: {apply_vep(RECS, VEP)} variant rows annotated")
        S["vep_ok"] = True
    except Exception as e:      # noqa: BLE001
        log(f"Ensembl VEP REST unavailable ({str(e)[:160]}); continuing with offline annotations")
        S["vep_ok"] = False
    try:
        GN = gnomad_exact(SHORT)
    except Exception as e:      # noqa: BLE001
        log(f"gnomAD remote tabix unavailable ({str(e)[:160]}); using the GraphQL API")
        GN = gnomad_graphql(SHORT)
    apply_gnomad(RECS, GN)
    S["gnomad_exact_n"] = len(GN)
    rescore(RECS)
    HYPS = build_hypotheses(RECS, GENES_DX, PHENO_TOP, W)
    S["ranking_stage"] = "after deep annotation (VEP, gnomAD v4.1 exact)"
    S["top_gene"] = HYPS[0]["gene"]
    md("**Top hypotheses after deep annotation**\n\n" + df_md(public_table(hyp_table(HYPS, 10))))

In [ ]:
def analyse_leader():
    """Evidence tables, panel status and chance-bystander analysis for the current leading hypothesis."""
    global TOP
    TOP = HYPS[0]
    S["top_key"] = sorted(TOP["key"])
    S["top_gene"] = TOP["gene"]
    S["top"] = dict(gene=TOP["gene"], model=TOP["model"], phase=TOP["phase"], epcr=round(TOP["epcr"], 4), variants=[])
    info = GENES_DX.get(TOP["gene"]) or {}
    S["top"].update(disease=info.get("disease_name"), dx_disease=info.get("dx_disease"), inheritance=info.get("inheritance"),
                    pheno_score=round(info.get("pheno", 0.0), 3), pheno_rank_blind=GENE_RANK_BLIND.get(TOP["gene"]))
    ordered = sorted(TOP["variants"], key=lambda v: -v["p_path"])
    for i, v in enumerate(ordered):
        extra = TOP.get("pm3", 0.0) if (TOP["model"] == "compound_het" and i == 1) else (1.0 if TOP["model"] in ("homozygous", "hemizygous") else 0.0)
        tab, total, post, label = acmg_table(v, extra, "PM3" if TOP["model"] == "compound_het" else "PM3 (homozygous)")
        g = v.get("gnomad") or {}
        S["top"]["variants"].append(dict(
            key=f"{v['chrom']}:{int(v['pos'])} {v['ref']}>{v['alt']}", chrom=v["chrom"], pos=int(v["pos"]), ref=v["ref"], alt=v["alt"],
            csq=v["csq"], hgvsc=v.get("hgvsc", ""), hgvsp=v.get("hgvsp", ""), mane=v.get("mane_refseq", ""), exon=v.get("exon", ""),
            zyg=v["zyg"], ad=f"{v.get('ad_ref')}/{v.get('ad_alt')}", gq=v.get("gq"), filt=v.get("filt"), pid=v.get("pid"),
            clnsig=v.get("clnsig", ""), clnrevstat=v.get("clnrevstat", ""), rsids=v.get("rsids", []), pubmed=v.get("pubmed", []),
            am=_num(v.get("am")), revel=_num(v.get("revel")), cadd=_num(v.get("cadd")), spliceai=_num(v.get("spliceai")),
            loftee=v.get("loftee", ""), nmd_escape=v.get("nmd_escape"), frac_lost=_num(v.get("frac_lost")), aa_pos=v.get("aa_pos"),
            gnomad=g, points=total, posterior=round(post, 4), classification=label, evidence=tab.to_dict("records")))
        md(f"**{v['gene']} {v.get('hgvsc') or ''} {v.get('hgvsp') or ''}** — `{v['chrom']}:{int(v['pos'])} {v['ref']}>{v['alt']}` · "
           f"{total:g} points · posterior {post:.3f} · **{label}** (automated)\n\n" + df_md(tab))
    try:
        res = http_json("GET", f"https://panelapp.genomicsengland.co.uk/api/v1/genes/{TOP['gene']}/?format=json").get("results", [])
        S["panelapp"] = [dict(panel=x["panel"]["name"], id=x["panel"]["id"], version=x["panel"].get("version", ""),
                              mode=x.get("mode_of_inheritance", "")) for x in res if str(x.get("confidence_level")) == "3"]
        log(f"PanelApp: {TOP['gene']} is a green gene on {len(S['panelapp'])} Genomics England panels")
    except Exception as e:      # noqa: BLE001
        log(f"PanelApp lookup skipped ({str(e)[:120]})")
    if TOP["model"] == "compound_het":
        try:
            S["bystander"] = bystander_analysis(TOP["gene"], ordered[1])
            b = S["bystander"]
            log(f"chance-bystander: P(an unrelated person carries a rare {b['variant_class']} variant in {b['gene']}) = "
                f"{b['carrier_probability']:.4f}" + (f"; at least as damaging by AlphaMissense: {b['carrier_probability_strict']:.4f}"
                                                     if "carrier_probability_strict" in b else "")
                + f"; posterior that the weaker allele is the second causal allele: {b['posterior_second_allele']}")
        except Exception as e:      # noqa: BLE001
            log(f"chance-bystander analysis skipped ({str(e)[:160]})")


with stage("leading hypothesis: evidence tables, panel status, chance-bystander analysis"):
    analyse_leader()

In [ ]:
with stage("write + validate submission (after deep annotation)", required=True):
    SUB_ROWS, parsed = write_submission(HYPS)
    chk = verify_in_source([v for h in SUB_ROWS for v in h["variants"]])
    S["source_check"] = chk
    if not DEMO:
        assert all(c["found"] and not c["synthetic"] for c in chk), f"submitted allele not found in the source VCF: {chk}"
    else:
        S["demo_score"] = official_score(parsed, frozenset(SPIKE_CASES[0]["variants"]))
        log(f"DEMO self-score against the spiked truth (organisers' scorer): {S['demo_score']}")
    md("**Submission rows**\n\n" + df_md(pd.read_csv(SUBMISSION_CSV, dtype=str, keep_default_na=False)))

## 11 · Deep-learning evidence on the GPUs

Three orthogonal models are applied to the leading genes only (a few dozen alleles), each on its own device:

* **SpliceAI** (Jaganathan 2019; five-model ensemble, 10-kb context, ±500 bp scan) — scores every rare non-coding
  and synonymous allele in the leading genes, so that a deep-intronic or cryptic-splice second hit is not missed.
  Scores ≥ 0.2 enter the evidence model; alleles with ≤ 0.1 keep their benign-leaning default.
* **ESM-2** (650 M-parameter protein language model; masked-marginal log-likelihood ratios) — an in-silico saturation
  mutagenesis of the leading protein. The candidate missense change is placed on the distribution of all 19·L
  substitutions, and against ClinVar-pathogenic and population-observed missense variants of the same gene.
  It is reported as orthogonal support and is not added to the ACMG points (one calibrated predictor only).
* **AlphaFold** structural context — predicted-structure confidence, burial and 3D distance from the candidate residue
  to known pathogenic missense positions.

In [ ]:
def ensure_modules(mods):
    """Install missing Python modules at run time ({import name: pip arguments}); used on images without them
    (e.g. a TPU host has no CUDA builds and may lack TensorFlow or transformers)."""
    import importlib.util
    for name, spec in mods.items():
        if importlib.util.find_spec(name) is None:
            log(f"installing {spec} (module {name} not in this image)")
            sh(f"'{sys.executable}' -m pip install -q --no-input {spec}", name=f"pip_{name}", timeout=1200)
            importlib.invalidate_caches()


def load_spliceai_models():
    """The five published SpliceAI Keras models (weights installed at run time from PyPI; CC BY-NC 4.0)."""
    import importlib.util
    ensure_modules({"tensorflow": "tensorflow-cpu tf-keras"})
    if importlib.util.find_spec("spliceai") is None:
        sh(f"'{sys.executable}' -m pip install -q --no-input --no-deps spliceai", name="pip_spliceai", timeout=600)
        importlib.invalidate_caches()
    base = Path(importlib.util.find_spec("spliceai").submodule_search_locations[0]) / "models"
    paths = [base / f"spliceai{i}.h5" for i in range(1, 6)]
    import tensorflow as tf
    gpus = tf.config.list_physical_devices("GPU")
    if len(gpus) > 1:                       # leave GPU 0 to PyTorch
        tf.config.set_visible_devices(gpus[-1], "GPU")
    for g in tf.config.get_visible_devices("GPU"):
        try:
            tf.config.experimental.set_memory_growth(g, True)
        except Exception:       # noqa: BLE001
            pass
    try:
        import keras
        models = [keras.models.load_model(str(p), compile=False) for p in paths]
    except Exception as e:      # noqa: BLE001 - legacy H5 under Keras 3: fall back to tf_keras
        log(f"keras loader failed ({type(e).__name__}); using tf_keras")
        import tf_keras
        models = [tf_keras.models.load_model(str(p), compile=False) for p in paths]
    return models, len(tf.config.get_visible_devices("GPU"))


_OH = np.zeros((256, 4), dtype=np.float32)
for _i, _b in enumerate(b"ACGT"):
    _OH[_b, _i] = 1.0


def one_hot(seq):
    return _OH[np.frombuffer(seq.upper().encode("ascii"), dtype=np.uint8)]


def spliceai_delta(models, fasta, chrom, pos, ref, alt, strand, dist=500):
    """Raw (unmasked) SpliceAI delta scores for one allele, following the reference implementation."""
    cov = 2 * dist + 1
    wid = 10000 + cov
    start = pos - wid // 2 - 1
    if start < 0 or len(ref) > dist or len(alt) > dist:
        return None
    seq = str(fasta[chrom][start:start + wid]).upper()
    if len(seq) != wid or seq[wid // 2:wid // 2 + len(ref)] != ref.upper():
        return None
    x_ref, x_alt = seq, seq[:wid // 2] + alt.upper() + seq[wid // 2 + len(ref):]
    xr, xa = one_hot(x_ref)[None], one_hot(x_alt)[None]
    if strand == "-":
        xr, xa = xr[:, ::-1, ::-1], xa[:, ::-1, ::-1]
    if xr.shape == xa.shape:            # SNV / MNV: reference and alternative in one batch
        y = np.mean([np.asarray(m(np.concatenate([xr, xa]), training=False)) for m in models], axis=0)
        yr, ya = y[:1], y[1:]
    else:
        yr = np.mean([np.asarray(m(xr, training=False)) for m in models], axis=0)
        ya = np.mean([np.asarray(m(xa, training=False)) for m in models], axis=0)
    if strand == "-":
        yr, ya = yr[:, ::-1], ya[:, ::-1]
    rl, al = len(ref), len(alt)
    if rl > 1 and al == 1:          # deletion: pad the alternative track
        ya = np.concatenate([ya[:, :cov // 2 + al], np.zeros((1, rl - al, 3), dtype=ya.dtype), ya[:, cov // 2 + al:]], axis=1)
    elif rl == 1 and al > 1:        # insertion: collapse the inserted positions
        ya = np.concatenate([ya[:, :cov // 2], ya[:, cov // 2:cov // 2 + al].max(axis=1, keepdims=True), ya[:, cov // 2 + al:]], axis=1)
    if yr.shape != ya.shape:
        return None
    d_acc, d_don = ya[0, :, 1] - yr[0, :, 1], ya[0, :, 2] - yr[0, :, 2]
    res = dict(DS_AG=float(d_acc.max()), DS_AL=float((-d_acc).max()), DS_DG=float(d_don.max()), DS_DL=float((-d_don).max()),
               DP_AG=int(d_acc.argmax() - cov // 2), DP_AL=int((-d_acc).argmax() - cov // 2),
               DP_DG=int(d_don.argmax() - cov // 2), DP_DL=int((-d_don).argmax() - cov // 2))
    res["max"] = max(res["DS_AG"], res["DS_AL"], res["DS_DG"], res["DS_DL"])
    return res


with stage("SpliceAI on the leading genes (GPU)"):
    if not CFG["RUN_GPU_MODELS"]:
        raise RuntimeError("disabled in CFG")
    from pyfaidx import Fasta
    todo = {}
    for v in RECS:
        if v["gene"] in TOP_GENES and v["gene"] in GENE_POS and v["csq"] not in LOF_CSQ:
            todo[(vkey(v), v["gene"])] = v
    todo = sorted(todo.values(), key=lambda v: (TOP_GENES.index(v['gene']), -v['p_path']))[:200]
    models, n_tf_gpu = load_spliceai_models()
    fa = Fasta(RES["ref"], as_raw=True, sequence_always_upper=True)
    t0, n_scored, n_new = time.time(), 0, 0
    for v in todo:
        r = spliceai_delta(models, fa, v["chrom"], int(v["pos"]), v["ref"], v["alt"], GENE_POS[v["gene"]][3])
        if r is None:
            continue
        n_scored += 1
        v["spliceai_custom"] = round(r["max"], 3)
        v["spliceai_custom_detail"] = r
        prev = _num(v.get("spliceai"))
        if prev is None or r["max"] > prev:         # wider scan window than the pre-computed scores: keep the larger delta
            v["spliceai"] = round(r["max"], 3)
            n_new += 1
    S["spliceai"] = dict(n_alleles=n_scored, seconds=round(time.time() - t0, 1), device="GPU" if n_tf_gpu else "CPU",
                         n_ge_0p2=int(sum(1 for v in todo if (v.get("spliceai_custom") or 0) >= 0.2)))
    log(f"SpliceAI: {n_scored} alleles in {TOP_GENES} scored on {S['spliceai']['device']} in {S['spliceai']['seconds']}s; "
        f"{S['spliceai']['n_ge_0p2']} with delta >= 0.2")
    del models
    rescore(RECS)
    HYPS = build_hypotheses(RECS, GENES_DX, PHENO_TOP, W)
    S["ranking_stage"] = "after deep annotation and SpliceAI"
    SUB_ROWS, parsed = write_submission(HYPS)
    if sorted(HYPS[0]["key"]) != S.get("top_key"):
        log("the leading hypothesis changed after SpliceAI: rebuilding its evidence tables")
        analyse_leader()
    S["top_gene"] = HYPS[0]["gene"]
    md("**Top hypotheses after SpliceAI**\n\n" + df_md(public_table(hyp_table(HYPS, 6))[["rank", "EPCR", "gene", "model", "variants", "evidence"]]))

In [ ]:
AA20 = "ACDEFGHIKLMNPQRSTVWY"


def esm_landscape(seq, model_name="facebook/esm2_t33_650M_UR50D", batch=8, devices=None, budget_s=None):
    """Masked-marginal log-likelihood ratios for every substitution: array [L, 20] in AA20 order (wild type = 0).
    One model replica per GPU, positions split between them; or one float32 replica on the CPU of a large host, which
    gives up (TimeoutError) when the first batches project a run longer than `budget_s`."""
    import torch
    from transformers import AutoTokenizer, EsmForMaskedLM
    tok = AutoTokenizer.from_pretrained(model_name)
    devices = devices or [f"cuda:{i}" for i in range(torch.cuda.device_count())]
    if devices == ["cpu"]:
        torch.set_num_threads(NCPU)
    ids = tok(seq, return_tensors="pt")["input_ids"][0]         # <cls> seq <eos>
    if len(ids) != len(seq) + 2:
        raise RuntimeError("ESM tokenisation does not map one token per residue")
    aa_ids = torch.tensor([tok.convert_tokens_to_ids(a) for a in AA20])
    L = len(seq)
    out = np.zeros((L, 20), dtype=np.float32)
    replicas = [EsmForMaskedLM.from_pretrained(model_name).to(d).eval() for d in devices]
    replicas = [r.half() if d.startswith("cuda") else r for r, d in zip(replicas, devices)]
    shards = [list(range(L))[i::len(devices)] for i in range(len(devices))]
    errors = []

    def worker(m, dev, pos_list):
        try:
            t_start = time.time()
            with torch.no_grad():
                for i in range(0, len(pos_list), batch):
                    if budget_s and i == 2 * batch:
                        projected = (time.time() - t_start) / i * len(pos_list)
                        if projected > budget_s:
                            raise TimeoutError(f"projected {projected / 60:.0f} min on {dev} exceeds the {budget_s / 60:.0f}-min budget")
                    chunk = pos_list[i:i + batch]
                    x = ids.repeat(len(chunk), 1).to(dev)
                    for j, p in enumerate(chunk):
                        x[j, p + 1] = tok.mask_token_id
                    logits = m(input_ids=x).logits.float()
                    for j, p in enumerate(chunk):
                        lp = torch.log_softmax(logits[j, p + 1], dim=-1)[aa_ids.to(dev)].cpu().numpy()
                        out[p] = lp - lp[AA20.index(seq[p])] if seq[p] in AA20 else lp
        except Exception as e:      # noqa: BLE001
            errors.append(e)

    threads = [threading.Thread(target=worker, args=(m, d, sh_)) for m, d, sh_ in zip(replicas, devices, shards)]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    del replicas
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if errors:
        raise errors[0]
    return out, devices


_ESM_CPU = {}
ESM_JOB = r"""
import json, sys, time
import numpy as np
import torch
from transformers import AutoTokenizer, EsmForMaskedLM
cfg = json.load(open(sys.argv[1]))
AA20 = "ACDEFGHIKLMNPQRSTVWY"
torch.set_num_threads(cfg["threads"])
tok = AutoTokenizer.from_pretrained(cfg["model"])
m = EsmForMaskedLM.from_pretrained(cfg["model"]).eval()
seq = cfg["seq"]
ids = tok(seq, return_tensors="pt")["input_ids"][0]
assert len(ids) == len(seq) + 2, "tokenisation is not one token per residue"
aa_ids = torch.tensor([tok.convert_tokens_to_ids(a) for a in AA20])


def score(positions, budget=None):
    res, t0, B = {}, time.time(), cfg["batch"]
    with torch.no_grad():
        for i in range(0, len(positions), B):
            if budget and i == 2 * B and (time.time() - t0) / i * len(positions) > budget:
                return None, (time.time() - t0) / i * len(positions)
            chunk = positions[i:i + B]
            x = ids.repeat(len(chunk), 1)
            for j, p in enumerate(chunk):
                x[j, p + 1] = tok.mask_token_id
            lg = m(input_ids=x).logits.float()
            for j, p in enumerate(chunk):
                lp = torch.log_softmax(lg[j, p + 1], -1)[aa_ids].numpy()
                res[p] = lp - lp[AA20.index(seq[p])] if seq[p] in AA20 else lp
    return res, None


full, proj = score(list(range(len(seq))), cfg["budget_s"]) if cfg["full"] else (None, None)
if full is not None:
    np.savez(sys.argv[2], llr=np.stack([full[p] for p in range(len(seq))]), full=1)
else:
    rows, _ = score(sorted(set(cfg["positions"])))
    np.savez(sys.argv[2], pos=np.array(sorted(rows)), rows=np.stack([rows[p] for p in sorted(rows)]), full=0,
             projected_s=proj if proj else -1.0)
"""


def esm_subprocess(wseq, positions0, full, budget_s=None, model_name="facebook/esm2_t33_650M_UR50D"):
    """ESM-2 in a child process on the CPU: the full landscape (within `budget_s`, else only `positions0`) or only
    `positions0` (0-based window indices). Returns dict(llr=[L,20] or None, rows={pos0: [20]}, projected_s)."""
    job, cfg, out = WORK / "esm_job.py", WORK / "esm_job.json", WORK / "esm_job.npz"
    job.write_text(ESM_JOB)
    cfg.write_text(json.dumps(dict(model=model_name, seq=wseq, positions=[int(p) for p in positions0], full=bool(full),
                                   budget_s=budget_s, threads=NCPU, batch=8)))
    out.unlink(missing_ok=True)
    env = {**os.environ, "TORCH_DEVICE_BACKEND_AUTOLOAD": "0", "PJRT_DEVICE": "CPU", "CUDA_VISIBLE_DEVICES": ""}
    sh(f"'{sys.executable}' '{job}' '{cfg}' '{out}'", name="esm_subprocess", env=env,
       timeout=int((budget_s or 0) + 3600))
    z = np.load(out)
    if int(z["full"]):
        return dict(llr=z["llr"], rows={}, projected_s=None)
    return dict(llr=None, rows={int(p): r for p, r in zip(z["pos"], z["rows"])}, projected_s=float(z["projected_s"]))


def esm_single(seq, pos0, model_name="facebook/esm2_t33_650M_UR50D"):
    """CPU fallback: one masked position only (the model is loaded once and cached; child process without a GPU)."""
    if N_GPU == 0:
        return esm_subprocess(seq, [pos0], full=False, model_name=model_name)["rows"][pos0]
    import torch
    from transformers import AutoTokenizer, EsmForMaskedLM
    if model_name not in _ESM_CPU:
        _ESM_CPU[model_name] = (AutoTokenizer.from_pretrained(model_name), EsmForMaskedLM.from_pretrained(model_name).eval())
    tok, m = _ESM_CPU[model_name]
    ids = tok(seq, return_tensors="pt")["input_ids"]
    ids[0, pos0 + 1] = tok.mask_token_id
    with torch.no_grad():
        lp = torch.log_softmax(m(input_ids=ids).logits[0, pos0 + 1].float(), dim=-1)
    lp = lp[[tok.convert_tokens_to_ids(a) for a in AA20]].numpy()
    return lp - lp[AA20.index(seq[pos0])]


with stage("ESM-2 missense landscape of the leading protein (GPU)"):
    if not CFG["RUN_GPU_MODELS"]:
        raise RuntimeError("disabled in CFG")
    TOP = HYPS[0]
    mis = [v for v in TOP["variants"] if v["csq"] == "missense"]
    if not mis:
        raise RuntimeError("the leading hypothesis has no missense allele")
    v = mis[0]
    tx = TX[GENE_TX[v["gene"]][0]]
    seq = PROT.get(tx["protein"], "").rstrip("*")
    m = re.match(r"p\.([A-Za-z]{3})(\d+)([A-Za-z]{3})", v.get("hgvsp") or "")
    inv = {b: a for a, b in AA3.items()}
    if not (seq and m):
        raise RuntimeError("protein sequence or protein change unavailable")
    wt, pos1, mt = inv.get(m.group(1), "?"), int(m.group(2)), inv.get(m.group(3), "?")
    if mt not in AA20 or pos1 > len(seq) or seq[pos1 - 1] != wt:
        raise RuntimeError("protein sequence does not match the annotated protein change")
    ensure_modules({"torch": "torch --index-url https://download.pytorch.org/whl/cpu", "transformers": "transformers"})
    t0 = time.time()
    # ESM-2 was trained on <= 1,022 residues: longer proteins are scored in a 1,022-residue window around the change
    off = 0 if len(seq) <= 1022 else max(0, min(pos1 - 1 - 511, len(seq) - 1022))
    wseq = seq[off:off + 1022]
    LLR, cpu_rows = None, {}
    if N_GPU > 0:
        LLR, devs = esm_landscape(wseq)
    else:
        need = sorted({int(_num(x["aa_pos"])) - 1 - off for x in TOP["variants"] if _num(x.get("aa_pos")) is not None
                       and 0 <= int(_num(x["aa_pos"])) - 1 - off < len(wseq)} | {pos1 - 1 - off})
        r = esm_subprocess(wseq, need, full=NCPU >= 32, budget_s=CFG.get("ESM_CPU_BUDGET_MIN", 40) * 60)
        LLR, devs = r["llr"], [f"cpu x{NCPU} (child process)"]
        cpu_rows = {p0 + 1 + off: row for p0, row in r["rows"].items()}
        if LLR is None and NCPU >= 32:
            log(f"ESM-2 full landscape skipped on CPU (projected {r['projected_s'] / 60:.0f} min); scored positions {sorted(cpu_rows)}")
    if LLR is not None:
        cand = float(LLR[pos1 - 1 - off, AA20.index(mt)])
        flat = LLR.ravel()
        flat = flat[flat != 0]
        ESM_CTX = dict(gene=v["gene"], seq=seq, off=off, wseq=wseq, llr=LLR)       # reused by the allele-level triage (Track 2)
        S["esm"] = dict(model="ESM-2 650M (esm2_t33_650M_UR50D)", protein=tx["protein"], length=len(seq),
                        window=f"{off + 1}-{off + len(wseq)}", change=f"{wt}{pos1}{mt}",
                        llr=round(cand, 2), percentile_more_damaging_than=round(float((flat > cand).mean()) * 100, 1),
                        devices=devs, seconds=round(time.time() - t0, 1))
    else:
        row = cpu_rows.get(pos1)
        if row is None:
            row = esm_single(wseq, pos1 - 1 - off)
        cand = float(row[AA20.index(mt)])
        ESM_CTX = dict(gene=v["gene"], seq=seq, off=off, wseq=wseq, llr=None, rows={**cpu_rows, pos1: row})  # CPU: single positions
        S["esm"] = dict(model="ESM-2 650M (single position, CPU)", protein=tx["protein"], length=len(seq), change=f"{wt}{pos1}{mt}",
                        llr=round(cand, 2), seconds=round(time.time() - t0, 1))
    log(f"ESM-2: {S['esm']}")
    if LLR is not None:
        # reference sets from public data: ClinVar P/LP missense in this gene and population-observed missense (gnomAD)
        chrom, gs, ge = GENE_POS[v["gene"]][0], GENE_POS[v["gene"]][1], GENE_POS[v["gene"]][2]
        am_map = am_region(chrom, gs, ge)

        def llr_of(pv):
            mm = re.match(r"([A-Z])(\d+)([A-Z])$", pv)
            if not mm or not (off < int(mm.group(2)) <= off + len(wseq)) or seq[int(mm.group(2)) - 1] != mm.group(1) or mm.group(3) not in AA20:
                return None
            return float(LLR[int(mm.group(2)) - 1 - off, AA20.index(mm.group(3))])

        cvg = CLINVAR_PLP[(CLINVAR_PLP.gene == v["gene"]) & CLINVAR_PLP.is_missense]
        plp = [llr_of(am_map[(r.pos, r.ref, r.alt)][1]) for r in cvg.itertuples() if (r.pos, r.ref, r.alt) in am_map]
        plp = [x for x in plp if x is not None]
        S["esm"]["clinvar_plp_missense_n"] = len(plp)
        S["esm"]["clinvar_plp_median_llr"] = round(float(np.median(plp)), 2) if plp else None
        fig, ax = plt.subplots(figsize=(7.2, 3.0))
        ax.hist(flat, bins=60, color=VIZ["neutral"], label=f"all {len(flat):,} possible substitutions")
        ymax = ax.get_ylim()[1]
        if plp:
            ax.scatter(plp, np.full(len(plp), ymax * 0.06), s=26, color=VIZ["s2"], zorder=3, linewidths=0,
                       label=f"ClinVar P/LP missense in {v['gene']} (n={len(plp)})")
        ax.axvline(cand, color=VIZ["s1"], linewidth=2, label=f"candidate {wt}{pos1}{mt} (LLR {cand:.1f})")
        ax.set_ylim(0, ymax * 1.05)
        ax.set_xlabel("ESM-2 masked-marginal log-likelihood ratio (more negative = less tolerated)")
        ax.set_ylabel("substitutions")
        ax.set_title(f"{v['gene']} in-silico saturation mutagenesis (ESM-2): where the candidate change falls")
        ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=3)
        save_fig(fig, "esm_landscape")

In [ ]:
with stage("AlphaFold structural context of the missense allele"):
    TOP = HYPS[0]
    mis = [v for v in TOP["variants"] if v["csq"] == "missense" and _num(v.get("aa_pos")) is not None]
    if not mis:
        raise RuntimeError("the leading hypothesis has no missense allele")
    v = mis[0]
    res_no = int(_num(v["aa_pos"]))
    u = http_json("GET", "https://rest.uniprot.org/uniprotkb/search",
                  params={"query": f"gene_exact:{v['gene']} AND organism_id:9606 AND reviewed:true", "format": "json",
                          "fields": "accession,ft_domain,ft_region,ft_variant,length"})["results"][0]
    acc = u["primaryAccession"]
    feats = [dict(type=f["type"], name=f.get("description", ""), start=f["location"]["start"]["value"], end=f["location"]["end"]["value"])
             for f in u.get("features", []) if f["type"] in ("Domain", "Region")]
    here = [f for f in feats if f["start"] and f["end"] and f["start"] <= res_no <= f["end"]]
    pdb_url = http_json("GET", f"https://alphafold.ebi.ac.uk/api/prediction/{acc}")[0]["pdbUrl"]
    pdb = HTTP.get(pdb_url, timeout=120).text
    ca, cb = {}, {}
    for line in pdb.splitlines():
        if line.startswith("ATOM") and line[12:16].strip() in ("CA", "CB"):
            r, xyz_ = int(line[22:26]), np.array([float(line[30:38]), float(line[38:46]), float(line[46:54])])
            if line[12:16].strip() == "CA":
                ca[r] = (xyz_, float(line[60:66]))
                cb.setdefault(r, xyz_)          # glycine: CA stands in for CB
            else:
                cb[r] = xyz_
    if res_no not in ca:
        raise RuntimeError("residue not in the AlphaFold model")
    xyz, plddt = ca[res_no]
    dists = {r: float(np.linalg.norm(c - xyz)) for r, (c, _) in ca.items()}
    chrom, gs, ge = GENE_POS[v["gene"]][0], GENE_POS[v["gene"]][1], GENE_POS[v["gene"]][2]
    am_map = am_region(chrom, gs, ge)
    cvg = CLINVAR_PLP[(CLINVAR_PLP.gene == v["gene"]) & CLINVAR_PLP.is_missense]
    plp_res = sorted({int(re.sub(r"\D", "", am_map[(r.pos, r.ref, r.alt)][1])) for r in cvg.itertuples() if (r.pos, r.ref, r.alt) in am_map} - {res_no})
    near = sorted(((r, round(dists[r], 1)) for r in plp_res if r in dists), key=lambda x: x[1])[:5]
    # disease-annotated missense variants curated by UniProt (literature-reported patient alleles, e.g. "in MVA1; ...")
    dis = []
    for f in u.get("features", []):
        alt_ = (f.get("alternativeSequence") or {})
        d_ = f.get("description", "")
        m_ = re.match(r"in ([A-Z][A-Z0-9-]{1,});?", d_)
        one_aa = len(alt_.get("alternativeSequences") or []) == 1 and f["location"]["start"]["value"] == f["location"]["end"]["value"]
        somatic = re.search(r"cancer|tumou?r|carcinoma|cell line|somatic", d_, re.I)
        if f["type"] == "Natural variant" and m_ and one_aa and not somatic:
            r = int(f["location"]["start"]["value"])
            if r != res_no and r in dists:
                dis.append(dict(residue=r, change=f"{alt_.get('originalSequence')}{r}{alt_['alternativeSequences'][0]}",
                                disease=m_.group(1), distance_A=round(dists[r], 1),
                                same_domain=any(f2["type"] == "Domain" and f2["start"] <= r <= f2["end"] for f2 in here)))
    dis.sort(key=lambda x: x["distance_A"])

    cb_ids = np.array(sorted(cb))
    cb_xyz = np.stack([cb[r] for r in cb_ids])
    n_within = ((np.linalg.norm(cb_xyz[:, None] - cb_xyz[None], axis=-1) <= 10.0).sum(1) - 1)
    BURIAL = dict(zip(cb_ids.tolist(), n_within.tolist()))      # CB neighbours within 10 Å (a burial proxy)

    def burial(r):
        return BURIAL[r]
    dom_res = [r for f in here if f["type"] == "Domain" for r in range(f["start"], f["end"] + 1) if r in cb]
    mane_len = len(PROT.get(TX[GENE_TX[v["gene"]][0]]["protein"], "").rstrip("*")) if v["gene"] in GENE_TX else 0
    AF_CTX = dict(gene=v["gene"], acc=acc, ca=ca, cb=cb, burial=BURIAL, feats=feats, disease_missense=dis,      # reused by Track 2
                  numbering_matches_mane=bool(mane_len) and max(ca) == mane_len)
    if not AF_CTX["numbering_matches_mane"]:
        log(f"AlphaFold/UniProt length {max(ca)} differs from the MANE protein ({mane_len}): residue-level reuse in Track 2 disabled")
    S["structure"] = dict(uniprot=acc, model=pdb_url.split("/")[-1], residue=res_no, plddt=round(plddt, 1),
                          neighbours_within_10A=int(sum(1 for d in dists.values() if 0 < d <= 10.0)),
                          domain="; ".join(f"{f['name']} ({f['start']}-{f['end']})" for f in here) or "none annotated",
                          nearest_clinvar_plp_missense=[dict(residue=r, distance_A=d) for r, d in near],
                          disease_missense_uniprot=dis[:6], n_disease_missense_uniprot=len(dis),
                          burial_cb_10A=burial(res_no),
                          burial_percentile_in_domain=(round(float(np.mean([burial(r) < burial(res_no) for r in dom_res])) * 100)
                                                       if dom_res else None))
    log(f"AlphaFold context: {S['structure']}")

## 12 · Phase of the leading pair: what the data can and cannot say

A compound-heterozygous diagnosis needs the two alleles on opposite haplotypes. With one short-read genome and no
parents, phase is only observable when both sites sit on the same read pair, or can be chained through heterozygous
sites that do. This section measures that directly instead of assuming it:

1. read-backed phase tags from the caller (`PID` / `PGT`);
2. the heterozygous sites of the proband around and between the two alleles, and the largest gap that reads would
   have to bridge;
3. if the alleles cannot be linked, an explicit statement of what was inferred instead (a model-based probability,
   not a measurement) and of the laboratory test that would settle it.

In [ ]:
def phase_dossier(hyp, max_link=600):
    a, b = sorted(hyp["variants"], key=lambda v: int(v["pos"]))
    chrom, lo, hi = a["chrom"], int(a["pos"]), int(b["pos"])
    dist = hi - lo
    raw_chrom = chrom if S.get("vcf_chr_style", True) else chrom[3:]
    out = dict(distance_bp=dist, caller_phase=hyp["phase"], pid_a=a.get("pid"), pid_b=b.get("pid"))
    txt = sh(f"bcftools query -r {raw_chrom}:{max(1, lo - 2000)}-{hi + 2000} -i 'GT=\"het\"' -f '%POS\\t%REF\\t%ALT\\t%FILTER[\\t%GT]\\n' '{S['vcf_raw']}'",
             name="phase_hets", capture=True)
    hets = sorted({int(l.split("\t")[0]) for l in txt.splitlines() if l.strip()} | {lo, hi})
    between = [p for p in hets if lo <= p <= hi]
    gaps = np.diff(between) if len(between) > 1 else np.array([dist])
    out.update(n_het_sites_between=len(between) - 2, largest_gap_bp=int(gaps.max()), max_read_link_bp=max_link,
               nearest_het_to_a_bp=int(min([abs(p - lo) for p in hets if p != lo], default=10 ** 9)),
               nearest_het_to_b_bp=int(min([abs(p - hi) for p in hets if p != hi], default=10 ** 9)))
    if hyp["phase"] in ("cis", "trans"):
        out["verdict"] = f"read-backed phase from the caller: {hyp['phase']}"
        out["phaseable_by_reads"] = True
    elif gaps.max() <= max_link:
        out["verdict"] = ("the two alleles can be chained through intervening heterozygous sites; read-level chaining is attempted "
                          "in the read-level module")
        out["phaseable_by_reads"] = True
    else:
        out["verdict"] = (f"not phaseable with short reads: the alleles are {dist:,} bp apart and the chain of heterozygous sites is "
                          f"broken by a gap of {int(gaps.max()):,} bp (read pairs link at most ~{max_link} bp)")
        out["phaseable_by_reads"] = False
    amplicon = dist + 1000
    cpos = [re.search(r"c\.(\d+)[ACGT]>", v.get("hgvsc") or "") for v in (a, b)]
    cdna = abs(int(cpos[0].group(1)) - int(cpos[1].group(1))) if all(cpos) else None
    out["cdna_distance_nt"] = cdna
    out["confirmation"] = ("testing either parent for the two alleles (the standard clinical route)"
                           + (f"; or a single long-range PCR amplicon of ~{amplicon / 1000:.1f} kb spanning both sites, read on a long-read platform"
                              if amplicon <= 20000 else "; or long-read sequencing of the locus")
                           + (f"; or, because the alleles are only {cdna:,} nt apart in the mRNA, one RT-PCR amplicon from the child's cultured "
                              "cells with an NMD inhibitor (see the Track 2 report)" if cdna and cdna < dist / 3 else ""))
    return out


with stage("phase analysis"):
    TOP = HYPS[0]
    if TOP["model"] != "compound_het":
        S["phase"] = dict(verdict=f"not applicable: the leading hypothesis is {TOP['model']}")
        log(S["phase"]["verdict"])
    else:
        hdr_names = sh(f"bcftools index -s '{S['vcf_raw']}'", name="phase_idx", capture=True)
        S["vcf_chr_style"] = "chr1\t" in hdr_names or hdr_names.startswith("chr")
        S["phase"] = phase_dossier(TOP)
        b = S.get("bystander") or {}
        if b:
            S["phase"]["model_based_probability"] = dict(
                statement=("If the gene is the cause and the stronger allele is pathogenic, the weaker allele is either the second "
                           "causal allele (necessarily in trans) or a coincidental bystander while the true second hit is hidden."),
                bystander_probability=round(b.get("carrier_probability_strict", b["carrier_probability"]), 5),
                posterior_second_allele=b["posterior_second_allele"])
        for k, v in S["phase"].items():
            log(f"phase {k}: {v}")

## 13 · Does the method generalise? A spike-in benchmark on a public genome

One solved case says little about a method. The same pipeline is therefore run on synthetic recessive cases built
on the public GIAB HG002 genome: for each case, one ClinVar pathogenic truncating allele and one ClinVar pathogenic
missense allele of a single-gene autosomal-recessive disease are spiked in as an unphased heterozygous pair, and
the "patient" is given six HPO terms sampled from that disease — some replaced by a less specific parent term, plus
one unrelated noise term. The ranking is **blind to the diagnosis** (HPO terms only). Two settings are reported:

* **ClinVar-aware** — the realistic setting for known alleles;
* **ClinVar-blind** — the spiked alleles' ClinVar records are hidden, so they must be found from consequence,
  rarity and the missense predictor alone (a proxy for novel alleles).

The benchmark uses only public data and can be reproduced by anyone without access to the patient genome.

In [ ]:
def run_benchmark(cand_all, pool_all, cases):
    bg = cand_all[~cand_all.rsid.str.startswith("SPIKE_")].reset_index(drop=True)
    pool_bg = pool_all[~pool_all.rsid.str.startswith("SPIKE_")].reset_index(drop=True)
    bg_recs = score_variants(bg, GENES_BLIND, LOF_GENES)
    rows = []
    for cs in cases:
        ids = [f"SPIKE_{cs['case']}_0", f"SPIKE_{cs['case']}_1"]
        sp = cand_all[cand_all.rsid.isin(ids) & (cand_all.gene == cs["gene"])]
        truth = frozenset(cs["variants"])
        found = frozenset(zip(sp.chrom, sp.pos.astype(int), sp.ref, sp.alt))
        if found != truth:      # an allele did not survive normalisation / annotation as a candidate in the intended gene
            rows.append(dict(case=cs["case"], gene=cs["gene"], disease=cs["disease_name"][:60], rank_aware=None, rank_blind=None,
                             epcr_aware=None, epcr_blind=None, top_gene_blind=None,
                             note="spiked allele not annotated as a candidate in the intended gene"))
            continue
        gt, _ = PE.score_genes(cs["hpo"], None, diagnosis=None)
        ptop = max(v["pheno"] for v in gt.values()) or 1.0
        res = {}
        for mode in ("aware", "blind"):
            sp2 = sp.copy()
            if mode == "blind":
                sp2["clnsig"], sp2["clnrevstat"] = "", ""
            recs = bg_recs + score_variants(sp2, gt, LOF_GENES)
            hyps = build_hypotheses(recs, gt, ptop, W)
            r = next((i + 1 for i, h in enumerate(hyps) if h["key"] == truth), None)
            res[mode] = (r, hyps[r - 1]["epcr"] if r else None, hyps[0]["gene"])
        rows.append(dict(case=cs["case"], gene=cs["gene"], disease=cs["disease_name"][:60], rank_aware=res["aware"][0],
                         rank_blind=res["blind"][0], epcr_aware=res["aware"][1], epcr_blind=res["blind"][1],
                         top_gene_blind=res["blind"][2], note=""))
    return pd.DataFrame(rows)


with stage("spike-in benchmark (public genome)"):
    if not CFG["RUN_BENCHMARK"]:
        raise RuntimeError("disabled in CFG")
    if DEMO:
        bench_cand, bench_pool, bench_cases = CAND_ALL, POOL_ALL, SPIKE_CASES
    else:
        hg = fetch(HG002_VCF, WORK / "data" / "HG002.vcf.gz", min_bytes=100_000_000)
        fetch(HG002_VCF + ".tbi", WORK / "data" / "HG002.vcf.gz.tbi", min_bytes=100_000)
        bench_cases = make_spike_cases(60)
        bench_cand, bench_pool, _, _ = annotate_vcf(str(hg), "bench", spikes=bench_cases, sex="unknown")
    BENCH = run_benchmark(bench_cand, bench_pool, bench_cases)
    ok = BENCH[BENCH.rank_aware.notna() | BENCH.rank_blind.notna()]
    n = len(ok)

    def rate(col, k):
        return float((ok[col].fillna(10 ** 6) <= k).mean()) if n else float("nan")

    S["benchmark"] = dict(n_cases=int(n), n_not_evaluable=int(len(BENCH) - n),
                          aware=dict(top1=round(rate("rank_aware", 1), 3), top3=round(rate("rank_aware", 3), 3), top10=round(rate("rank_aware", 10), 3)),
                          blind=dict(top1=round(rate("rank_blind", 1), 3), top3=round(rate("rank_blind", 3), 3), top10=round(rate("rank_blind", 10), 3)),
                          median_epcr_when_top1=round(float(ok.loc[ok.rank_aware == 1, "epcr_aware"].median()), 3) if (ok.rank_aware == 1).any() else None)
    log(f"benchmark on {n} synthetic recessive cases (HPO only, no diagnosis): {S['benchmark']}")
    fig, ax = plt.subplots(figsize=(6.4, 3.0))
    ks, x = [1, 3, 10], np.arange(3)
    for j, (mode, label, col) in enumerate((("aware", "ClinVar-aware", VIZ["s1"]), ("blind", "ClinVar-blind (alleles treated as novel)", VIZ["s2"]))):
        vals = [S["benchmark"][mode][f"top{k}"] * 100 for k in ks]
        bars = ax.bar(x + (j - 0.5) * 0.36, vals, width=0.32, color=col, label=label)
        for b, v in zip(bars, vals):
            ax.text(b.get_x() + b.get_width() / 2, v + 1.5, f"{v:.0f}%", ha="center", va="bottom", fontsize=8, color=VIZ["ink2"])
    ax.set_xticks(x)
    ax.set_xticklabels([f"top {k}" for k in ks])
    ax.set_ylim(0, 112)
    ax.set_ylabel("cases with the true pair recovered (%)")
    ax.set_title(f"Spike-in benchmark: {n} synthetic recessive cases on GIAB HG002")
    ax.grid(axis="x", visible=False)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2)
    save_fig(fig, "benchmark")
    miss = ok[(ok.rank_blind.fillna(10 ** 6) > 1)][["case", "gene", "disease", "rank_aware", "rank_blind", "top_gene_blind"]]
    if len(miss):
        md("**Cases where the true pair was not ranked first in the ClinVar-blind setting**\n\n" + df_md(miss, max_rows=15))

## 14 · Track 2 — from mechanism to approved-drug hypotheses

Two layers, both reproducible from public sources:

1. **Automated network pass (gene-agnostic).** The causal gene's interaction neighbourhood (Open Targets, score ≥ 0.7,
   cross-checked with STRING) is queried for every drug and clinical candidate. Hard gates are applied in order:
   *approved* (maximum clinical stage = approval) → *direction* (does the drug push its target in the direction that
   worsens or relieves a loss-of-function / hypomorphic state?) → *antimitotic or genotoxic mechanism* (a separate caution
   class in a chromosomal-instability disorder). Direction comes from the drug's action type (ChEMBL / Open Targets)
   combined with a short, cited sign map of the network nodes — the only hand-curated input, printed in full.
2. **Literature-curated layer.** Mechanism-level hypotheses that no drug–target database can reach (for example,
   raising the abundance of a protein that has no activator). Every citation is resolved live against Europe PMC
   and printed with its title; a citation that does not resolve is flagged rather than used.

The same code is then run unchanged for the other genes of the disorder's virtual panel, to show it generalises.

In [ ]:
OT_API = "https://api.platform.opentargets.org/api/v4/graphql"
# Sign map: +1 = the node's activity protects in a loss-of-function / hypomorphic state (inhibiting it is harmful);
# -1 = the node's activity lowers the causal protein (inhibiting it could help); 0 = indeterminate. Each entry cites
# its source. Nodes not listed are treated as indeterminate. The causal gene itself is always +1.
SIGN_MAP = {
    "BUB1B": {
        "SIRT2": (+1, "SIRT2 deacetylates BUBR1-K668 and raises BUBR1 abundance; SIRT2 overexpression extends lifespan of BubR1-hypomorphic mice", ["24825348"]),
        "CREBBP": (-1, "CBP acetylates BUBR1-K668, counteracting SIRT2 and promoting BUBR1 loss", ["24825348"]),
        "AURKB": (+1, "Aurora B phosphorylation of BUBR1-S39 is required for protective BUBR1-K250 acetylation", ["41533580"]),
        "KAT2B": (+1, "PCAF acetylates BUBR1-K250, which blocks its APC/C-dependent degradation", ["19407811"]),
        "CDC20": (-1, "APC/C-CDC20 degrades BUBR1 in mitosis", ["19407811"]),
        "PPP2R1A": (+1, "BUBR1's KARD recruits PP2A-B56 to kinetochores and its C-terminal pseudokinase domain promotes that recruitment, so "
                        "PP2A-B56 activity is part of the function the child's missense allele may impair", ["33207204", "27350047", "23079597"]),
        "HDAC1": (0, "opposite effects at K250 (must stay acetylated) and K668 (must stay deacetylated): net direction unknown", ["19407811", "24825348"]),
        "HDAC2": (0, "as HDAC1", ["19407811", "24825348"]),
        "HDAC4": (0, "as HDAC1", ["19407811", "24825348"]),
    },
}
EXTRA_TARGETS = {"BUB1B": ["SIRT2", "CREBBP", "KAT2B"]}          # mechanism nodes from the literature, outside the PPI network
CAUTION_MOA = re.compile(r"tubulin|microtubule|aurora|polo-like|TTK|kinesin|topoisomerase|proteasome|DNA alkylat|DNA cross|"
                         r"DNA polymerase|DNA synthesis|ribonucleotide reductase|thymidylate|DNA intercal", re.I)


def ot(query, variables, timeout=180):
    d = http_json("POST", OT_API, json={"query": query, "variables": variables}, timeout=timeout)
    if d.get("errors"):
        raise RuntimeError(f"Open Targets: {d['errors'][0].get('message', '')[:200]}")
    return d["data"]


def network_drug_pass(gene):
    gid = GENE_POS[gene][4]
    d = ot("""query($id: String!) { meta { apiVersion { x y z } } target(ensemblId: $id) { approvedSymbol isEssential
              pathways { pathwayId pathway }
              interactions(scoreThreshold: 0.7, page: {index: 0, size: 150}) { count rows { targetB { id approvedSymbol } score } } } }""",
           {"id": gid})
    t = d["target"] or {}
    ver = ".".join(str(v) for v in d["meta"]["apiVersion"].values())
    partners = {}
    for r in (t.get("interactions") or {}).get("rows") or []:
        tb = r.get("targetB") or {}
        if tb.get("id", "").startswith("ENSG") and tb.get("approvedSymbol"):
            partners[tb["id"]] = (tb["approvedSymbol"], max(r.get("score") or 0.0, partners.get(tb["id"], ("", 0.0))[1] or 0.0))
    nodes = {gid: (gene, 1.0, "causal gene")}
    for k, (sym, sc) in partners.items():
        nodes.setdefault(k, (sym, sc, "interaction partner"))
    for sym in EXTRA_TARGETS.get(gene, []):
        if sym in GENE_POS:
            nodes.setdefault(GENE_POS[sym][4], (sym, None, "literature mechanism node"))
    rows = []
    ids = list(nodes)
    for i in range(0, len(ids), 50):
        res = ot("""query($ids: [String!]!) { targets(ensemblIds: $ids) { id approvedSymbol
                  drugAndClinicalCandidates { rows { drug { id name maximumClinicalStage
                     mechanismsOfAction { rows { mechanismOfAction actionType targets { approvedSymbol } } } } } } } }""",
                 {"ids": ids[i:i + 50]})
        for tg in res.get("targets") or []:
            sym, score, why = nodes.get(tg["id"], (tg["approvedSymbol"], None, ""))
            for row in (tg.get("drugAndClinicalCandidates") or {}).get("rows") or []:
                dr = row["drug"]
                moas = [m for m in (dr.get("mechanismsOfAction") or {}).get("rows", [])
                        if any(x.get("approvedSymbol") == sym for x in (m.get("targets") or []))] or (dr.get("mechanismsOfAction") or {}).get("rows", [])
                for m in moas[:1]:
                    rows.append(dict(target=sym, node=why, interaction_score=score, drug=dr["name"].title(), chembl=dr["id"],
                                     stage=dr.get("maximumClinicalStage") or "", action=(m.get("actionType") or "").upper(),
                                     mechanism=m.get("mechanismOfAction") or ""))
    df = pd.DataFrame(rows, columns=["target", "node", "interaction_score", "drug", "chembl", "stage", "action", "mechanism"])
    signs = SIGN_MAP.get(gene, {})

    def classify(r):
        if r.stage != "APPROVAL":
            return "not approved"
        if CAUTION_MOA.search(r.mechanism or ""):
            return "CAUTION: antimitotic / genotoxic mechanism"
        s = +1 if r.target == gene else signs.get(r.target, (0, "", []))[0]
        a = +1 if r.action in ("ACTIVATOR", "AGONIST", "POSITIVE ALLOSTERIC MODULATOR", "STABILISER", "OPENER") else (
            -1 if r.action in ("INHIBITOR", "ANTAGONIST", "BLOCKER", "NEGATIVE ALLOSTERIC MODULATOR", "DEGRADER", "INVERSE AGONIST") else 0)
        if s == 0 or a == 0:
            return "indeterminate direction"
        return "plausibly beneficial direction" if s * a > 0 else "CAUTION: harmful direction"

    if len(df):
        df["verdict"] = df.apply(classify, axis=1)
    else:
        df["verdict"] = []
    summary = dict(gene=gene, open_targets_api=ver, essential=t.get("isEssential"), n_pathways=len(t.get("pathways") or []),
                   n_network_nodes=len(nodes), n_drug_target_rows=int(len(df)),
                   n_approved=int((df.stage == "APPROVAL").sum()) if len(df) else 0,
                   n_direct_drugs=int((df.target == gene).sum()) if len(df) else 0,
                   verdicts=df[df.stage == "APPROVAL"].drop_duplicates(["drug", "verdict"]).verdict.value_counts().to_dict() if len(df) else {},
                   pathways=[p["pathway"] for p in (t.get("pathways") or [])][:8])
    return df, summary


# Literature-curated hypotheses (citations resolved live below). Grades: A human trial in a relevant population;
# B in vivo in the same gene defect; C human patient cells, or human trials in another disease with the same molecular
# lesion (e.g. a nonsense allele); D mechanistic inference only; X = direction of effect uncertain or plausibly harmful.
# `lever` names the allele or pathway the hypothesis acts on; the allele-level numbers (stop codon, NMD, tolerance of the
# inserted amino acids, peptides) are computed in section 14b and printed with the hypotheses in the report.
CURATED = [
    dict(rank=1, lever="stop allele: read-through",
         hypothesis="Read-through of the nonsense allele with an approved macrolide, tested first in patient-derived cells",
         agents="azithromycin (approved worldwide; decades of long-term paediatric use in cystic fibrosis, primary ciliary dyskinesia and "
                "bronchiectasis; oral, no nephro- or ototoxicity of the aminoglycoside kind, and it accumulates to high intracellular "
                "concentrations - the exposure argument the other read-through agents lack)",
         mechanism="in a reporter assay across six premature stop codons from primary ciliary dyskinesia patients, azithromycin stimulated "
                   "read-through only at the UGA codons - the class this child's stop allele belongs to - and the effect was modest; in mice "
                   "carrying a germline nonsense allele of the tumour-suppressor Apc it read through that allele, restored full-length Apc, "
                   "lowered beta-catenin / cyclin D1 and prevented intestinal tumour formation. That is the closest precedent we identified for "
                   "read-through acting as cancer prevention in a germline nonsense cancer-predisposition genotype, which is the architecture "
                   "of this child's stop allele",
         grade="C / C",
         caveat="the grade is C, not B, because the in vivo evidence is in another gene: this report reserves B for in vivo data in "
                "the same gene defect. In that same reporter study azithromycin was less efficient than aminoglycosides, and at the two non-UGA stop codons it "
                "REDUCED read-through by more than half, so the effect is codon-specific rather than general; the most responsive reporters "
                "had a UGA followed by cytosine, whereas this child's UGA is followed by adenine, a less favourable context; viability of "
                "primary nasal epithelial cells was preserved, but azithromycin was among the compounds that caused a small, statistically "
                "significant fall in ciliary beat frequency (remaining within the normal range), whereas amlexanox and gentamicin caused none; "
                "the Apc work is a mouse tumour-prevention study in another gene, given in drinking water, where the LOWEST dose was the most "
                "effective (a non-monotonic dose-response) and the authors themselves raise a microbiome confound; read-through of BUB1B has "
                "never been tested; macrolides carry QT prolongation, antimicrobial-resistance and (with long exposure) hearing-loss concerns, "
                "so this is a laboratory hypothesis, not a proposal to treat",
         validation="patient-derived fibroblasts or LCLs: allelic ratio on the cDNA amplicon, N- and C-terminal BUBR1 immunoblots, "
                    "allele-resolved peptides, nocodazole arrest, premature chromatid separation, micronuclei",
         pmids=["37276642", "37385919", "34066907", "41015699"]),
    dict(rank=2, lever="stop allele: NMD inhibition + read-through",
         hypothesis="Rescue protein from the nonsense allele with a dual NMD-inhibitor / read-through drug",
         agents="amlexanox (oral tablet long marketed in Japan for asthma / allergic rhinitis - current marketing status not verifiable from "
                "public regulatory records, so availability is not an argument here; the US topical paste was discontinued in 2017; oral "
                "dosing was also tested in a 42-patient randomised trial in adults with type 2 diabetes, which shows oral tolerability only)",
         mechanism="the nonsense allele's mRNA is predicted to be degraded by NMD, as MVA truncating-allele transcripts are absent in "
                   "patient cells; amlexanox stabilises NMD-prone mRNAs and induces read-through, and for a compound-heterozygous "
                   "missense / nonsense patient genotype (another gene, aspartylglucosaminuria) it restored full-length protein from the "
                   "nonsense allele with a synergistic gain in activity - the same architecture as this child's pair. Two products could "
                   "result: full-length BUBR1 carrying a near-cognate amino acid at the stop, and - where only NMD is relieved - the "
                   "N-terminal fragment. A truncated BUBR1 of 715 BUBR1 residues (731X) still supported the checkpoint and stable "
                   "attachments, whereas another truncation (753X) was expressed only weakly, which the authors note need not mean "
                   "reduced stability, so whether this fragment is stable, functional, or instead "
                   "sequesters CDC20 / BUB3 / PP2A-B56 as a dominant negative is an open, testable question",
         grade="C / D",
         caveat="that rescue was measured in a heterologous expression system, not in patient primary cells; whether the concentrations that "
                "worked in cells are reachable in a person is the gating question and we leave it open rather than guess - amlexanox acted at "
                "5-25 micromolar in the studies that reported the lowest effective concentrations, and at 25-125 micromolar in "
                "others, so roughly 1.5-37 microgram/mL for its molecular weight of 298.3, and we found no peer-reviewed human "
                "oral pharmacokinetic study reporting a plasma Cmax to compare it against (the only quantitative pharmacokinetic data we could "
                "verify are from a dog study), so no fold-difference is claimed here and human exposure data would have to be obtained before "
                "any systemic route could be argued at all; it was also less efficient than aminoglycosides in a head-to-head "
                "reporter comparison; it inhibits TBK1 / IKK-epsilon and TBK1 has mitotic functions, so the net effect on segregation must be measured; "
                "and sustained NMD inhibition itself raises R-loops and DNA damage, which is an unattractive trade in a chromosomal-instability "
                "cancer-predisposition syndrome",
         validation="patient-derived fibroblasts or LCLs: allelic ratio on the cDNA amplicon, N- and C-terminal BUBR1 immunoblots, "
                    "allele-resolved peptides, nocodazole arrest, premature chromatid separation, micronuclei",
         pmids=["22938201", "29247835", "37513945", "20516114", "27702906", "26656453", "28683283", "42788676", "34066907", "41954618"]),
    dict(rank=3, lever="missense allele / pathway: BUBR1 stability",
         hypothesis="NAD+ augmentation to raise residual BUBR1 via SIRT2",
         agents="niacin (nicotinic acid; raises NAD+ by the Preiss-Handler route and does not product-inhibit sirtuins); nicotinamide as the "
                "comparator arm (long-term exposure is documented at 1.2 g/m2/day for five years in a diabetes-prevention trial whose "
                "participants were relatives aged 3-40, so it included children and growth was unaffected); nicotinamide riboside / NMN are "
                "supplements and outside scope",
         mechanism="MVA missense alleles in or near the kinase domain lower BUBR1 through faster protein turnover; SIRT2 keeps "
                   "BUBR1-K668 deacetylated and stable; NMN raised BubR1 in vivo and SIRT2 extended median lifespan of "
                   "BubR1-hypomorphic mice. Niacin corrected systemic NAD+ deficiency and improved muscle performance in adults with "
                   "mitochondrial myopathy, which is why it is the probe of choice here",
         grade="B / X",
         caveat="the in vivo support is a genetic manipulation (SIRT2 overexpression), not the drug - the same limitation for which "
                "senolytics are set aside below, and the reason this is graded B/X rather than higher; nicotinamide is itself a sirtuin product-inhibitor, so the net effect on SIRT2 activity is unproven and niacin is the "
                "cleaner probe; correcting an NAD+ deficiency does not guarantee clinical benefit - in a child with mitochondrial myopathy "
                "and NAD+ deficiency, niacin produced a metabolic response but did not halt the disease, which is the honest ceiling on this "
                "kind of hypothesis; the cancer data are not reassuring so much as absent (nicotinamide reduced keratinocyte cancers in one "
                "randomised trial but not in a second, and an NAD+ precursor suppressed liver-cancer growth in mice; none of these studies "
                "involved chromosomal instability or rhabdomyosarcoma), so an effect on tumour metabolism cannot be excluded and must be "
                "measured in the child's own cells before anything else",
         validation="patient-derived cells: BUBR1 protein and half-life (cycloheximide chase), K668 acetylation, mitotic timing, micronuclei, "
                    "and proliferation of the aneuploid fraction, with and without niacin / NAM",
         pmids=["20516114", "24825348", "39490401", "26488693", "36856616", "12297502", "42449929", "32386566", "39173541", "15043959", "36986177"]),
    dict(rank=4, lever="stop allele: laboratory benchmark only (an assay control, not a candidate)",
         hypothesis="Aminoglycoside read-through as the ex-vivo benchmark - NOT proposed for use in this child",
         agents="gentamicin (laboratory positive control in cultured patient-derived cells only)",
         mechanism="UGA is the most read-through-prone stop in reporter assays and read-through inserts Trp, Cys or Arg there; "
                   "aminoglycosides are the most potent read-through agents available, so gentamicin establishes whether this particular "
                   "allele is read-through-competent at all and calibrates how much protein the approved agents above could ever deliver",
         grade="C / X",
         caveat="systemic use is excluded here, not merely cautioned: gentamicin is nephrotoxic (acute kidney injury in 20-33 % of exposed "
                "children) and irreversibly ototoxic, the i.v. read-through trials excluded pre-existing renal impairment, and this child has "
                "nephrocalcinosis; in Duchenne patients neither the stop codon nor the +4 nucleotide predicted response and one read-through "
                "epitope was immunogenic - response must be measured, not inferred",
         validation="same cultured-cell assays as #1 and #2, as the positive control that shows whether this allele is read-through-competent",
         pmids=["10939566", "22479203", "27702906", "35234826", "38366625", "28691931", "20517938", "27848094", "39098506"]),
    dict(rank=5, lever="symptom: nephrocalcinosis",
         hypothesis="Cause-directed treatment of nephrocalcinosis after a tubular work-up",
         agents="thiazide (hypercalciuria); potassium citrate / alkali (distal RTA, hypocitraturia)",
         mechanism="independent of BUBR1: prematurity and renal tubulopathies explain most paediatric nephrocalcinosis; part of the "
                   "failure to thrive may be renal and correctable; the closest paediatric interventional evidence is indirect - in 11 "
                   "children with X-linked hypophosphataemia whose nephrocalcinosis was caused by their phosphate and vitamin D treatment, "
                   "a thiazide was followed by no radiological progression, in an uncontrolled before-and-after comparison",
         grade="A (indication) / D (MVA-specific)",
         caveat="drug choice depends on urine calcium, citrate, oxalate, pH and serum bicarbonate; this is a work-up recommendation for the "
                "treating nephrologist, not a drug proposal, and prescribing blind could harm",
         validation="standard nephrology work-up; renal function follow-up", pmids=["41615714", "41867929", "41311422", "11433085"]),
]
# Considered and set aside - kept in the report because a reviewer will ask about each of them.
SET_ASIDE = [
    dict(item="Ataluren (read-through)", reason="EU conditional authorisation not renewed (European Commission decision, 28 March "
         "2025) and never approved in the US; a UK licence is reported to have been retained for its Duchenne indication, so it is a "
         "geographically restricted comparator rather than a candidate. It does act on the mechanism of interest - it promotes insertion of "
         "near-cognate tRNAs at premature stops - but availability, not mechanism, is what rules it out here", pmids=["27702906"]),
    dict(item="Hydroxychloroquine (aneuploidy-selective chemoprevention)",
         reason="the screen compound was chloroquine, not hydroxychloroquine; in mosaic variegated aneuploidy the aneuploid cells are "
         "largely normal tissue, so selecting against them could worsen growth failure, and the aneuploidy-vulnerability field has moved on",
         pmids=["21315436", "28775166", "40054068"]),
    dict(item="Metformin (senescence / inflammation)",
         reason="the aneuploidy-selective energy-stress compound in the original screen was AICAR (not approved), not metformin; and at "
         "millimolar concentrations in vitro, under low glucose, metformin itself induced genotoxic stress (micronuclei, nucleoplasmic "
         "bridges) - a direction worth avoiding in a chromosomal-instability syndrome, even though the same study found it non-genotoxic "
         "in high glucose", pmids=["18516091", "22048312", "21315436", "35568294"]),
    dict(item="ATP-site kinase inhibitors as pharmacological chaperones of the destabilised kinase domain",
         reason="the human BUBR1 kinase domain bound no nucleotide in thermal-shift assays, so there is no pocket for such a ligand",
         pmids=["24107129", "22698286"]),
    dict(item="Antisense skipping of the exon that carries the stop", reason="assessed in the allele-level triage (frame, how structured "
         "the exon's residues are, natural isoforms); an n-of-1 ASO is not an approved drug", pmids=["20516114"]),
    dict(item="Senolytic dasatinib + quercetin", reason="p16-positive cell clearance in BubR1-hypomorphic mice was genetic (INK-ATTAC), "
         "not pharmacological; the dasatinib label warns about bone growth in children (short stature here)",
         pmids=["22048312", "25754370"]),
]
CAUTION_CURATED = [
    dict(item="Growth hormone", reason="second-neoplasm risk (RR 1.77) in a cancer-predisposition syndrome with prior rhabdomyosarcoma; growth failure is cell-autonomous", pmids=["35529328"]),
    dict(item="Vinca alkaloids / taxanes and other antimitotics", reason="BubR1-insufficient cells slip through mitotic arrest; anticipate altered response and report outcomes - NOT a reason to withhold curative therapy", pmids=["14744753", "16411201"]),
    dict(item="Topoisomerase-II poisons and other genotoxic agents", reason="appear automatically as approved drugs on the network (above); genotoxic load in a chromosomal-instability syndrome warrants monitoring for secondary malignancy", pmids=["31053147"]),
    dict(item="Ionising radiation for imaging", reason="MVA surveillance guidance advises avoiding radiation exposure: ultrasound / MRI rather than CT where possible", pmids=["39264246"]),
    dict(item="Systemic aminoglycosides", reason="AKI in 20-33 % of exposed children; read-through trials excluded pre-existing renal impairment", pmids=["27848094", "38366625"]),
    dict(item="Sustained systemic NMD inhibition", reason="blocking nonsense-mediated decay raises R-loops and DNA damage and is being developed "
         "deliberately as a way to kill genomically unstable cells; relieving NMD is the point of the amlexanox hypothesis, so this trade-off must "
         "be measured in cells (micronuclei, DNA-damage markers) before any systemic use is ever contemplated in a chromosomal-instability syndrome",
         pmids=["41954618", "42711693", "39667907"]),
    dict(item="HDAC inhibitors as mechanism-based therapy", reason="opposite-signed acetylation sites on BUBR1 make the net effect indeterminate; mitotic histone deacetylation is required for accurate segregation", pmids=["19407811", "24825348", "35922507"]),
    dict(item="Aurora B inhibitors", reason="would remove the phosphorylation required for protective BUBR1-K250 acetylation", pmids=["41533580"]),
    dict(item="Genotoxic conditioning / HSCT", reason="the one published MVA1 transplant ended in graft rejection and death; MDS with monosomy 7 is a surveillance target", pmids=["31053147"]),
]
# Verified references used in the report's prose (resolved live like the others)
TRACK2_TEXT_PMIDS = ["39264246", "11746029", "15108198", "42434306", "40028477", "40473777", "27605262", "25741868", "41257875",
                     "32720330", "16411201", "15475955", "18932004", "15159543", "22698286", "23079597", "15208629", "18516091",
                     "20516114", "21552266", "28553959", "34009673", "39169218", "31738183", "33207204", "42824587", "34066907"]


def resolve_pmids(pmids):
    out = {}
    for p in sorted(set(pmids)):
        try:
            r = http_json("GET", "https://www.ebi.ac.uk/europepmc/webservices/rest/search",
                          params={"query": f"EXT_ID:{p} AND SRC:MED", "format": "json", "resultType": "lite", "pageSize": 1}, timeout=60)
            res = (r.get("resultList") or {}).get("result") or []
            out[p] = (dict(title=res[0].get("title", ""), journal=res[0].get("journalTitle", ""), year=res[0].get("pubYear", ""),
                           authors=(res[0].get("authorString") or "").split(",")[0]) if res else None)
        except Exception:       # noqa: BLE001
            out[p] = None
        time.sleep(0.15)
    return out


with stage("Track 2: drug-repurposing evidence"):
    if not CFG["RUN_TRACK2"]:
        raise RuntimeError("disabled in CFG")
    g = HYPS[0]["gene"]
    T2_DF, T2_SUM = network_drug_pass(g)
    S["track2"] = dict(network=T2_SUM)
    log(f"Track 2 network pass for {g}: {T2_SUM}")
    appr = T2_DF[T2_DF.stage == "APPROVAL"].drop_duplicates(["drug", "target"]).sort_values(["verdict", "target", "drug"])
    appr.to_csv(OUT / "track2_network_drugs.csv", index=False)
    md(f"**Approved drugs acting on the {g} network ({T2_SUM['n_network_nodes']} nodes; Open Targets {T2_SUM['open_targets_api']})**\n\n"
       + df_md(appr[["target", "node", "drug", "action", "mechanism", "verdict"]], max_rows=40))
    md("**Sign map used for direction (the only hand-curated input)**\n\n" + df_md(pd.DataFrame(
        [dict(node=k, sign=v[0], rationale=v[1], pmids=", ".join(v[2])) for k, v in SIGN_MAP.get(g, {}).items()])))
    # generality: identical code for the other genes of the stated indication's panel
    gen = []
    for g2 in [x for x in S["pheno"].get("dx_panel", []) if x != g and x in GENE_POS][:6]:
        try:
            _, s2 = network_drug_pass(g2)
            gen.append(dict(gene=g2, network_nodes=s2["n_network_nodes"], drugs_on_gene=s2["n_direct_drugs"],
                            approved_rows=s2["n_approved"], **{k: v for k, v in s2["verdicts"].items()}))
        except Exception as e:      # noqa: BLE001
            gen.append(dict(gene=g2, error=str(e)[:80]))
    S["track2"]["generality"] = gen
    if gen:
        md("**Same automated pass for the other genes of the indication panel**\n\n" + df_md(pd.DataFrame(gen).fillna(0)))
    # literature layer, citations resolved live
    all_p = [p for h in CURATED for p in h["pmids"]] + [p for c in CAUTION_CURATED + SET_ASIDE for p in c["pmids"]] + \
            [p for v in SIGN_MAP.get(g, {}).values() for p in v[2]] + TRACK2_TEXT_PMIDS
    PMIDS = resolve_pmids(all_p)
    bad = [p for p, v in PMIDS.items() if not v]
    S["track2"].update(curated=CURATED if g == "BUB1B" else [], caution=CAUTION_CURATED if g == "BUB1B" else [],
                       set_aside=SET_ASIDE if g == "BUB1B" else [],
                       pmids={p: v for p, v in PMIDS.items() if v}, unresolved_pmids=bad)
    log(f"citations resolved live: {len(PMIDS) - len(bad)} of {len(PMIDS)}" + (f"; unresolved (not used): {bad}" if bad else ""))
    if g == "BUB1B":
        md("**Ranked hypotheses (literature layer)**\n\n" + df_md(pd.DataFrame(CURATED)[["rank", "lever", "hypothesis", "agents", "grade", "caveat"]]))
        md("**Considered and set aside**\n\n" + df_md(pd.DataFrame(SET_ASIDE)[["item", "reason"]]))
        md("**Caution list for the clinical team**\n\n" + df_md(pd.DataFrame(CAUTION_CURATED)[["item", "reason"]]))
    else:
        md(f"The literature-curated layer is specific to BUBR1 biology and is not applied to {g}.")

## 14b · Allele-level therapeutic triage (gene-agnostic)

Drug–target databases answer "which drug acts on a protein near this gene?". For a loss-of-function disorder the more
useful question is allele-specific: **what can still be done with each of the child's two alleles?** This stage answers
it from the reference sequence, the MANE gene model, ESM-2 and the AlphaFold model, for whichever gene leads:

* **Premature stop codons** — stop-codon identity and the following (+4) nucleotide, the main sequence determinants of
  drug-induced read-through; whether nonsense-mediated decay removes the transcript; which amino acids read-through would
  insert (near-cognate tRNAs) and how ESM-2 rates them at that residue; whether the stop lies in an exon whose skipping
  keeps the reading frame (antisense exon skipping), how structured that exon's residues are, and whether a natural
  isoform already skips it.
* **Missense alleles** — domain, AlphaFold confidence, burial, distance to literature-reported patient alleles.
* **The pair** — the distance between the alleles in the mRNA (one RT-PCR amplicon from patient RNA phases them
  directly) and allele-specific tryptic peptides that tell apart the protein made from each allele, so a read-through
  product can be measured in the child's own cells.
* **The path to a firm diagnosis** — which evidence would move the weaker allele from VUS to (likely) pathogenic.
* **Scalability** — the same triage over every ClinVar P/LP allele in the genes of the disorder's panel.

In [ ]:
STOP_CODONS = {"TAA", "TAG", "TGA"}
STOP_ORDER = {"TGA": 1, "TAG": 2, "TAA": 3}           # read-through permissiveness, most to least permissive
NEAR_COGNATE = {"TGA": "WCR", "TAG": "QYK", "TAA": "QYK"}     # amino acids inserted at each stop during induced read-through
COMP = str.maketrans("ACGTN", "TGCAN")
CODON = {a + b + c: aa for (a, b, c), aa in
         zip(itertools.product("TCAG", repeat=3), "FFLLSSSSYY**CC*WLLLLPPPPHHQQRRRRIIIMTTTTNNKKSSRRVVVVAAAADDEEGGGG")}
MONO = dict(G=57.02146, A=71.03711, S=87.03203, P=97.05276, V=99.06841, T=101.04768, C=103.00919, L=113.08406, I=113.08406,
            N=114.04293, D=115.02694, Q=128.05858, K=128.09496, E=129.04259, M=131.04049, H=137.05891, F=147.06841,
            R=156.10111, Y=163.06333, W=186.07931)
LEVER_ORDER = ["read-through: UGA stop", "read-through: UAG / UAA stop", "exon skipping: lesion in an in-frame exon",
               "truncated protein escapes NMD", "missense: protein-level", "splice: splice-switching candidate",
               "start-loss: translation must restart downstream", "no allele-specific lever"]


def cds_model(tid, fa):
    """Coding sequence (5'→3'), the genomic position of every coding base, and its CDS segment index."""
    t = TX[tid]
    seq, gpos, seg = [], [], []
    for i, (s, e) in enumerate(t["cds"]):
        sub = fa[t["chrom"]][s - 1:e]
        if t["strand"] == "+":
            seq.append(sub)
            gpos.extend(range(s, e + 1))
        else:
            seq.append(sub[::-1].translate(COMP))
            gpos.extend(range(e, s - 1, -1))
        seg.extend([i] * (e - s + 1))
    return dict(tid=tid, seq="".join(seq), gpos=gpos, seg=seg, idx={p: i for i, p in enumerate(gpos)})


def coding_change(m, pos, ref, alt):
    """Codon-level consequence of a coding SNV (None when it is not a coding SNV or the reference does not match)."""
    t = TX[m["tid"]]
    if len(ref) != 1 or len(alt) != 1 or pos not in m["idx"]:
        return None
    i = m["idx"][pos]
    plus = t["strand"] == "+"
    if m["seq"][i] != (ref if plus else ref.translate(COMP)):
        return None
    k = i // 3
    wt = m["seq"][3 * k:3 * k + 3]
    mut = wt[:i % 3] + (alt if plus else alt.translate(COMP)) + wt[i % 3 + 1:]
    nxt = m["seq"][3 * k + 3] if 3 * k + 3 < len(m["seq"]) else ""
    return dict(c=i + 1, residue=k + 1, wt_codon=wt, mut_codon=mut, wt_aa=CODON.get(wt, "?"), mut_aa=CODON.get(mut, "?"),
                plus4=nxt, segment=m["seg"][i])


def exon_info(m, i):
    """What skipping CDS segment i would do: frame, residues removed, and whether it is a whole internal exon."""
    t = TX[m["tid"]]
    s, e = t["cds"][i]
    n = len(t["cds"])
    length = e - s + 1
    first = sum(b - a + 1 for a, b in t["cds"][:i])
    exon_no = next((k + 1 for k, (xs, xe) in enumerate(t["exons"]) if xs <= s and e <= xe), None)
    whole = (s, e) in set(map(tuple, t["exons"]))
    return dict(exon=exon_no, n_exons=len(t["exons"]), length_nt=length, in_frame=length % 3 == 0,
                skippable=bool(whole and 0 < i < n - 1), aa_from=first // 3 + 1, aa_to=(first + length - 1) // 3 + 1,
                genomic=(s, e), flank_up=t["cds"][i - 1] if i > 0 else None, flank_dn=t["cds"][i + 1] if i < n - 1 else None)


def natural_skipping(gene, ex):
    """Annotated protein-coding Ensembl transcripts that skip this exon but keep both flanking exons."""
    if not (ex["flank_up"] and ex["flank_dn"]):
        return []
    d = http_json("GET", f"https://rest.ensembl.org/lookup/symbol/homo_sapiens/{gene}", params={"expand": 1},
                  headers={"Content-Type": "application/json"}, timeout=60)
    hits = []
    for tr in d.get("Transcript", []):
        if tr.get("biotype") != "protein_coding":
            continue
        exs = [(x["start"], x["end"]) for x in tr.get("Exon", [])]

        def ov(a, b):
            return any(xs <= b and xe >= a for xs, xe in exs)
        if not ov(*ex["genomic"]) and ov(*ex["flank_up"]) and ov(*ex["flank_dn"]):
            hits.append(tr["id"])
    return hits


def esm_row(gene, residue):
    """ESM-2 log-likelihood ratios (AA20 order) for every substitution at one residue, and the protein-wide values
    (None when only single positions were scored on CPU; a missing position is then scored on CPU on demand)."""
    ctx = globals().get("ESM_CTX")
    if not ctx or ctx["gene"] != gene:
        return None, None
    k = residue - 1 - ctx["off"]
    if ctx.get("llr") is not None:
        if not 0 <= k < len(ctx["llr"]):
            return None, None
        flat = ctx["llr"].ravel()
        return ctx["llr"][k], flat[flat != 0]
    rows = ctx.setdefault("rows", {})
    if residue not in rows and 0 <= k < len(ctx.get("wseq", "")):
        try:
            rows[residue] = esm_single(ctx["wseq"], k)
        except Exception as e:      # noqa: BLE001
            log(f"ESM-2 single-position scoring of residue {residue} skipped ({str(e)[:100]})")
    return rows.get(residue), None


def af_ctx(gene):
    """The AlphaFold/UniProt context, only when it belongs to this gene and uses the MANE residue numbering."""
    ctx = globals().get("AF_CTX")
    return ctx if ctx and ctx["gene"] == gene and ctx.get("numbering_matches_mane") else None


def af_at(gene, residues):
    ctx = af_ctx(gene)
    if not ctx:
        return None
    vals = [ctx["ca"][r][1] for r in residues if r in ctx["ca"]]
    return round(float(np.mean(vals)), 1) if vals else None


def features_overlapping(gene, a, b):
    ctx = af_ctx(gene)
    if not ctx:
        return []
    return [f"{f['name']} ({f['start']}-{f['end']})" for f in ctx["feats"] if f["start"] and f["end"] and f["start"] <= b and f["end"] >= a]


PROTEASES = {        # cleavage rules (C-terminal side unless noted); the first that yields observable peptides is used
    "trypsin": lambda s, i: s[i] in "KR" and s[i + 1] != "P",
    "Glu-C": lambda s, i: s[i] == "E",
    "chymotrypsin": lambda s, i: s[i] in "FWY" and s[i + 1] != "P",
    "Asp-N": lambda s, i: s[i + 1] == "D",          # cleaves N-terminal to Asp
    "Lys-C": lambda s, i: s[i] == "K",
}


def digest_peptide(seq, residue, protease="trypsin"):
    """(peptide, first residue, last residue) containing a 1-based residue after complete digestion with one protease."""
    rule = PROTEASES[protease]
    cuts = [0] + [i + 1 for i in range(len(seq) - 1) if rule(seq, i)] + [len(seq)]
    for a, b in zip(cuts, cuts[1:]):
        if a < residue <= b:
            return seq[a:b], a + 1, b
    return "", 0, 0


def observable(p):
    return 7 <= len(p) <= 30


def site_peptides(forms, residue, avoid=None, min_delta_da=0.02):
    """forms: {label: (protein sequence, made_by)}, each the protein actually made by that allele / product. Returns the first
    protease whose peptides covering the residue are MS-observable, do not also span the other allele's site (`avoid`), and
    differ pairwise in monoisotopic mass by at least `min_delta_da`."""
    for prot_ in PROTEASES:
        peps = {k: digest_peptide(sq, residue, prot_) for k, (sq, _) in forms.items()}
        masses = sorted(peptide_mass(x[0]) for x in peps.values())
        if (all(observable(x[0]) for x in peps.values())
                and not any(avoid and lo <= avoid <= hi for _, lo, hi in peps.values())
                and all(b - a >= min_delta_da for a, b in zip(masses, masses[1:]))):
            return [dict(protease=prot_, form=k, peptide=peps[k][0], mass=peptide_mass(peps[k][0]), made_by=forms[k][1]) for k in forms]
    return []


def peptide_mass(p):
    return round(sum(MONO.get(a, 0.0) for a in p) + 18.01056, 3)


def tx_model(tid, fa):
    """Spliced transcript (exons 5'→3'), the genomic position and exon number of every base, and the first coding base."""
    t = TX[tid]
    seq, gpos, exn = [], [], []
    for k, (s, e) in enumerate(t["exons"]):
        sub = fa[t["chrom"]][s - 1:e]
        if t["strand"] == "+":
            seq.append(sub)
            gpos.extend(range(s, e + 1))
        else:
            seq.append(sub[::-1].translate(COMP))
            gpos.extend(range(e, s - 1, -1))
        exn.extend([k + 1] * (e - s + 1))
    first = t["cds"][0][0] if t["strand"] == "+" else t["cds"][0][1]
    return dict(seq="".join(seq), gpos=gpos, exon=exn, cds0=gpos.index(first), chrom=t["chrom"])


def positions_with_variants(chrom, positions, source):
    """How many of the given genomic positions carry a variant: in the child's VCF ('child', count only — no genotypes
    leave this function) or a public variant with minor-allele frequency >= 1 % (Ensembl REST, reference coordinates)."""
    if not positions:
        return 0
    a, b = min(positions), max(positions)
    if source == "child":
        raw = chrom if S.get("vcf_chr_style", True) else chrom[3:]
        txt = sh(f"bcftools query -r {raw}:{a}-{b} -f '%POS\\n' '{S['vcf_raw']}'", name="primer_site_child", capture=True)
        return len({int(x) for x in txt.split()} & set(positions))
    d = http_json("GET", f"https://rest.ensembl.org/overlap/region/human/{chrom.replace('chr', '')}:{a}-{b}",
                  params={"feature": "variation"}, headers={"Content-Type": "application/json"}, timeout=60)
    return len({p for x in d if (x.get("minor_allele_freq") or 0) >= 0.01
                for p in range(int(x["start"]), int(x["end"]) + 1)} & set(positions))


def design_rtpcr_primers(m, c1, c2, fa, min_gap=60, flank=450, n=3):
    """Primer3 primers for one RT-PCR amplicon spanning both coding positions (c1 < c2), each primer >= `min_gap` nt from
    the nearer allele so that both sites are read by Sanger or amplicon sequencing. Pairs are ranked by: no variant of the
    child under a primer (allele dropout would distort the allelic ratio), no common public SNP, a primer spanning an
    exon-exon junction (no product from genomic DNA), then the Primer3 penalty."""
    ensure_modules({"primer3": "primer3-py"})
    import primer3
    tm = tx_model(m["tid"], fa)
    s = tm["seq"]
    if s[tm["cds0"]:tm["cds0"] + len(m["seq"])] != m["seq"]:
        raise RuntimeError("exon and CDS models disagree")
    i1, i2 = tm["cds0"] + c1 - 1, tm["cds0"] + c2 - 1
    lo, hi = max(0, i1 - flank), min(len(s), i2 + flank + 1)
    design = getattr(primer3.bindings, "design_primers", None) or primer3.bindings.designPrimers
    span = i2 - i1 + 2 * min_gap + 1
    r = design(dict(SEQUENCE_ID="amplicon", SEQUENCE_TEMPLATE=s[lo:hi], SEQUENCE_TARGET=[i1 - lo - min_gap, span]),
               dict(PRIMER_OPT_SIZE=22, PRIMER_MIN_SIZE=18, PRIMER_MAX_SIZE=27, PRIMER_OPT_TM=60.0, PRIMER_MIN_TM=57.0,
                    PRIMER_MAX_TM=63.0, PRIMER_MIN_GC=40.0, PRIMER_MAX_GC=60.0, PRIMER_MAX_POLY_X=4, PRIMER_PAIR_MAX_DIFF_TM=3.0,
                    PRIMER_NUM_RETURN=25, PRIMER_PRODUCT_SIZE_RANGE=[[span, span + 2 * (flank - min_gap)]]))
    pairs = []
    for k in range(r.get("PRIMER_PAIR_NUM_RETURNED", 0)):
        ls, ll = r[f"PRIMER_LEFT_{k}"]
        rs, rl = r[f"PRIMER_RIGHT_{k}"]
        left = range(lo + ls, lo + ls + ll)
        right = range(lo + rs - rl + 1, lo + rs + 1)
        ex_l, ex_r = [tm["exon"][j] for j in left], [tm["exon"][j] for j in right]

        def spans(ex):
            return len(set(ex)) > 1 and min(collections.Counter(ex).values()) >= 4      # >= 4 nt on each side of the junction
        gp = [tm["gpos"][j] for j in list(left) + list(right)]
        pairs.append(dict(left=r[f"PRIMER_LEFT_{k}_SEQUENCE"], right=r[f"PRIMER_RIGHT_{k}_SEQUENCE"],
                          tm_left=round(r[f"PRIMER_LEFT_{k}_TM"], 1), tm_right=round(r[f"PRIMER_RIGHT_{k}_TM"], 1),
                          product_nt=r[f"PRIMER_PAIR_{k}_PRODUCT_SIZE"], exons_left="-".join(map(str, sorted(set(ex_l)))),
                          exons_right="-".join(map(str, sorted(set(ex_r)))), junction_spanning=spans(ex_l) or spans(ex_r),
                          genomic_span_bp=abs(tm["gpos"][right[-1]] - tm["gpos"][left[0]]) + 1,
                          site_offsets=(i1 - left[0] + 1, i2 - left[0] + 1), _gpos=gp,
                          penalty=round(r[f"PRIMER_PAIR_{k}_PENALTY"], 2)))
    out = []
    for p in sorted(pairs, key=lambda p: (not p["junction_spanning"], p["penalty"])):
        gp = p.pop("_gpos")
        p["child_variants_under_primers"] = positions_with_variants(tm["chrom"], gp, "child") if S.get("vcf_raw") else None
        try:
            p["common_snps_under_primers"] = positions_with_variants(tm["chrom"], gp, "public")
        except Exception as e:      # noqa: BLE001 - REST unavailable: report as unchecked
            p["common_snps_under_primers"] = None
            log(f"public SNP check of primers skipped ({str(e)[:80]})")
        if not p["child_variants_under_primers"] and not p["common_snps_under_primers"]:
            out.append(p)
        if len(out) == n:
            break
    return out


def reclassification_path(v, partner, pm3_now):
    """Evidence that would move the weaker allele's automated ACMG class, applied cumulatively on the point scale of
    Tavtigian 2020. PM3 follows the pipeline's ClinGen SVI rule: in trans with a Pathogenic partner = moderate (2),
    with a Likely pathogenic partner = supporting (1)."""
    pm3_trans = 2.0 if partner["p_path"] >= 0.99 else 1.0 if partner["p_path"] >= 0.90 else 0.0
    steps = [("current automated evidence (phase unknown" + (f", PM3 {pm3_now:+g}" if pm3_now else "") + ")", pm3_now),
             (f"+ PM3 confirmed in trans (partner allele {'Pathogenic' if pm3_trans == 2 else 'Likely pathogenic' if pm3_trans == 1 else 'below LP'}; "
              "parental testing or the cDNA amplicon)", pm3_trans - pm3_now),
             ("+ PP4: phenotype highly specific for the gene's disorder (ClinGen caveats for heterogeneous disorders apply)", 1),
             ("+ PS3_moderate: allele-specific readout in the child's cells (peptides from this allele's protein; its half-life) "
              "shows reduced protein or function", 2)]
    rows, extra = [], 0.0
    for label, pts in steps:
        extra += pts
        tot = v["points"] + extra
        rows.append(dict(step=label, points=tot, posterior=round(points_to_posterior(tot), 3), classification=points_class(tot)))
    return rows


def classify_clinvar(r, m):
    """Allele-specific lever class of one ClinVar P/LP record."""
    mc = r.mc or ""
    seg = m["seg"][m["idx"][r.pos]] if r.pos in m["idx"] else (m["seg"][m["idx"][r.pos + 1]] if r.pos + 1 in m["idx"] else None)
    ex = exon_info(m, seg) if seg is not None else None
    in_frame_exon = bool(ex and ex["skippable"] and ex["in_frame"])
    if "nonsense" in mc:
        cc = coding_change(m, r.pos, r.ref, r.alt)
        stop = cc["mut_codon"] if cc and cc["mut_codon"] in STOP_CODONS else None
        lever = ("read-through: UGA stop" if stop == "TGA" else "read-through: UAG / UAA stop" if stop
                 else "exon skipping: lesion in an in-frame exon" if in_frame_exon else "no allele-specific lever")
        return lever, stop, in_frame_exon
    if "frameshift" in mc:
        esc, _ = nmd_status(r.gene, r.pos)
        return ("exon skipping: lesion in an in-frame exon" if in_frame_exon else "truncated protein escapes NMD" if esc
                else "no allele-specific lever"), None, in_frame_exon
    if re.search(r"splice_(donor|acceptor)", mc):
        return "splice: splice-switching candidate", None, in_frame_exon
    if "initiator_codon" in mc:      # a lost start codon is also annotated as missense, but then no protein is made from it
        return "start-loss: translation must restart downstream", None, False
    if "missense" in mc:
        return "missense: protein-level", None, False
    return "no allele-specific lever", None, False


LEVER_RATIONALE = {
    "read-through: UGA stop": "UGA is the stop codon most responsive to read-through agents in reporter assays, so this allele is a "
                             "candidate for the read-through hypotheses - to be measured, not assumed",
    "read-through: UAG / UAA stop": "a premature stop, but not the most read-through-prone codon: lower prior than UGA",
    "exon skipping: lesion in an in-frame exon": "the lesion sits in an exon whose length is a multiple of three, so skipping it would "
                                                "keep the reading frame (an antisense approach, not an approved drug)",
    "truncated protein escapes NMD": "predicted to escape nonsense-mediated decay, so a truncated protein is likely made: its stability "
                                     "and any dominant-negative effect are the questions",
    "splice: splice-switching candidate": "a canonical splice-site lesion, the class splice-switching oligonucleotides address",
    "missense: protein-level": "no nucleic-acid-level lever; the handle is protein abundance or function",
    "start-loss: translation must restart downstream": "the start codon is lost, so protein can only be made if translation restarts "
                                                       "at a later start site; none of the levers in this triage applies",
    "no allele-specific lever": "none of the allele-level levers in this triage apply",
}


def _short_allele(a, limit=24):
    """ClinVar holds some very long indel alleles; keep the file readable without losing the length."""
    a = str(a)
    return a if len(a) <= limit else f"{a[:limit]}...({len(a)} bp)"


def lever_atlas(genes, fa):
    rows = []
    for g in genes:
        if g not in GENE_TX or g not in GENE_POS:
            continue
        m = cds_model(GENE_TX[g][0], fa)
        cv = CLINVAR_PLP[CLINVAR_PLP.gene == g].drop_duplicates(["chrom", "pos", "ref", "alt"])
        for r in cv.itertuples():
            lever, stop, inframe = classify_clinvar(r, m)
            seg = m["seg"][m["idx"][r.pos]] if r.pos in m["idx"] else None
            ex = exon_info(m, seg) if seg is not None else None
            cc = coding_change(m, r.pos, r.ref, r.alt) if len(str(r.ref)) == 1 and len(str(r.alt)) == 1 else None
            rows.append(dict(gene=g, chrom=r.chrom, pos=int(r.pos), ref=_short_allele(r.ref), alt=_short_allele(r.alt),
                             key=f"{r.chrom}:{r.pos} {_short_allele(r.ref)}>{_short_allele(r.alt)}",
                             protein_change=(f"{cc['wt_aa']}{cc['residue']}" + (cc.get("mut_aa") or "")) if cc and cc.get("residue") else "",
                             exon=(ex or {}).get("exon") or "", n_exons=(ex or {}).get("n_exons") or "",
                             molecular_consequence=(r.mc or "").replace("|", ";"),
                             clinical_significance=r.clnsig, review_status=r.clnrevstat,
                             lever=lever, lever_rationale=LEVER_RATIONALE.get(lever, ""),
                             stop_codon_rna=(stop or "").replace("T", "U"), in_frame_exon=inframe,
                             note=("nonsense-tagged indel: no single stop codon computed" if "nonsense" in (r.mc or "") and not stop else "")))
    return pd.DataFrame(rows, columns=["gene", "chrom", "pos", "ref", "alt", "key", "protein_change", "exon", "n_exons",
                                       "molecular_consequence", "clinical_significance", "review_status",
                                       "lever", "lever_rationale", "stop_codon_rna", "in_frame_exon", "note"])


def lever_figure(gene, alleles, prot_len, ptc, mis):
    """Panel A: the protein, its domains, AlphaFold confidence and where the alleles fall.
    Panel B: ESM-2 tolerance of every substitution at the stop-codon residue (what read-through would insert)."""
    ctx = af_ctx(gene)
    row = esm_row(gene, ptc["residue"])[0] if ptc else None
    nrow = 2 if row is not None else 1
    fig, axes = plt.subplots(nrow, 1, figsize=(7.4, 5.8 if nrow == 2 else 3.0), gridspec_kw=dict(height_ratios=[1.15, 1][:nrow]))
    axes = np.atleast_1d(axes)
    ax = axes[0]
    if ctx:
        res = sorted(ctx["ca"])
        ax.plot(res, [ctx["ca"][r][1] for r in res], color=VIZ["ink2"], linewidth=1.0, label="AlphaFold confidence (pLDDT)")
        for f in ctx["feats"]:
            if f["type"] == "Domain" and f["start"] and f["end"]:
                ax.axvspan(f["start"], f["end"], ymin=0, ymax=0.06, color=VIZ["neutral"], linewidth=0)
                ax.text((f["start"] + f["end"]) / 2, 7, f["name"], ha="center", va="bottom", fontsize=7.5, color=VIZ["ink2"])
        dis = ctx.get("disease_missense") or []
        if dis:
            ax.scatter([d["residue"] for d in dis], [ctx["ca"][d["residue"]][1] for d in dis], s=30, color=VIZ["s3"], zorder=3,
                       edgecolors=VIZ["surface"], linewidths=1.5, label=f"literature-reported patient missense (UniProt, n={len(dis)})")
    if ptc and ptc.get("exon"):
        ex = ptc["exon"]
        ax.axvspan(ex["aa_from"], ex["aa_to"], color=VIZ["s2"], alpha=0.12, linewidth=0)
        ax.text((ex["aa_from"] + ex["aa_to"]) / 2, 103, f"exon {ex['exon']}", ha="center", va="bottom", fontsize=7.5, color=VIZ["ink2"])
    if ptc:
        ax.axvline(ptc["residue"], color=VIZ["s2"], linewidth=2, label=f"{ptc['label']} ({ptc['stop_rna']} stop)")
    if mis:
        ax.axvline(mis["residue"], color=VIZ["s1"], linewidth=2, label=mis["label"])
    ax.set_xlim(0, prot_len + 1)
    ax.set_ylim(0, 112)
    ax.set_xlabel(f"{gene} protein residue")
    ax.set_ylabel("pLDDT")
    ax.set_title(f"{gene}: where the child's alleles fall")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=2)
    if row is not None:
        ax = axes[1]
        wt = ptc["wt_aa"]
        items = sorted(((a, float(row[AA20.index(a)])) for a in AA20 if a != wt), key=lambda x: x[1])
        near = set(NEAR_COGNATE.get(ptc["stop_dna"], ""))
        cols = [VIZ["s2"] if a in near else VIZ["neutral"] for a, _ in items]
        ax.bar(range(len(items)), [x for _, x in items], color=cols, width=0.72)
        ax.set_xticks(range(len(items)))
        ax.set_xticklabels([f"{wt}{ptc['residue']}{a}" for a, _ in items], rotation=90, fontsize=7.5)
        for j, (a, x) in enumerate(items):
            if a in near:
                ax.text(j, x - 0.25, f"{x:.1f}", ha="center", va="top", fontsize=7.5, color=VIZ["ink"])
        if mis and mis.get("llr") is not None:
            ax.axhline(mis["llr"], color=VIZ["s1"], linewidth=1.5)
            ax.text(len(items) - 0.5, mis["llr"] + 0.25,
                    f"the child's {mis['short']} (LLR {mis['llr']:.1f}) — a different residue, shown for scale only",
                    ha="right", va="bottom",
                    fontsize=7.5, color=VIZ["ink2"])
        ax.axhline(0, color=VIZ["axis"], linewidth=0.8)
        ax.set_ylabel("ESM-2 LLR")
        ax.set_title(f"What read-through would insert at residue {ptc['residue']}: near-cognate amino acids (orange) "
                     f"vs all others", fontsize=9.5)
    fig.tight_layout()
    return save_fig(fig, "allele_levers")


with stage("Track 2: allele-level therapeutic triage"):
    if not CFG["RUN_TRACK2"]:
        raise RuntimeError("disabled in CFG")
    from pyfaidx import Fasta
    fa = Fasta(RES["ref"], as_raw=True, sequence_always_upper=True)
    TOP = HYPS[0]
    g = TOP["gene"]
    m = cds_model(GENE_TX[g][0], fa)
    t = TX[m["tid"]]
    prot = PROT.get(t["protein"], "").rstrip("*")
    L_out, ptc, mis = [], None, None
    for v in TOP["variants"]:
        cc = coding_change(m, int(v["pos"]), v["ref"], v["alt"])
        a = dict(key=f"{v['chrom']}:{int(v['pos'])} {v['ref']}>{v['alt']}", hgvsc=v.get("hgvsc", ""), hgvsp=v.get("hgvsp", ""),
                 csq=v["csq"], c=cc["c"] if cc else None)
        if cc and cc["mut_codon"] in STOP_CODONS:
            ex = exon_info(m, cc["segment"])
            seg_res = range(ex["aa_from"], ex["aa_to"] + 1)
            ex.update(mean_plddt=af_at(g, seg_res), features=features_overlapping(g, ex["aa_from"], ex["aa_to"]))
            try:
                ex["natural_skipping_isoforms"] = natural_skipping(g, ex) if ex["skippable"] and ex["in_frame"] else None
            except Exception as e:      # noqa: BLE001
                ex["natural_skipping_isoforms"] = None
                log(f"Ensembl isoform lookup skipped ({str(e)[:100]})")
            row, flat = esm_row(g, cc["residue"])
            nc = {}
            if row is not None:
                for aa in NEAR_COGNATE[cc["mut_codon"]]:
                    x = float(row[AA20.index(aa)])
                    nc[aa] = dict(llr=round(x, 2), pct_less_tolerated=round(float((flat < x).mean()) * 100, 1) if flat is not None else None,
                                  restores_wild_type=aa == cc["wt_aa"])
            ptc = a
            a.update(kind="premature stop", residue=cc["residue"], wt_aa=cc["wt_aa"], stop_dna=cc["mut_codon"],
                     stop_rna=cc["mut_codon"].replace("T", "U"), plus4=cc["plus4"],
                     readthrough_rank=STOP_ORDER[cc["mut_codon"]], near_cognate_aas=NEAR_COGNATE[cc["mut_codon"]],
                     plus4_rna=cc["plus4"].replace("T", "U"), native_residues_kept=cc["residue"] - 1,
                     nmd_escape=v.get("nmd_escape"), plddt=af_at(g, [cc["residue"]]), near_cognate=nc, exon=ex,
                     label=v.get("hgvsp") or a["key"], truncated_product=f"1-{cc['residue'] - 1}",
                     truncated_kda=round(peptide_mass(prot[:cc["residue"] - 1]) / 1000, 1) if prot else None,
                     truncated_keeps=features_overlapping(g, 1, cc["residue"] - 1),
                     truncated_loses=features_overlapping(g, cc["residue"], len(prot)))
        elif cc and v["csq"] == "missense":
            row, flat = esm_row(g, cc["residue"])
            st = S.get("structure", {}) if (af_ctx(g)
                                            and (S.get("structure") or {}).get("residue") == cc["residue"]) else {}
            mis = a
            a.update(kind="missense", residue=cc["residue"], wt_aa=cc["wt_aa"], mut_aa=cc["mut_aa"],
                     llr=round(float(row[AA20.index(cc["mut_aa"])]), 2) if row is not None else None,
                     plddt=af_at(g, [cc["residue"]]), domain=st.get("domain"), burial_percentile=st.get("burial_percentile_in_domain"),
                     nearest_patient_missense=(st.get("disease_missense_uniprot") or [])[:3],
                     label=v.get("hgvsp") or a["key"], short=f"{cc['wt_aa']}{cc['residue']}{cc['mut_aa']}")
        L_out.append(a)
    S["levers"] = dict(gene=g, transcript=t.get("version") or m["tid"], protein_length=len(prot), alleles=L_out)
    md("**Allele-level triage**\n\n" + df_md(pd.DataFrame([{k: x.get(k) for k in ("key", "hgvsp", "kind", "residue", "stop_rna", "plus4",
                                                                                   "nmd_escape", "plddt", "llr")} for x in L_out])))

    # ---- the pair: cDNA phasing amplicon, allele-resolved peptides, path to reclassification -----------------------------
    if TOP["model"] == "compound_het" and all(x.get("c") for x in L_out):
        c1, c2 = sorted(x["c"] for x in L_out)
        gd = abs(int(TOP["variants"][0]["pos"]) - int(TOP["variants"][1]["pos"]))
        S["levers"]["pair"] = dict(cdna_distance_nt=c2 - c1, genomic_distance_bp=gd, amplicon_nt=c2 - c1 + 300,
                                   fold_shorter=round(gd / max(c2 - c1, 1), 1))
        log(f"the alleles are {gd:,} bp apart in the genome but {c2 - c1:,} nt apart in the mRNA: one RT-PCR amplicon "
            f"of ~{c2 - c1 + 300:,} nt from the child's RNA phases them")
        try:
            prim = design_rtpcr_primers(m, c1, c2, fa)
            if prim:
                S["levers"]["pair"]["primers"] = prim
                S["levers"]["pair"]["amplicon_nt"] = prim[0]["product_nt"]
                md("**RT-PCR primers for the phasing / NMD-escape amplicon** (Primer3; no variant of the child and no common SNP "
                   "under either primer)\n\n" + df_md(pd.DataFrame(prim)))
        except Exception as e:      # noqa: BLE001
            log(f"RT-PCR primer design skipped ({type(e).__name__}: {str(e)[:120]})")
    if ptc and mis and prot and TOP["model"] == "compound_het":
        def put(seq, res, aa):
            return seq[:res - 1] + aa + seq[res:]
        mis_prot = put(prot, mis["residue"], mis["mut_aa"])          # missense allele: wild type at the stop site
        rt = [aa for aa in NEAR_COGNATE[ptc["stop_dna"]] if aa != ptc["wt_aa"]]
        restores = ptc["wt_aa"] in NEAR_COGNATE[ptc["stop_dna"]]
        downstream = mis["residue"] > ptc["residue"]
        rt_any = put(prot, ptc["residue"], rt[0]) if rt else prot  # stop allele, full length: wild type at the missense site
        peps = site_peptides({f"{mis['wt_aa']}{mis['residue']} (reference)":
                              (rt_any, "the stop allele — only by read-through, since the truncated product ends before this residue"
                               if downstream else "the stop allele (truncated product if NMD is relieved, or read-through)"),
                              mis["short"]: (mis_prot, "the missense allele")}, mis["residue"], avoid=ptc["residue"])
        forms = {f"{ptc['wt_aa']}{ptc['residue']} (reference)":
                 (mis_prot, "the missense allele" + (" (or read-through restoring the wild-type residue)" if restores else ""))}
        for aa in rt:
            forms[f"{ptc['wt_aa']}{ptc['residue']}{aa}"] = (put(prot, ptc["residue"], aa), f"read-through of the stop allele inserting {aa}")
        peps += site_peptides(forms, ptc["residue"], avoid=mis["residue"]) if rt else []
        S["levers"]["peptides"] = peps
        S["levers"]["peptide_logic"] = dict(missense_downstream_of_stop=downstream, near_cognate_restores_wild_type=restores)
        if peps:
            md("**Allele-resolved peptides for targeted mass spectrometry**\n\n" + df_md(pd.DataFrame(peps), floatfmt="{:.3f}"))
    if TOP["model"] == "compound_het":
        weak = min(TOP["variants"], key=lambda v: v["p_path"])
        strong = max(TOP["variants"], key=lambda v: v["p_path"])
        wk = f"{weak['chrom']}:{int(weak['pos'])} {weak['ref']}>{weak['alt']}"
        cls = next((x.get("classification") for x in (S.get("top") or {}).get("variants", []) if x.get("key") == wk), None)
        S["levers"]["reclassification"] = dict(allele=weak.get("hgvsp") or wk, current_class=cls,
                                               path=reclassification_path(weak, strong, float(TOP.get("pm3") or 0.0)))
        md("**Path to reclassifying the weaker allele**\n\n" + df_md(pd.DataFrame(S["levers"]["reclassification"]["path"])))

    # ---- scalability: the same triage over every ClinVar P/LP allele in the disorder's panel ---------------------------
    panel = [g] + [x for x in (S.get("pheno") or {}).get("dx_panel", []) if x != g]
    ATLAS = lever_atlas(panel, fa)
    if len(ATLAS):
        tab = ATLAS.groupby(["gene", "lever"]).size().unstack(fill_value=0)
        tab = tab[[c for c in LEVER_ORDER if c in tab.columns]]
        tab["total"] = tab.sum(1)
        tab = tab.sort_values("total", ascending=False)
        non = ATLAS[ATLAS.stop_codon_rna != ""]
        S["levers"]["atlas"] = dict(n_alleles=int(len(ATLAS)), n_genes=int(ATLAS.gene.nunique()),
                                    by_gene=tab.reset_index().to_dict("records"),
                                    by_lever=ATLAS.lever.value_counts().reindex([c for c in LEVER_ORDER if c in set(ATLAS.lever)]).to_dict(),
                                    nonsense_by_stop=non.stop_codon_rna.value_counts().to_dict(),
                                    top_gene_nonsense_by_stop=non[non.gene == g].stop_codon_rna.value_counts().to_dict(),
                                    n_no_criteria=int(ATLAS.review_status.fillna("").str.contains("no_assertion|no_classification").sum()))
        ATLAS.to_csv(OUT / "lever_atlas.csv", index=False)       # reusable product: one row per public ClinVar allele
        md(f"**Allele-specific lever atlas: {len(ATLAS)} ClinVar P/LP alleles in {ATLAS.gene.nunique()} genes of the panel**\n\n"
           + df_md(tab.reset_index()) + "\n\nSaved as `lever_atlas.csv` (public ClinVar alleles only).")
    try:
        lever_figure(g, L_out, len(prot), ptc, mis)
    except Exception as e:      # noqa: BLE001
        log(f"lever figure skipped ({type(e).__name__}: {str(e)[:120]})")
    log(f"allele-level triage: {json.dumps({k: v for k, v in S['levers'].items() if k in ('pair', 'atlas')}, default=str)[:600]}")

## 14c · Positive control: does the triage classifier recover published answers?

Stage 14b *asserts* that each allele maps to a particular lever. An assertion is not a method. This stage is the
positive control. A hand-curated table of **published premature-stop alleles in genes outside this child's panel** —
alleles for which the read-through literature already reports what happens — is pushed through the *same* functions
that triage the child's own alleles (`cds_model`, `coding_change`, `classify_clinvar`), and the predictions are
compared with the published answer.

* **What is held out.** Only the gene symbol and the HGVS coding change go in. The stop codon is recomputed from the
  GRCh38 reference and the MANE Select gene model; the published stop codon is the answer key and never an input.
* **Why the stop codon is the right thing to score.** It is the single quantity the whole read-through branch turns
  on: `STOP_ORDER` ranks UGA above UAG and UAA, `NEAR_COGNATE` decides which amino acids read-through would insert,
  and the +4 nucleotide modifies both. Get the codon wrong and every downstream claim about that allele is wrong.
* **Provenance.** Every PMID below was checked against Europe PMC (`EXT_ID:<pmid> AND SRC:MED`) while the table was
  assembled, and the allele is named in the cited paper's title or abstract. Candidates whose allele or citation
  could not be pinned down are listed in `unverified_dropped` rather than quietly discarded.
* **Public data only.** The child's VCF, genotypes and phenotype are not read here. The inputs are the public
  reference genome, the public MANE gene model and the literature table in the next cell.

The agreement figure is deliberately a *necessary* condition, not a sufficient one: it shows the classifier reads
the genome correctly, not that read-through will work in this child. The honest limits are reported with it.

In [ ]:
# Published premature-stop alleles with read-through literature, in genes that are not in this child's panel.
# `expected_stop` is the answer key: it follows from the published coding change and the genetic code, and is never
# read from the reference. `hgvsc` is written on the MANE Select transcript the pipeline uses, which for MECP2 is not
# the transcript the original report numbered against (see `note`).
PUBLISHED_PTC = [
    # --- cystic fibrosis (CFTR, NM_000492.4 / ENST00000003084; the legacy CF numbering is the MANE numbering) --------
    dict(gene="CFTR", hgvsc="c.178G>T", hgvsp="p.Glu60Ter", expected_stop="TAG", pmids=["36143233"],
         readthrough="achieved in cells (G418 or ELX-02 with CFTR modulators); an N-terminal allele that escapes NMD"),
    dict(gene="CFTR", hgvsc="c.263T>G", hgvsp="p.Leu88Ter", expected_stop="TGA", pmids=["36143233"],
         readthrough="achieved in cells (G418 or ELX-02 with CFTR modulators); an N-terminal allele that escapes NMD"),
    dict(gene="CFTR", hgvsc="c.366T>A", hgvsp="p.Tyr122Ter", expected_stop="TAA", pmids=["17394637", "28376702", "35487895"],
         readthrough="achieved; gentamicin read-through was at least five-fold that of the three UGA alleles tested "
                     "alongside it, and this allele does not need UPF2/UPF3 for its decay"),
    dict(gene="CFTR", hgvsc="c.1624G>T", hgvsp="p.Gly542Ter", expected_stop="TGA",
         pmids=["18272502", "24251786", "38725656", "17394637"],
         readthrough="achieved in a CF mouse model, in patient bronchial cells and by mass spectrometry of the product"),
    dict(gene="CFTR", hgvsc="c.1657C>T", hgvsp="p.Arg553Ter", expected_stop="TGA", pmids=["24251786"],
         readthrough="achieved, but the weakest of the four common CFTR stop alleles tested with NB124"),
    dict(gene="CFTR", hgvsc="c.3484C>T", hgvsp="p.Arg1162Ter", expected_stop="TGA",
         pmids=["24251786", "38725656", "17394637"],
         readthrough="achieved; among the most responsive CFTR alleles to synthetic aminoglycosides"),
    dict(gene="CFTR", hgvsc="c.3587C>G", hgvsp="p.Ser1196Ter", expected_stop="TGA", pmids=["38725656"],
         readthrough="achieved; Cys, Arg and Trp incorporation at the UGA confirmed by tandem mass spectrometry"),
    dict(gene="CFTR", hgvsc="c.3846G>A", hgvsp="p.Trp1282Ter", expected_stop="TGA",
         pmids=["24251786", "17290305", "17394637"],
         readthrough="achieved, but only in patients whose cells leave enough nonsense transcript for NMD to spare"),
    # --- Hurler syndrome (IDUA, NM_000203.5 / ENST00000514224) -------------------------------------------------------
    dict(gene="IDUA", hgvsc="c.208C>T", hgvsp="p.Gln70Ter", expected_stop="TAG", pmids=["11159948"],
         readthrough="achieved in patient fibroblasts (gentamicin), restoring 2.8 % of normal iduronidase activity"),
    dict(gene="IDUA", hgvsc="c.1205G>A", hgvsp="p.Trp402Ter", expected_stop="TAG", pmids=["11159948"],
         readthrough="achieved in patient fibroblasts (gentamicin), restoring 2.8 % of normal iduronidase activity"),
    # --- Rett syndrome (MECP2) ---------------------------------------------------------------------------------------
    dict(gene="MECP2", hgvsc="c.538C>T", hgvsp="p.Arg180Ter", expected_stop="TGA", pmids=["21120444"],
         readthrough="achieved in HeLa cells and in ear fibroblasts of a knock-in mouse (gentamicin; NB54 and NB84 "
                     "more effective)",
         note="published as p.Arg168Ter on NM_004992.4 (MANE Plus Clinical); the MANE Select transcript "
              "NM_001110792.2 numbers the same residue 180, so the coding change is restated on it"),
    # --- recessive dystrophic epidermolysis bullosa (COL7A1, NM_000094.4 / ENST00000681320) --------------------------
    dict(gene="COL7A1", hgvsc="c.751C>T", hgvsp="p.Gln251Ter", expected_stop="TAG", pmids=["25155989"],
         readthrough="achieved in patient keratinocytes (geneticin, gentamicin, paromomycin), restoring full-length "
                     "type VII collagen"),
    dict(gene="COL7A1", hgvsc="c.1732C>T", hgvsp="p.Arg578Ter", expected_stop="TGA", pmids=["25155989", "38366625"],
         readthrough="achieved in patient cells and in patients' skin after intravenous gentamicin"),
    # --- Herlitz junctional epidermolysis bullosa (LAMB3, NM_000228.3 / ENST00000356082) -----------------------------
    dict(gene="LAMB3", hgvsc="c.870T>A", hgvsp="p.Cys290Ter", expected_stop="TGA", pmids=["29946029"],
         readthrough="achieved in laminin-beta-3-null keratinocytes (gentamicin), restoring laminin 332 assembly"),
    dict(gene="LAMB3", hgvsc="c.1903C>T", hgvsp="p.Arg635Ter", expected_stop="TGA", pmids=["29946029"],
         readthrough="achieved in laminin-beta-3-null keratinocytes (gentamicin), restoring laminin 332 assembly"),
    # --- tumour-suppressor nonsense allele (TP53, NM_000546.6 / ENST00000269305) -------------------------------------
    dict(gene="TP53", hgvsc="c.637C>T", hgvsp="p.Arg213Ter", expected_stop="TGA", pmids=["31120902"],
         readthrough="achieved in cancer cells (G418, strongly potentiated by mefloquine), restoring functional p53"),
]

# Candidates assembled and then excluded, kept in the record so that the agreement figure cannot be read as a
# selection of the cases that happened to work.
VALIDATION_DROPPED = [
    dict(gene="DMD", allele="eight premature stop codons of Duchenne / Becker patients, two of them refractory to "
                            "aminoglycosides", pmids=["14991821"],
         reason="the paper reports the read-through level per allele but does not name the alleles, so no coding "
                "change could be attached to a citation"),
    dict(gene="DMD", allele="the mdx UAA stop codon", pmids=["10939566"],
         reason="a mouse allele; the human orthologous position is not the reported one"),
    dict(gene="COL7A1", allele="p.Arg163Ter", pmids=["25155989"],
         reason="c.487C>T does not give an arginine codon on MANE Select NM_000094.4 (codon 163 is CTG), so the "
                "published label and the coding change could not be reconciled"),
    dict(gene="CFTR", allele="the six common CFTR premature stops surveyed for inserted amino acids", pmids=["41015699"],
         reason="the six alleles are not named in the abstract, so they were not added as separate entries"),
]

# The priors the classifier encodes, and the published work they come from. Checked, not tuned: these thresholds are
# fixed before the run and no leaderboard or panel feedback touches them.
VALIDATION_BASIS = [
    dict(prior="UGA is read through more readily than UAG or UAA (`STOP_ORDER`)", pmids=["10939566", "41015699"]),
    dict(prior="a cytosine at +4 raises read-through; the order is C > U > A >= G (`plus4` in the triage)",
         pmids=["10939566", "22479203"]),
    dict(prior="read-through inserts the amino acids of near-cognate tRNAs, set by the stop codon (`NEAR_COGNATE`)",
         pmids=["25733896", "27702906"]),
    dict(prior="nonsense-mediated decay governs how much transcript is left for read-through to act on",
         pmids=["17290305", "35487895"]),
]

_PubRow = collections.namedtuple("_PubRow", "gene chrom pos ref alt mc")        # the fields `classify_clinvar` reads


def _hgvsc_snv(h):
    """(CDS position, reference base, alternate base) of a coding substitution written `c.<n><ref>><alt>`."""
    m = re.fullmatch(r"c\.(\d+)([ACGT])>([ACGT])", h or "")
    return (int(m.group(1)), m.group(2), m.group(3)) if m else None


def _hgvsp_ter(h):
    """(residue, one-letter wild-type amino acid) of a protein change written `p.<Xxx><n>Ter`."""
    inv = {b: a for a, b in AA3.items()}
    m = re.fullmatch(r"p\.([A-Za-z]{3})(\d+)Ter", h or "")
    return (int(m.group(2)), inv.get(m.group(1))) if m else None


def validate_ptc(e, fa, exclude=()):
    """Run one published allele through the production triage and compare with the published answer. Returns a row
    with `computed_stop=None` and a `skipped` reason whenever the allele cannot be placed on the gene model; never
    raises, so one bad row cannot cost the rest of the control."""
    g = e["gene"]
    row = dict(gene=g, hgvsc=e["hgvsc"], hgvsp=e.get("hgvsp", ""), expected_stop=e["expected_stop"],
               expected_stop_rna=e["expected_stop"].replace("T", "U"), computed_stop=None, computed_stop_rna="",
               lever=None, expected_lever=("read-through: UGA stop" if e["expected_stop"] == "TGA"
                                           else "read-through: UAG / UAA stop"),
               transcript="", residue=None, wt_aa="", plus4_rna="", in_frame_exon=None, match=False,
               lever_match=False, readthrough=e.get("readthrough", ""), pmids=list(e.get("pmids", [])),
               note=e.get("note", ""), skipped=None)
    if g in exclude:        # a gene of this child's own panel would stop being an independent control
        row["skipped"] = "gene is in this child's diagnostic panel, so it is not an independent control"
        return row
    if g not in GENE_TX or g not in GENE_POS:
        row["skipped"] = "gene absent from the MANE gene model"
        return row
    pr = _hgvsc_snv(e["hgvsc"])
    if pr is None:
        row["skipped"] = "coding change is not a simple substitution"
        return row
    c, ref_t, alt_t = pr
    m = cds_model(GENE_TX[g][0], fa)
    row["transcript"] = TX[m["tid"]].get("version") or m["tid"]
    if not 1 <= c <= len(m["seq"]):
        row["skipped"] = f"CDS position {c} is beyond the {len(m['seq'])} nt MANE coding sequence"
        return row
    plus = TX[m["tid"]]["strand"] == "+"
    pos = m["gpos"][c - 1]              # HGVS c. alleles are in transcript orientation; the triage wants genomic ones
    ref_g, alt_g = (ref_t, alt_t) if plus else (ref_t.translate(COMP), alt_t.translate(COMP))
    cc = coding_change(m, pos, ref_g, alt_g)
    if cc is None:
        row["skipped"] = "the reference base at this CDS position does not match the published coding change"
        return row
    pp = _hgvsp_ter(e.get("hgvsp", ""))
    if pp and (cc["residue"], cc["wt_aa"]) != pp:       # a second, independent check on the coordinate mapping
        row["skipped"] = (f"computed residue {cc['wt_aa']}{cc['residue']} does not match the published "
                          f"{pp[1]}{pp[0]}")
        return row
    r = _PubRow(gene=g, chrom=GENE_POS[g][0], pos=pos, ref=ref_g, alt=alt_g, mc="SO:0001587|nonsense")
    lever, stop, in_frame = classify_clinvar(r, m)      # the production classifier, unchanged
    row.update(computed_stop=stop, computed_stop_rna=(stop or "").replace("T", "U"), lever=lever,
               residue=int(cc["residue"]), wt_aa=cc["wt_aa"], plus4_rna=cc["plus4"].replace("T", "U"),
               in_frame_exon=bool(in_frame), match=bool(stop == e["expected_stop"]),
               lever_match=bool(lever == row["expected_lever"]))
    return row


with stage("Track 2: triage validation on published alleles"):
    if not CFG["RUN_TRACK2"]:
        raise RuntimeError("disabled in CFG")
    from pyfaidx import Fasta
    fav = Fasta(RES["ref"], as_raw=True, sequence_always_upper=True)
    own = {x for x in ((S.get("pheno") or {}).get("dx_panel") or []) if x} | {(S.get("levers") or {}).get("gene")}
    VROWS = [validate_ptc(e, fav, exclude=own) for e in PUBLISHED_PTC]      # plain dicts: already JSON-serialisable
    V = pd.DataFrame(VROWS)
    done = V[V.skipped.isna()]
    n_tested, n_match = int(len(done)), int(done.match.sum())
    n_lever = int(done.lever_match.sum())

    # ---- agreement, per gene and as a lever-vs-published-codon table ------------------------------------------------
    by_gene = [dict(gene=g, n_tested=int((d.skipped.isna()).sum()), n_codon_match=int(d.match.sum()),
                    n_skipped=int(d.skipped.notna().sum()),
                    published_codons=", ".join(sorted(set(d.expected_stop_rna))))
               for g, d in V.groupby("gene", sort=True)]
    conf = pd.crosstab(done.lever, done.expected_stop_rna) if n_tested else pd.DataFrame()
    if n_tested:
        conf = conf.reindex(index=[c for c in LEVER_ORDER if c in conf.index],
                            columns=[c for c in ("UGA", "UAG", "UAA") if c in conf.columns], fill_value=0)
    S["lever_validation"] = dict(
        n_entries=int(len(PUBLISHED_PTC)), n_tested=n_tested, n_codon_match=n_match,
        frac_codon_match=round(n_match / n_tested, 3) if n_tested else 0.0,
        n_lever_match=n_lever, n_skipped=int(len(V) - n_tested),
        genes=sorted(set(V.gene)), n_pmids=len({p for e in PUBLISHED_PTC for p in e["pmids"]}),
        n_pmids_cited=len({p for d in PUBLISHED_PTC + VALIDATION_DROPPED + VALIDATION_BASIS for p in d["pmids"]}),
        published_codon_counts=done.expected_stop_rna.value_counts().to_dict() if n_tested else {},
        by_gene=by_gene,
        confusion=conf.reset_index().to_dict("records") if n_tested else [],
        entries=VROWS,
        unverified_dropped=VALIDATION_DROPPED, basis=VALIDATION_BASIS,
        pmid_source="Europe PMC REST (EXT_ID:<pmid> AND SRC:MED), checked when the table was assembled")
    VOUT = V.assign(pmids=V.pmids.map(lambda p: "; ".join(p)), skipped=V.skipped.fillna(""))
    VOUT.to_csv(OUT / "lever_validation.csv", index=False)    # reusable product: the control set and its predictions

    md(f"**Positive control: {n_tested} published premature-stop alleles in {done.gene.nunique() if n_tested else 0} "
       f"genes outside this child's panel**\n\nOnly the gene symbol and the HGVS coding change were supplied; the stop "
       f"codon was recomputed from GRCh38 and the MANE Select model by the same `cds_model` / `coding_change` / "
       f"`classify_clinvar` path that triages the child's alleles.\n\n"
       + df_md(VOUT[["gene", "hgvsc", "hgvsp", "transcript", "residue", "expected_stop_rna", "computed_stop_rna",
                     "plus4_rna", "lever", "match", "pmids", "skipped"]], max_rows=len(VOUT)))
    if n_tested:
        md(f"**Computed lever against the published stop codon** (agreement "
           f"{n_match}/{n_tested} on codon identity, {n_lever}/{n_tested} on lever class)\n\n"
           + df_md(conf.reset_index()))
    md("**What this does and does not establish**\n\n"
       "The control shows that the triage places a published coding change on the right codon of the right "
       "transcript and assigns the lever its own rules imply - the necessary condition for every read-through claim "
       "downstream. It does not show that read-through will work in this child. Two honest limits: every allele that "
       "could be tied to a named citation is read-through-positive in the literature, with graded rather than binary "
       "efficiency, so the set cannot measure specificity against true negatives; and CFTR p.Tyr122Ter is a UAA "
       "allele whose reported gentamicin read-through exceeded that of three UGA alleles tested beside it, which is "
       "why `STOP_ORDER` is treated as a prior to be measured and not a rule - the +4 cytosine at that allele is the "
       "likely reason, and the triage reports the +4 nucleotide for exactly this purpose.\n\n"
       + df_md(pd.DataFrame([dict(prior=b["prior"], pmids="; ".join(b["pmids"])) for b in VALIDATION_BASIS]))
       + "\n\nSaved as `lever_validation.csv` (public alleles and public "
       "literature only). Candidates excluded while the table was assembled: "
       + "; ".join(f"{d['gene']} {d['allele']} ({d['reason']})" for d in VALIDATION_DROPPED) + ".")
    log(f"triage validation: {n_match}/{n_tested} published stop codons recovered "
        f"({S['lever_validation']['frac_codon_match']:.0%}), lever class {n_lever}/{n_tested}, "
        f"{S['lever_validation']['n_skipped']} skipped, {len(VALIDATION_DROPPED)} candidates dropped unverified, "
        f"codons tested: {json.dumps(S['lever_validation']['published_codon_counts'])}")

## 15 · Reports

Both write-ups are generated from the results of this run (nothing is typed in by hand), so they always match the
submission file. They are rewritten at the end of the notebook once the read-level module has finished.

* `<team>_track1_report.md` — methods write-up for Track 1, including the organisers' methods-template answers;
* `<team>_track2_report.md` — mechanism characterisation and ranked drug hypotheses for Track 2;
* `run_summary.json` — machine-readable summary (findings only; no genotype-scale data).

In [ ]:
REPORT_PMIDS = ["15475955", "16411201", "24344301", "20516114", "18932004", "15159543", "15208629", "23242215", "24825348",
                "19407811", "22698286", "23079597", "31053147", "29300386", "32720330", "30192042", "36413997", "40084623",
                "28518168", "38685113", "21552266", "28553959", "34009673"]


def fmt(x, nd=3):
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return "–"
    if isinstance(x, float):
        return f"{x:.{nd}g}"
    return str(x)


def clean_title(t):
    """Europe PMC titles carry HTML markup and entities (e.g. &lt;i&gt;): plain text for Markdown."""
    import html
    return re.sub(r"<[^>]+>", "", html.unescape(html.unescape(t or ""))).strip()


def cite(p, pm):
    v = pm.get(p)
    return f"{v['authors']} et al. {v['year']}, *{v['journal']}* (PMID {p})" if v else f"PMID {p}"


def run_table():
    return df_md(pd.DataFrame([dict(stage=k, status=v["status"], minutes=round(v["seconds"] / 60, 1), note=v["note"][:90])
                               for k, v in RUN.items()]), max_rows=60)


def wilson(k, n, z=1.96):
    """95 % Wilson score interval for k successes out of n."""
    if not n:
        return (None, None)
    ph = k / n
    d = 1 + z * z / n
    c = (ph + z * z / (2 * n)) / d
    h = z * math.sqrt(ph * (1 - ph) / n + z * z / (4 * n * n)) / d
    return (max(0.0, c - h), min(1.0, c + h))


def pct_ci(frac, n):
    lo, hi = wilson(round(frac * n), n)
    return f"{100 * frac:.0f} % (95 % CI {100 * lo:.0f}–{100 * hi:.0f})" if lo is not None else "–"


PARAMS = [   # (parameter, value, source) — what the ranking depends on, and where each value comes from
    ("ACMG evidence → points → posterior", "Bayesian point scale, prior 0.10, OddsPath 350", "Tavtigian 2018 / 2020 (literature)"),
    ("Missense predictor bands (PP3 / BP4)", "AlphaMissense and REVEL thresholds", "Pejaver 2022 ClinGen calibration (literature)"),
    ("PM2 / BS1 / BA1 frequency cut-offs", "≤ 1e-4 / > 1e-3 / ≥ 5 % (max of AF and FAF95)", "ClinGen SVI recommendations (literature)"),
    ("PM3 for the weaker allele", "+2 with a Pathogenic partner, +1 with a Likely pathogenic one, 0 when phase is unknown", "ClinGen SVI PM3 v1.0 (literature)"),
    ("PVS1 / NMD", "last-exon and 50-nt rules", "Abou Tayoun 2018 (literature)"),
    ("Phenotype factor", "(similarity / best similarity)^3", "heuristic, fixed before the run"),
    ("Mode-of-inheritance and phase factors", "e.g. 0.9 for unphased pairs", "heuristic, fixed before the run"),
    ("Indication prior", "genes on the stated diagnosis' panel floored at 0.95 of the best fit", "heuristic, fixed before the run; both rankings reported"),
    ("Other-gene rows in the CSV", "only when EPCR ≥ 0.25", "heuristic, fixed before the run"),
]


def competing_summary():
    others = [h for h in HYPS[1:30] if h["gene"] != HYPS[0]["gene"]]
    same = [h for h in HYPS[1:30] if h["gene"] == HYPS[0]["gene"]]
    lines = []
    if others:
        best = others[0]
        name = best["gene"] if reportable_gene(best["gene"]) else "a gene outside the indication (name withheld)"
        lines.append(f"The best hypothesis in any other gene is a {best['model'].replace('_', '-')} model in {name} with EPCR "
                     f"{best['epcr']:.3f} ({HYPS[0]['epcr'] / max(best['epcr'], 1e-6):.1f}-fold lower than the leading pair).")
    if same:
        n_sub = max(0, len(SUB_ROWS) - 1) if "SUB_ROWS" in globals() else None
        lines.append(f"{len(same)} lower-ranked alternative genotypes within {HYPS[0]['gene']} follow (other pairings and single alleles)"
                     + (f"; {n_sub} of them are submitted as fall-back rows so that a partially correct answer still scores "
                        "(a single allele already listed in a higher pair, and any variant with net benign evidence, are not submitted)."
                        if n_sub is not None else "."))
    return " ".join(lines)


# The organisers' required acknowledgement, verbatim from the Official Rules ("Citation and Acknowledgement"): any
# publication or public communication arising from the hackathon must include it.
ACK = ("This work was made possible through the Hackathon, organized by Sage Bionetworks in partnership with the MVA Society, "
       "Hugging Face, and BEACON (The Benchmarking, Evaluation, and Assessment Consortium for Science), with prize sponsorship "
       "from AWS and Anthropic. We are deeply grateful to the child and their family who generously contributed their data and "
       "their story to advance research into this rare disease. We acknowledge their trust in making this Hackathon possible.")


def track1_report():
    T, pm = S.get("top", {}), {**(S.get("track2", {}) or {}).get("pmids", {}), **S.get("report_pmids", {})}
    vs = T.get("variants", [])
    qc, an, rb, bench = S.get("qc", {}), S.get("aneuploidy", {}), S.get("robustness", {}), S.get("benchmark", {})
    ph, by, esm, st, rl = S.get("phase", {}), S.get("bystander", {}), S.get("esm", {}), S.get("structure", {}), S.get("reads", {})
    gene = T.get("gene", S.get("top_gene", "?"))
    sub = pd.read_csv(SUBMISSION_CSV, dtype=str, keep_default_na=False) if SUBMISSION_CSV.exists() else pd.DataFrame()
    L = []
    L.append(f"# {CFG['TEAM']} — Track 1 methods report: {CFG['MODEL_NAME']}\n")
    L.append(f"*Rare Disease, Real Kid: MVA Hackathon 2026 · generated automatically by the analysis notebook on "
             f"{time.strftime('%Y-%m-%d %H:%M UTC', time.gmtime())} · mode: {S.get('mode')}*\n")
    # ---------------- abstract ----------------
    v1 = vs[0] if vs else {}
    v2 = vs[1] if len(vs) > 1 else {}
    L.append("## Summary\n")
    L.append(
        f"We built **DuoRank**, a fully automated, public-data-only pipeline that ranks genotype hypotheses — homozygous, "
        f"compound-heterozygous, heterozygous and hemizygous — from a single-sample GRCh38 VCF and a list of HPO terms. "
        f"Variants are normalised and annotated offline (gnomAD v4.1, ClinVar, MANE consequences, AlphaMissense), scored in "
        f"the Bayesian point system of the ACMG/AMP guidelines (evidence strengths from published ClinGen calibrations; the few "
        f"ranking heuristics fixed before the run and listed in section 7), "
        f"and combined with a gene-level phenotype fit computed by information-content similarity against all "
        f"{fmt(S.get('pheno', {}).get('n_diseases'))} annotated Mendelian diseases. A short list is then re-annotated "
        f"(Ensembl VEP, exact gnomAD records, SpliceAI, ESM-2, AlphaFold structure), phase is analysed, and the raw "
        f"reads are optionally re-aligned around the leading locus for independent validation.\n")
    if T:
        L.append(
            f"**Result.** The leading hypothesis is a **{T['model'].replace('_', '-')} genotype in {gene}** "
            f"({T.get('dx_disease') or T.get('disease')}, {'/'.join(T.get('inheritance') or [])}): "
            + " + ".join(f"`{v['chrom']}:{v['pos']} {v['ref']}>{v['alt']}` ({v.get('hgvsc', '').split(':')[-1]} {v.get('hgvsp', '')}, "
                         f"automated class **{v['classification']}**)" for v in vs)
            + f", EPCR {T['epcr']:.3f}. Without the stated diagnosis (HPO terms only) the gene ranks "
              f"#{T.get('pheno_rank_blind')} of {fmt(S.get('pheno', {}).get('n_genes'))} on phenotype alone and the same genotype is "
              f"{'still' if S.get('top_same_blind') else 'not'} ranked first. {competing_summary()}\n")
    # ---------------- data ----------------
    L.append("## 1. Data and question\n")
    L.append(f"* Proband genome: single-sample WGS VCF, GRCh38 (Sentieon Haplotyper + GATK hard filters); raw FASTQ (4 lanes, paired).")
    L.append("* Phenotype (HPO): " + "; ".join(f"{h.get('label')} ({h['term']})" for h in S.get("hpo", []) if not h.get("relative"))
             + ". Terms that describe relatives rather than the proband were excluded from the similarity and are not listed here.")
    L.append(f"* Stated clinical indication (used as a transparent prior, both rankings reported): {S.get('diagnosis') or 'none'}")
    L.append(f"* Genome QC: Ti/Tv {fmt(qc.get('titv'))}, het/hom {fmt(qc.get('het_hom_ratio'))}, median depth at het sites "
             f"{fmt(qc.get('median_depth_het_sites'))}×, inferred sex {qc.get('inferred_sex')}, autozygosity F_ROH {fmt(qc.get('f_roh'))} "
             f"({qc.get('autozygosity_note', '')}).\n")
    # ---------------- method ----------------
    L.append("## 2. Method\n")
    L.append("| Step | What is done | Key choices |\n|---|---|---|")
    L.append("| Normalise | primary chromosomes, `chr` names, split multi-allelics, left-align (`bcftools norm`) | exact allele representation matches the scorer |")
    L.append(f"| Rarity | {S.get('af_source', 'gnomAD')} | ClinVar P/LP alleles exempt below 5 % (founder / hypomorphic alleles); VCF `INFO/AF` (within-sample) never used |")
    L.append(f"| Consequence | `bcftools csq` on MANE {S.get('mane_version', '')} transcripts; automated PVS1 / NMD rule | last-exon and 50-nt rule; LoF mechanism from ClinVar |")
    L.append("| Missense | AlphaMissense, ClinGen-calibrated bands (one predictor only; REVEL/CADD shown, not counted) | avoids double counting |")
    L.append("| Splicing | SpliceAI (pre-computed via VEP and re-run on GPU over a ±500 bp window for the leading genes) | non-coding second hits are scored, not filtered |")
    L.append(f"| Phenotype | Resnik best-match similarity vs {fmt(S.get('pheno', {}).get('n_diseases'))} HPO-annotated diseases (HPO {S.get('hpo_version')}) | relatives' terms excluded; indication used as a panel prior |")
    L.append("| Genotypes | hom / comp-het / het / hemi per gene; EPCR = P(genotype) × phenotype fit × mode fit × call quality | PM3 per ClinGen SVI; phase factor from read-backed tags |")
    L.append("| Second-hit rescue | rare non-coding variants in and around genes that already carry one credible allele | catches regulatory / deep-intronic hypomorphs |")
    L.append("| Output | top hypothesis + within-gene alternatives; other genes only if EPCR ≥ 0.25 | file parsed and self-scored by a re-implementation of the organisers' evaluation script before saving |\n")
    # ---------------- leading hypothesis ----------------
    if vs:
        L.append(f"## 3. The leading hypothesis in detail\n")
        L.append("| | " + " | ".join(f"allele {i + 1}" for i in range(len(vs))) + " |")
        L.append("|---|" + "---|" * len(vs))
        rows = [("GRCh38", lambda v: f"{v['chrom']}:{v['pos']} {v['ref']}>{v['alt']}"),
                ("HGVS (MANE)", lambda v: f"{v.get('mane', '')}:{v.get('hgvsc', '').split(':')[-1]} {v.get('hgvsp', '')}"),
                ("consequence", lambda v: f"{v['csq']}" + (f", exon {v['exon']}" if v.get("exon") else "")
                 + (f"; NMD {'escaped' if v.get('nmd_escape') else 'predicted'}" if v["csq"] in LOF_CSQ else "")),
                ("genotype / reads", lambda v: f"{v['zyg']}, AD {v.get('ad')}, GQ {fmt(v.get('gq'))}, FILTER {v.get('filt')}"),
                ("gnomAD v4.1", lambda v: (f"AC {fmt((v.get('gnomad') or {}).get('ac'))} / AN {fmt((v.get('gnomad') or {}).get('an'))}, "
                                           f"homozygotes {fmt((v.get('gnomad') or {}).get('nhomalt'))}, FAF95 {fmt((v.get('gnomad') or {}).get('faf95'))}")
                 if (v.get("gnomad") or {}).get("present") else "absent"),
                ("ClinVar", lambda v: (v.get("clnsig") or "not in ClinVar").replace("_", " ") + (f" ({v.get('clnrevstat', '').replace('_', ' ')})" if v.get("clnsig") else "")),
                ("AlphaMissense / REVEL / CADD", lambda v: f"{fmt(v.get('am'))} / {fmt(v.get('revel'))} / {fmt(v.get('cadd'))}"),
                ("SpliceAI max Δ", lambda v: fmt(v.get("spliceai"))),
                ("ACMG points → posterior", lambda v: f"{v['points']:g} → {v['posterior']:.3f}"),
                ("automated classification", lambda v: f"**{v['classification']}**")]
        for name, f in rows:
            L.append(f"| {name} | " + " | ".join(f(v) for v in vs) + " |")
        L.append("")
        for i, v in enumerate(vs, 1):
            L.append(f"*Allele {i} evidence:* " + "; ".join(f"{e['criterion']} ({e['points']:+g}: {e['basis']})" for e in v["evidence"]) + "\n")
        L.append(f"**Phenotype fit.** {gene} → {T.get('disease')} (similarity {T.get('pheno_score')}); rank #{T.get('pheno_rank_blind')} "
                 f"of {fmt(S.get('pheno', {}).get('n_genes'))} genes on HPO terms alone. "
                 + (f"Green gene on {len(S.get('panelapp', []))} Genomics England PanelApp panels, including "
                    + ", ".join(p["panel"] for p in S.get("panelapp", [])[:4]) + "." if S.get("panelapp") else "") + "\n")
        if by:
            L.append(f"**Is the weaker allele a coincidental bystander?** In gnomAD v4.1, {by['n_rare_variants']} rare (< 0.01 %) "
                     f"{by['variant_class']} variants in {gene} have a cumulative frequency of {by['cumulative_af']:.4f}, so an unrelated "
                     f"person carries one by chance with probability ≈ {by['carrier_probability']:.3f}"
                     + (f" (≈ {by['carrier_probability_strict']:.4f} counting only alleles at least as damaging by AlphaMissense)"
                        if "carrier_probability_strict" in by else "")
                     + f". Given a pathogenic first allele, the posterior that this allele is the second causal allele (rather than a "
                       f"bystander next to an undetected second hit) is {by['posterior_second_allele']['prior_0.5']:.3f} with an even prior "
                       f"on 'second hit visible as a rare coding variant', and {by['posterior_second_allele']['prior_0.2']:.3f} with a "
                       f"sceptical prior of 0.2.\n")
        if esm:
            L.append(f"**Protein language model (orthogonal, not counted in the points).** ESM-2 650M log-likelihood ratio for "
                     f"{esm.get('change')}: {esm.get('llr')}"
                     + (f"; more damaging than {esm.get('percentile_more_damaging_than')} % of all possible substitutions in the protein"
                        if esm.get("percentile_more_damaging_than") is not None else "")
                     + (f"; median for {esm.get('clinvar_plp_missense_n')} ClinVar P/LP missense variants in the gene: {esm.get('clinvar_plp_median_llr')}"
                        if (esm.get("clinvar_plp_missense_n") or 0) >= 5 else
                        f" (the gene has only {esm.get('clinvar_plp_missense_n') or 0} ClinVar P/LP missense allele(s), too few for a "
                        "gene-specific reference)")
                     + ". On its own this is moderate, not decisive, evidence of damage — consistent with a hypomorphic rather than a "
                       "null allele, which is what a second allele in a recessive disorder whose null/null genotype is lethal is expected to be.\n")
        if st:
            L.append(f"**Structure.** Residue {st.get('residue')} lies in: {st.get('domain')}; AlphaFold pLDDT {st.get('plddt')} "
                     f"({st.get('neighbours_within_10A')} residues within 10 Å). "
                     + ("Nearest ClinVar P/LP missense residues: " + ", ".join(f"{x['residue']} ({x['distance_A']} Å)" for x in st["nearest_clinvar_plp_missense"]) + ". "
                        if st.get("nearest_clinvar_plp_missense") else "")
                     + (f"UniProt lists {st.get('n_disease_missense_uniprot')} literature-reported patient missense alleles in the protein; the nearest are "
                        + ", ".join(f"{x['change']} ({x['disease']}, {x['distance_A']} Å)" for x in st["disease_missense_uniprot"][:3])
                        + (f" — reported {st['disease_missense_uniprot'][0]['disease']} missense alleles cluster around this residue in the same "
                           "domain, the expected location of a second, hypomorphic allele"
                           if sum(1 for x in st["disease_missense_uniprot"] if x["distance_A"] <= 30 and x.get("same_domain")) >= 3 else "")
                        + ". " if st.get("disease_missense_uniprot") else "") + "\n")
        if ph:
            L.append(f"**Phase.** {ph.get('verdict', '')}. The alleles are {fmt(ph.get('distance_bp'))} bp apart with "
                     f"{fmt(ph.get('n_het_sites_between'))} heterozygous call(s) between them in the provided VCF (largest gap "
                     f"{fmt(ph.get('largest_gap_bp'))} bp; the read-level re-call below counts sites independently). "
                     f"Trans configuration is therefore **inferred, not observed**; it would be confirmed by {ph.get('confirmation')}.\n")
        if rl:
            L.append("**Read-level validation from the raw FASTQ.** " + rl.get("summary", "") + "\n")
        if gene == "BUB1B" and T.get("model") == "compound_het":
            L.append(f"**Why this genotype fits the known biology.** MVA1 is caused by biallelic BUB1B variants, typically one "
                     f"truncating allele with one missense allele, often in the kinase domain ({cite('15475955', pm)}; "
                     f"{cite('16411201', pm)}). A complete loss of BubR1 is embryonic-lethal in mice, so live-born patients keep some "
                     f"function, and BubR1 dosage tracks the phenotype: a graded reduction first causes premature chromatid separation "
                     f"and then aneuploidy ({cite('18932004', pm)}), and hypomorphic mice age prematurely ({cite('15208629', pm)}). "
                     f"Patient kinase-domain missense proteins are unstable rather than catalytically dead ({cite('20516114', pm)}). "
                     f"The child's pair — a predicted-NMD nonsense allele with a kinase-domain missense allele — is therefore the expected "
                     f"null + hypomorph architecture. Some families carry a regulatory BUB1B allele instead ({cite('24344301', pm)}); this is why "
                     "rare non-coding variants around genes with one credible allele are rescued and scored.\n")
        rc = (S.get("levers") or {}).get("reclassification")
        if rc and rc.get("path"):
            L.append(f"**Path to a firm classification of {rc['allele']}.** Points are applied cumulatively (Tavtigian point scale; "
                     "PM3 per ClinGen SVI):\n\n" + df_md(pd.DataFrame(rc["path"])) + "\n")
        L.append("**What a clinical laboratory would do to confirm.** (1) Sanger confirmation of both alleles. (2) Phase: targeted "
                 "testing of either parent (the standard route, which also supplies PM3)"
                 + (f"; or, because the alleles are only {fmt(ph.get('cdna_distance_nt'))} nt apart in the mRNA, one RT-PCR amplicon from "
                    "cultured cells with an NMD inhibitor — Primer3 primers are given in the Track 2 report" if ph.get("cdna_distance_nt") else "")
                 + ". (3) Function: BUBR1 protein level and mitotic-checkpoint competence in the child's cells (PS3), and premature "
                   "chromatid separation / mosaic aneuploidy on cytogenetics (PP4). (4) Cascade testing for carriers in the family, "
                   "offered through the clinical team.\n")
        if len(sub):
            L.append("**Submitted rows**\n\n" + df_md(sub))
            L.append("")
    # ---------------- certainty ----------------
    L.append("## 4. How robust is the ranking?\n")
    if rb:
        wp = rb.get("weight_perturbation", {})
        L.append(f"* Every heuristic weight multiplied by a random factor in [½, 2] ({wp.get('n')} runs): the top hypothesis is unchanged in "
                 f"{100 * (wp.get('top1_unchanged') or 0):.0f} % of runs (worst rank {wp.get('worst_rank')}).")
        L.append(f"* Rank of the leading genotype without ClinVar: {rb.get('rank_without_clinvar')}; without the missense predictor: "
                 f"{rb.get('rank_without_missense_predictor')}; without the stated diagnosis: {rb.get('rank_without_diagnosis')}; "
                 f"without both ClinVar and the diagnosis: {rb.get('rank_without_clinvar_and_diagnosis')}.")
        loo = rb.get("leave_one_term_out_blind", [])
        if loo:
            L.append("* Leave-one-HPO-term-out (diagnosis-blind): " + ", ".join(f"{x['label']} → rank {x['rank_of_top']}" for x in loo) + ".")
    L.append(f"* {competing_summary()}")
    L.append("* Hypotheses in genes outside the stated indication are counted but not named in this report: the organisers are still "
             "deciding, with the IRB, how incidental findings about this child may be published.\n")
    # ---------------- aneuploidy ----------------
    if an:
        L.append("## 5. What the bulk genome can say about the aneuploidy itself\n")
        lim = an.get("limit", {})
        top_dev = sorted((r for r in an.get("table", []) if max(abs(r.get("z_depth") or 0), abs(r.get("z_baf") or 0)) >= 2),
                         key=lambda r: -max(abs(r.get("z_depth") or 0), abs(r.get("z_baf") or 0)))[:3]
        L.append(f"Per-chromosome depth (GC-corrected) and allele-balance dispersion at heterozygous SNVs showed no chromosome-wide "
                 f"imbalance passing the pre-set threshold (flagged chromosomes: {an.get('flagged') or 'none'}; {an.get('artefact_bins')} "
                 "collapsed-repeat bins excluded"
                 + (f"; the largest deviations, chromosome(s) {', '.join(str(r['chrom']) for r in top_dev)} at |z| ≤ "
                    f"{max(max(abs(r['z_depth']), abs(r['z_baf'])) for r in top_dev):.1f}, stay below the threshold of 4 and the raw depth "
                    f"still correlates with GC content (r = {lim.get('gc_correlation')}), so residual GC bias is the likelier explanation"
                    if top_dev else "") + "). With a "
                 f"between-chromosome SD of {lim.get('sd_depth_ratio')} in depth ratio, a clonal whole-chromosome gain or loss present in "
                 f"more than ~{100 * (lim.get('min_detectable_clone_fraction') or 0):.0f} % of cells would have been detected. "
                 "This is the expected result in mosaic *variegated* aneuploidy: different cells carry different gains and losses, which "
                 "average out in bulk DNA. Bulk WGS can therefore neither confirm nor exclude the cytogenetic phenotype — metaphase "
                 "karyotyping, interphase FISH or single-cell low-pass sequencing are the appropriate assays.\n")
        if "aneuploidy_screen" in S.get("figures", {}):
            L.append(f"![aneuploidy screen]({S['figures']['aneuploidy_screen']})\n")
    # ---------------- benchmark ----------------
    if bench:
        L.append("## 6. Does it generalise?\n")
        L.append(f"On {bench['n_cases']} synthetic recessive cases spiked into the public GIAB HG002 genome (ClinVar truncating + missense "
                 f"pair; 6 sampled HPO terms with imprecision and one noise term; no diagnosis given), the true pair was ranked first in "
                 f"{pct_ci(bench['aware']['top1'], bench['n_cases'])} (top-10: {100 * bench['aware']['top10']:.0f} %) of cases, and in "
                 f"{pct_ci(bench['blind']['top1'], bench['n_cases'])} (top-10: {100 * bench['blind']['top10']:.0f} %) when the spiked alleles' "
                 "ClinVar records were hidden. The benchmark uses only public data and is regenerated by the notebook. It is deliberately "
                 "simple and therefore optimistic: the HPO terms are sampled from the same annotations that the phenotype engine scores, every "
                 "case is a coding truncating + missense recessive pair, the hidden alleles are still ones on which the missense predictor's "
                 "calibration was built, and a GIAB benchmark genome is cleaner than a clinical callset. It shows that the machinery works "
                 "end to end, not the accuracy to expect on an unsolved case.\n")
        if "benchmark" in S.get("figures", {}):
            L.append(f"![benchmark]({S['figures']['benchmark']})\n")
    # ---------------- limitations ----------------
    L.append("## 7. Limitations\n")
    L.append("* Phase of the two alleles is inferred from the genetic model and population data, not observed; parental testing or a "
             "long-read amplicon would settle it.\n* The missense allele remains a variant of uncertain significance by automated ACMG "
             "criteria; its causal role rests on the pair-level argument (rarity, predictor, kinase-domain location, phenotype match).\n"
             "* Structural variants and repeat expansions genome-wide are not assessed (read-level screen covers the leading locus only).\n"
             "* Phenotype similarity depends on the completeness of HPO annotations (several of the child's features are not annotated "
             "to the disorder).\n* EPCR is a ranking score on a probability scale (an ACMG posterior × phenotype, mode, phase and quality "
             "factors). It is not normalised across hypotheses (a single allele and a pair containing it overlap) and it is not calibrated "
             "against clinical outcomes, so it orders hypotheses rather than stating a clinical risk.\n"
             "* Robustness checks perturb the heuristic weights, not the ACMG prior and odds, which dominate the score.\n")
    L.append("**Where every number comes from.** Literature parameters were taken as published; heuristic ones were fixed before the "
             "challenge data were opened and not changed afterwards.\n")
    L.append(df_md(pd.DataFrame(PARAMS, columns=["parameter", "value", "source"]), max_rows=20) + "\n")
    # ---------------- data handling ----------------
    L.append("## 8. Data handling and AI disclosure\n")
    L.append("* All genotype-scale files lived in ephemeral scratch storage and were deleted at the end of the run; only findings "
             "(this report, the ranked CSV, figures) were kept. Only short lists of candidate variants were sent to public annotation "
             "services (Ensembl VEP, gnomAD), no genotype tables.")
    L.append(f"* Generative AI: {CFG['LLM_DISCLOSURE']}\n")
    # ---------------- template ----------------
    L.append("## 9. Methods-template answers\n")
    t_min = sum(v["seconds"] for v in RUN.values()) / 60
    L.append("| Question | Answer |\n|---|---|")
    L.append(f"| Team name | {CFG['TEAM']} |\n| Model number | 1 ({CFG['MODEL_NAME']}) |")
    L.append(f"| Generative AI used | {CFG['LLM_DISCLOSURE']} |")
    L.append("| Automated output or manual curation? | Fully automated output of the notebook; no manual review or re-ranking. Evidence strengths follow published ClinGen calibrations; the remaining heuristics (section 7) were fixed before the challenge data were opened; no parameter was tuned on leaderboard feedback. |")
    L.append("| Public data only? | Yes: gnomAD v4.1, ClinVar, MANE, Ensembl VEP, AlphaMissense, SpliceAI, ESM-2, AlphaFold DB, UniProt, HPO, PanelApp, GIAB HG002 (benchmark). |")
    L.append("| Compound heterozygous pairs? | Yes — pairs are enumerated per gene and scored jointly with a phase factor; single alleles are kept as fall-back rows. |")
    L.append("| Secondary / incidental findings | Dominant cancer-predisposition and other genes are evaluated as competing primary hypotheses; no secondary rows are submitted and names of genes outside the indication are withheld pending the organisers' IRB guidance. |")
    hw = f"{NCPU} vCPU" + (f", {N_GPU}× GPU" if N_GPU else ", no GPU")
    L.append(f"| Run time and cost | {t_min:.0f} min on a free Kaggle notebook ({hw}); no paid compute. |\n")
    # ---------------- run log + references ----------------
    L.append("## 10. Run log\n")
    L.append(run_table() + "\n")
    L.append("## References (resolved live from Europe PMC)\n")
    for p in REPORT_PMIDS:
        if pm.get(p):
            L.append(f"* {cite(p, pm)} — {clean_title(pm[p]['title'])}")
    L.append("\n*" + ACK + "*")
    return "\n".join(L)


def _allele(lv, kind):
    return next((a for a in lv.get("alleles", []) if a.get("kind") == kind), None)


def fragment_note(ptc, pm):
    """How the truncated product compares with the published BUBR1 truncations (native residues kept, not the X number)."""
    kept = ptc.get("native_residues_kept")
    if kept is None:
        return ""
    if kept >= 715:
        return (f"the truncated product keeps {kept} native residues, all {715} of a truncation that still supported the "
                f"checkpoint (731X; the authors note it 'contains only 715 amino acids of BUBR1 sequence', so its last residues are not native), "
                f"plus {kept - 715} more; another truncation (753X) was expressed only weakly, so the stability of this one is an open question "
                f"({cite('20516114', pm)}). A stable fragment that keeps the CDC20-, BUB3- and PP2A-B56-binding motifs but lacks the "
                "pseudokinase could also sequester those partners (a dominant-negative risk), which the cell assay must exclude")
    return (f"the truncated product keeps {kept} native residues, fewer than the 715 of the shortest truncation shown to support the "
            f"checkpoint (731X) ({cite('20516114', pm)})")


def track2_summary(gene, lv, t2, pm):
    """One-paragraph summary: genotype → mechanism → lever → candidate → the experiment that tests it → scale."""
    if not lv:
        return ("The allele-level triage did not run in this notebook run (see the run log), so the summary is limited to the "
                "literature layer and the network pass below.")
    ptc, mis, pair, at = _allele(lv, "premature stop"), _allele(lv, "missense"), lv.get("pair") or {}, lv.get("atlas") or {}
    net = t2.get("network") or {}
    net_ok = net.get("n_network_nodes") is not None
    net_neg = net_ok and not (net.get("verdicts") or {}).get("plausibly beneficial direction")
    alleles = [a for a in lv.get("alleles", []) if a.get("kind")]
    if not ptc:
        return (f"The leading {gene} genotype has no premature stop codon among the alleles the triage could map, so the stop-codon "
                "levers do not apply; the hypotheses below come from the literature layer and the network pass.")
    second = (f"a missense allele ({mis['label']})" if mis else "a second allele" if len(lv.get("alleles", [])) > 1 else None)
    parts = [f"The child carries a nonsense allele ({ptc['label']})" + (f" and {second}" if second else "") + f" in {gene}. "
             + (f"No approved drug acts on the {gene} network in a direction that would help (section 5), so we asked instead "
                "what can still be done with each allele. " if net_neg else
                "Besides the network pass (section 5), we asked what can still be done with each allele. ")
             + f"The nonsense allele creates a {ptc['stop_rna']} stop"
             + (", the stop codon most prone to drug-induced read-through" if ptc["stop_rna"] == "UGA" else "")
             + ("; its mRNA is predicted to be removed by nonsense-mediated decay (NMD). " if ptc.get("nmd_escape") is False
                else "; it is predicted to escape nonsense-mediated decay. " if ptc.get("nmd_escape") else ". ")]
    if gene == "BUB1B":
        parts.append("\n\nTwo approved medicines act on that allele by different routes: **azithromycin**, which in a reporter assay stimulated "
                     "read-through specifically at UGA stops — the class this allele belongs to — and which, in mice carrying a germline "
                     "nonsense allele of a tumour-suppressor gene, restored the full-length protein and prevented tumours "
                     f"({cite('37276642', pm)}; {cite('34066907', pm)}); and **amlexanox**, which "
                     "both relieves NMD and induces read-through, and restored full-length protein from the nonsense allele of a "
                     f"missense + nonsense genotype in another gene, though in a heterologous expression system ({cite('29247835', pm)}). ")
    nc = {k: v for k, v in (ptc.get("near_cognate") or {}).items() if not v.get("restores_wild_type")}
    if nc:
        ins = ", ".join(f"{ptc['wt_aa']}{ptc['residue']}{k} LLR {v['llr']:.1f}" for k, v in nc.items())
        parts.append(f"ESM-2 scores the amino acids that read-through would insert at residue {ptc['residue']} ({ins}); these are "
                     "comparable with one another at this residue, not with scores elsewhere in the protein. ")
    if gene == "BUB1B" and fragment_note(ptc, pm):
        kept = ptc["native_residues_kept"]
        parts.append(f"\n\nIf NMD alone is relieved, a fragment of {kept} native residues would remain. "
                     + ("That includes everything in a truncation published to support the checkpoint " if kept >= 715 else
                        "That is fewer residues than the shortest truncation published to support the checkpoint ")
                     + f"({cite('20516114', pm)}); whether it is stable, functional, or instead sequesters its binding partners is "
                     "untested, and section 2 sets out how to find out. ")
    if pair:
        parts.append("\n\nThese predictions can be tested in the child's own cells in one experiment (section 2): one RT-PCR amplicon "
                     "spanning both alleles phases them and measures how much nonsense-allele mRNA escapes NMD"
                     + ("; allele-resolved peptides measure the protein each allele makes. " if lv.get("peptides") else ". "))
    if at:
        parts.append(f"The same triage over the {at['n_alleles']} ClinVar P/LP alleles in the {at['n_genes']} genes of the disorder's panel "
                     "that have any catalogued alleles shows how many other catalogued alleles share each lever (section 7), and it "
                     "runs unchanged for any gene.")
    return "".join(parts)


def predictions_table(lv, pm, gene):
    ptc, mis = _allele(lv, "premature stop"), _allele(lv, "missense")
    prot_name = "BUBR1" if gene == "BUB1B" else f"{gene} protein"
    rows = []
    if lv.get("pair"):
        rows.append(("baseline (untreated cells ± puromycin / cycloheximide)", f"RT-PCR amplicon ({lv['pair']['amplicon_nt']:,} nt)",
                     "two allele-specific molecule classes if the alleles are in trans; the nonsense-allele fraction rises when NMD is blocked"))
    if ptc:
        agent = "azithromycin; amlexanox" if gene == "BUB1B" else "NMD-inhibitor / read-through agent"
        rows.append((f"{agent}, dose series", "allelic ratio on the same amplicon" if lv.get("pair") else "nonsense-allele mRNA (RT-qPCR)",
                     "nonsense-allele mRNA fraction rises"))
        if lv.get("peptides"):
            down = (lv.get("peptide_logic") or {}).get("missense_downstream_of_stop")
            rows.append((f"{agent}; gentamicin (positive control)", "allele-resolved peptides (section 2)",
                         "read-through peptides appear only with drug"
                         + ("; so does the reference missense-site peptide, which only a full-length product of the stop allele can carry"
                            if down else "")))
        if ptc.get("truncated_kda"):
            rows.append((agent, f"immunoblot, N-terminal {prot_name} antibody",
                         f"a ~{ptc['truncated_kda']:.0f} kDa fragment (residues {ptc['truncated_product']}) if NMD is relieved and the fragment is stable"))
    if mis and gene == "BUB1B":
        rows.append(("niacin (nicotinamide as comparator)", "BUBR1 half-life (cycloheximide chase), K668 acetylation",
                     "longer half-life; more BUBR1 protein; no growth advantage for the aneuploid fraction"))
    if ptc and mis and gene == "BUB1B":
        # the two levers act on different copies of the gene, so the combination is the natural test (approved drugs only)
        rows.append(("azithromycin + niacin together (both copies at once)", "total BUBR1 (immunoblot), premature chromatid separation",
                     "the effects add, because each drug acts on a different copy; if they do not, one of the two levers is not working"))
    if gene == "BUB1B":
        rows.append(("every arm vs untreated and parental cells", "nocodazole arrest, premature chromatid separation, micronuclei",
                     "move toward parental values; no worsening of segregation (TBK1 inhibition by amlexanox is the specific risk)"))
    else:
        rows.append(("every arm vs untreated and parental cells", f"{prot_name} level and a disease-relevant functional readout",
                     "move toward parental values"))
    return "| Arm | Readout | Pre-registered prediction if the lever works |\n|---|---|---|\n" + "\n".join(f"| {a} | {b} | {c} |" for a, b, c in rows)


def track2_report():
    t2, pm = S.get("track2", {}), {**(S.get("track2", {}) or {}).get("pmids", {}), **S.get("report_pmids", {})}
    T, lv, st = S.get("top", {}), S.get("levers", {}) or {}, S.get("structure", {}) or {}
    gene, net = T.get("gene", S.get("top_gene")), t2.get("network", {})
    ptc, mis = _allele(lv, "premature stop"), _allele(lv, "missense")
    used = set()

    def c(p):
        used.add(p)
        return cite(p, pm)

    L = [f"# {CFG['TEAM']} — Track 2 report: from the {gene} genotype to approved-drug hypotheses\n",
         f"*Generated automatically by the analysis notebook on {time.strftime('%Y-%m-%d', time.gmtime())}. Hypotheses for follow-up, "
         "not evidence that any medicine works; nothing here is medical advice, and no agent named here is proposed for use in this "
         "child. Every candidate below is a laboratory question to be answered in patient-derived cells first.*\n",
         "> **Thesis.** When no approved drug fits the protein, treat the *alleles*. This child's two "
         f"{gene} alleles — a premature stop and a destabilising missense — each have a distinct, approved-drug lever, and a single "
         "experiment in the child's own cultured cells tests both levers while also supplying the evidence that would firm up the "
         "diagnosis.\n"]
    # what is actually new here, said plainly: the panel scores innovation, and a method is easy to miss in prose
    if lv.get("alleles"):
        L.append("## What is new here\n")
        L.append("Conventional drug repurposing asks which drug acts on the protein a gene makes. For a loss-of-function "
                 "disorder that question often has no answer, and section 5 shows it returning none here. This report asks a "
                 "different question — what can still be done with each *allele* — and makes it computable:\n\n"
                 "* an allele-level triage that reads the stop codon, the decay prediction, the amino acids read-through would "
                 "insert, the exon's frame and the predicted structure, and assigns each allele a lever; it is gene-agnostic and "
                 "runs from public data alone;\n"
                 "* a protein language model used not to score the variant the child has, but to score the residues a drug would "
                 "*introduce*, which is what decides whether read-through is worth attempting;\n"
                 "* one RT-PCR amplicon, designed automatically with the primers printed in section 2, that answers the phase "
                 "question, the decay question and the drug question from the same cells;\n"
                 "* allele-resolved peptides whose masses differ, so the protein made by each copy of the gene can be measured "
                 "separately rather than inferred;\n"
                 "* the same triage run across every catalogued disease-causing allele in the disorder's gene panel, so the "
                 "answer for this child indexes the other children it would apply to.\n")
    # a one-screen map of the argument, before the prose: judges skim before they read
    if lv.get("alleles"):
        ptc0, mis0, pair0 = _allele(lv, "premature stop"), _allele(lv, "missense"), lv.get("pair") or {}
        top = [h for h in (t2.get("curated") or []) if h.get("rank") in (1, 2, 3)]
        L.append("## At a glance\n")
        L.append("| | Allele 1 | Allele 2 |\n|---|---|---|")
        L.append(f"| What it is | {ptc0['label'] if ptc0 else '—'} — premature stop | {mis0['label'] if mis0 else '—'} — missense |")
        L.append(f"| What it does to the protein | {'little or no protein from this copy (mRNA predicted to decay by NMD)' if ptc0 and ptc0.get('nmd_escape') is False else 'truncated product'} "
                 f"| protein made, but less stable or less active |")
        L.append("| The lever | restore protein from this copy: read-through, and relief of NMD "
                 "| raise the abundance of what this copy makes — if the change acts by destabilising the protein rather than by "
                 "disrupting an interaction surface, which the cell experiment decides |")
        L.append("| Approved agents to test (laboratory first) | "
                 + "; ".join(h["agents"].split("(")[0].strip() for h in top if "stop allele" in h.get("lever", "")) + " | "
                 + "; ".join(h["agents"].split("(")[0].strip() for h in top if "missense" in h.get("lever", "")) + " |\n")
        if pair0:
            L.append(f"**One experiment answers both, and the diagnosis too.** The two alleles sit {pair0['genomic_distance_bp']:,} bp "
                     f"apart in the genome but only {pair0['cdna_distance_nt']:,} nt apart in the mRNA, so a single "
                     f"{pair0['amplicon_nt']:,}-nt RT-PCR amplicon from cultured cells (primers in section 2) reads both alleles on the "
                     "same molecule — read on a long-read platform, or cloned so that individual molecules are Sanger-sequenced, because the two "
                     "sites are too far apart for a single Sanger read to resolve them. It settles whether the alleles are in trans, "
                     "measures how much of the stop allele's mRNA survives NMD, and "
                     "with allele-resolved peptides measures the protein each allele makes under each drug. The same result supplies the "
                     "evidence that would move the second allele out of 'uncertain significance' (section 7). Because the two levers "
                     "act on different copies of the gene, the same cells can also test them together (a read-through agent with "
                     "niacin), where the prediction is that their effects add.\n")
        L.append("**Two things here are usable without waiting for any experiment.** Section 6 lists the medicines and procedures "
                 "that warrant extra care in a child who has both chromosomal instability and a cancer predisposition, each with "
                 "its published reason — that is for the treating team, today. Section 7 shows how the same cell experiment that "
                 "tests the drug hypotheses would also supply the evidence needed to reclassify the second allele, which is what "
                 "turns a probable diagnosis into a firm one and opens carrier testing for the family. Neither depends on any "
                 "medicine working.\n")
        L.append("**What this report does not claim.** That any of these agents should be given to this child, or that read-through "
                 "delivers enough protein to help — that is the open question, and sections 1 and 2 state what would settle it and what would "
                 "falsify each hypothesis.\n")
    L.append("## Summary\n")
    L.append(track2_summary(gene, lv, t2, pm) + "\n")
    L.append("**Method in brief.** (1) *Allele-level triage* — computed for each allele from the reference sequence, the MANE gene "
             "model, ESM-2 and the AlphaFold model; (2) *literature layer* — graded hypotheses whose every citation is resolved live; "
             "(3) *automated network pass* over Open Targets with direction gates; (4) *scalability* — the triage repeated over every "
             "ClinVar P/LP allele in the disorder's gene panel. Grades: A human trial in a relevant population · B in vivo evidence in "
             "the same gene defect (the intervention may be genetic rather than the drug itself) · C human patient cells or trials in another disease with the same molecular lesion · D mechanistic inference · "
             "X direction of effect uncertain.\n")

    # ---- 1. mechanism ---------------------------------------------------------------------------------------------------
    L.append("## 1. Mechanism of the variants\n")
    if gene == "BUB1B":
        L.append("*The genotype this report builds on came from five independent runs of this pipeline on the child's data, on three "
                 "kinds of machine (a dual-GPU machine, a TPU host and a plain four-CPU machine) and with two different "
                 "population-frequency sources, the second used when a download of the first failed. The top-ranked genotype and its "
                 "coordinates were identical in every run. The lower-ranked alternatives within the same gene did change between "
                 "runs (three to five rows), so only the top-ranked pair is relied on here. This is a statement about repeat runs of "
                 "this pipeline, not an independent confirmation of the diagnosis, and it is recorded because the rest of this report "
                 "rests on it.*\n")
    for v in T.get("variants", []):
        kind = ("null allele — " + ("nonsense-mediated decay predicted, no protein from this copy" if not v.get("nmd_escape") else "truncated protein escaping NMD")
                if v["csq"] in LOF_CSQ else f"{v['csq']} — " + ("hypomorphic candidate: protein made but expected to be less stable or less active"
                                                               if v["csq"] == "missense" else "regulatory / splice candidate"))
        L.append(f"* `{v.get('hgvsp') or v['key']}` ({v['classification']}): {kind}.")
    L.append("")
    if gene == "BUB1B":
        L.append(
            "**Loss of function with residual activity.** BUBR1 is the pseudokinase core of the mitotic checkpoint complex that holds "
            f"APC/C-CDC20 until every kinetochore is attached, and it recruits PP2A-B56 to stabilise correct attachments "
            f"({c('22698286')}; {c('23079597')}). Complete loss is cell-lethal ({c('15159543')}), so affected children carry one "
            f"null allele plus one allele with residual function ({c('15475955')}); in a graded knockdown of a human cell line, premature chromatid "
            f"separation rose first and aneuploidy appeared only below about 50 % of normal BUBR1 ({c('18932004')}). **The therapeutic lever is therefore a modest, "
            "proportional increase in residual BUBR1**; full rescue is neither expected nor required.\n")
        L.append(
            "**How much would be enough?** This question decides whether a weak intervention is worth testing at all. A graded knockdown "
            "in a human cell line — not patient cells — gives a dose-response rather than a threshold: premature chromatid separation rose "
            f"first, and aneuploidy appeared only below about half of normal BUBR1 ({c('18932004')}). Mice engineered to express only low amounts of BUBR1 are "
            f"viable and reproduce much of the MVA spectrum ({c('15208629')}), and combining different mutant alleles showed that which "
            f"alleles are present changes severity beyond the total protein level alone ({c('31738183')}). Read-through typically restores "
            "only a low single-digit percentage of a protein, and that is from one allele. **We do not claim that this is enough: "
            "the gap between what read-through delivers and what this genotype needs is the central unresolved question of this "
            "report, and we have found no data that closes it for BUB1B.** What can be done is to measure that gap rather than "
            "argue about it, which is the purpose of the gentamicin arm: as the most potent read-through agent available it sets "
            "the ceiling for how much protein this particular allele can yield, turning an open question into a number before any "
            "clinical question is raised. If that ceiling sits far below the level at which premature chromatid separation "
            "improves, the stop-codon hypotheses should be abandoned and the missense allele's abundance is the remaining lever.\n")
        dis = st.get("disease_missense_uniprot") or []
        L.append(
            f"**Two allele classes, one of each here.** In patient cells, MVA alleles fall into two classes ({c('20516114')}): truncating "
            "alleles whose transcripts are absent (NMD), and missense alleles in or near the kinase domain that lower BUBR1 through faster "
            "protein turnover. The same study showed that a truncated BUBR1 lacking the kinase domain (731X, 715 BUBR1 residues) still "
            "supports the checkpoint and stable microtubule attachments when expressed from a cDNA (no NMD), whereas a slightly "
            "longer truncation (753X) was expressed at lower levels, which the authors note need not mean reduced stability."
            + (f" The child's missense change lies in the {st.get('domain')} (AlphaFold pLDDT {st.get('plddt')}), "
               f"{dis[0]['distance_A']} Å from the reported {dis[0]['disease']} allele {dis[0]['change']}"
               + (f"; it is less buried than {100 - st['burial_percentile_in_domain']} % of the domain's residues, so a surface "
                  "or interface effect is as plausible as unfolding" if st.get("burial_percentile_in_domain") is not None and st['burial_percentile_in_domain'] < 50 else "")
               + ". Whether it acts like the published kinase-domain alleles (low abundance through faster turnover) or through a "
               "functional defect is the first question the cell experiment answers (cycloheximide chase, section 2)."
               if dis else "") + "\n")
        L.append("| Consequence of BUBR1 insufficiency | Child's feature it plausibly explains | Evidence |\n|---|---|---|")
        L.append(f"| chromosome missegregation → variegated aneuploidy | the diagnosis | {c('15475955')} |")
        L.append(f"| cancer predisposition (Wilms tumour, rhabdomyosarcoma, haematological) | rhabdomyosarcoma | {c('11746029')}; {c('39264246')} |")
        L.append(f"| p16-driven senescence, progeroid 'cachectic dwarfism' in hypomorphic mice | small for gestational age, short stature, failure to thrive, muscle atrophy | {c('15208629')}; {c('18516091')} |")
        L.append(f"| carrier parents show intermediate premature chromatid separation (method-dependent) | family counselling: whether carrier parents have any reproductive phenotype is **unproven** — one unreplicated 2026 report of BUB1B variants in recurrent pregnancy loss, and a negative screen of aneuploid miscarriages | {c('11746029')}; {c('15108198')}; {c('42434306')}; {c('40028477')} |")
        L.append("| — (not explained by the checkpoint model) | nephrocalcinosis: prematurity and renal tubulopathies are more parsimonious | see the nephrocalcinosis hypothesis in section 3 |\n")

    # ---- 2. allele-level triage ------------------------------------------------------------------------------------------
    if lv.get("alleles"):
        L.append("## 2. Allele-level therapeutic triage (computed)\n")
        L.append("Drug-target databases ask which drug acts on a protein near the gene. For a loss-of-function disorder the more useful "
                 "question is what can still be done with each allele; this section answers it from sequence, gene model, protein "
                 "language model and predicted structure. The code is gene-agnostic.\n")
        if ptc:
            ex = ptc.get("exon") or {}
            nc = ptc.get("near_cognate") or {}
            rank_txt = {1: "first", 2: "second", 3: "third"}.get(ptc.get("readthrough_rank"), "?")
            rows = [("stop codon / next nucleotide", f"{ptc['stop_rna']}, followed by {ptc.get('plus4_rna') or ptc['plus4']}; "
                     f"{ptc['stop_rna']} ranks {rank_txt} of the three stop codons for read-through (UGA > UAG > UAA)",
                     f"UGA is the most read-through-prone stop in reporter assays ({c('10939566')}); a +4 C enhances read-through, "
                     f"while stop identity alone did not change the gentamicin response in another study ({c('22479203')}), and in "
                     f"Duchenne patients neither predicted response ({c('20517938')}): measure, do not infer"),
                    ("nonsense-mediated decay", "predicted" if ptc.get("nmd_escape") is False else "escapes" if ptc.get("nmd_escape") else "unknown",
                     f"exon {ex.get('exon')} of {ex.get('n_exons')}; NMD removes the template for read-through, hence the value of a dual NMD-inhibitor / read-through drug")]
            restore = [k for k, v in nc.items() if v.get("restores_wild_type")]
            nc = {k: v for k, v in nc.items() if not v.get("restores_wild_type")}
            if nc:
                comp = (f". Every substitution the triage scores at residue {ptc['residue']} lands in a narrow band, which says "
                        "the position is weakly constrained rather than that any particular insertion would work; these scores "
                        "are comparable with each other at this residue, but not with scores at a different residue, so the "
                        f"child's own {mis['short']} (LLR {mis['llr']:.1f}) is given for scale only and not as a benchmark"
                        ) if mis and mis.get("llr") is not None else ""
                rows.append(("what read-through would insert", ", ".join(f"{ptc['wt_aa']}{ptc['residue']}{k}: LLR {v['llr']:.1f}"
                                                                       + (f" (better tolerated than {v['pct_less_tolerated']:.0f} % of all possible substitutions)" if v.get("pct_less_tolerated") is not None else "") for k, v in nc.items()),
                             f"near-cognate tRNAs insert {', '.join(AA3.get(a, a) for a in ptc.get('near_cognate_aas', ''))} at {ptc['stop_rna']} "
                             f"({c('27702906')}){comp}"
                             + (f"; {AA3.get(restore[0], restore[0])} insertion would restore the wild-type residue" if restore else "")))
            rows.append(("structure at the stop residue", f"AlphaFold pLDDT {fmt(ptc.get('plddt'))}", ""))
            rows.append(("truncated product if NMD is relieved", f"residues {ptc['truncated_product']} (~{fmt(ptc.get('truncated_kda'))} kDa); keeps "
                         + ("; ".join(ptc.get("truncated_keeps") or []) or "—") + "; loses " + ("; ".join(ptc.get("truncated_loses") or []) or "—"),
                         (fragment_note(ptc, pm) if gene == "BUB1B" else "")))
            if ex:
                if not ex.get("in_frame"):
                    verdict = "not eligible: skipping would shift the frame"
                elif not ex.get("skippable"):
                    verdict = "not eligible: first or last coding exon, or only partly coding"
                elif ex.get("mean_plddt") is None:
                    verdict = "candidate in principle; structure of the exon's residues not assessed"
                elif ex["mean_plddt"] >= 70:
                    verdict = "lower priority: the exon's residues are structured"
                else:
                    verdict = "candidate: the exon's residues are largely disordered"
                iso = ex.get("natural_skipping_isoforms")
                rows.append(("antisense exon skipping", f"exon {ex['exon']}: {ex['length_nt']} nt ({'in frame' if ex['in_frame'] else 'frame-shifting'}), "
                             f"spanning residues {ex['aa_from']}-{ex['aa_to']} (the codons at each boundary are split across "
                             f"neighbouring exons), mean pLDDT {fmt(ex.get('mean_plddt'))}; natural isoform skipping it: "
                             + ("none annotated" if iso == [] else ", ".join(iso) if iso else "not checked"), verdict))
            L.append(f"**Nonsense allele {ptc['label']}**\n")
            L.append("| Property | Value | What follows |\n|---|---|---|\n" + "\n".join(f"| {a} | {b} | {x} |" for a, b, x in rows) + "\n")
        if mis:
            near = mis.get("nearest_patient_missense") or []
            tail = (f". No approved drug binds this domain: BUBR1 is a pseudokinase ({c('22698286')}) whose kinase domain bound no "
                    f"nucleotide in a pseudokinase thermal-shift survey ({c('24107129')}), so a small-molecule chaperone is not available; "
                    "the lever is protein abundance (the missense-allele lead, hypothesis 3).\n"
                    if gene == "BUB1B" else ".\n")
            L.append(f"**Missense allele {mis['label']}** — {mis.get('domain') or 'no annotated domain'}; AlphaFold pLDDT {fmt(mis.get('plddt'))}; "
                     f"ESM-2 LLR {fmt(mis.get('llr'))}"
                     + (f"; less buried than {100 - mis['burial_percentile']} % of the domain's residues" if mis.get("burial_percentile") is not None else "")
                     + ("; nearest literature-reported patient missense: " + ", ".join(f"{x['change']} ({x['disease']}, {x['distance_A']} Å)" for x in near) if near else "")
                     + tail)
        if S.get("figures", {}).get("allele_levers"):
            L.append(f"![allele-level levers]({S['figures']['allele_levers']})\n")
        L.append("### One experiment in the child's own cells tests every lever\n")
        pr = lv.get("pair")
        if pr:
            L.append(f"* **Phase and NMD from one RT-PCR.** The alleles are {pr['genomic_distance_bp']:,} bp apart in the genome — too far "
                     f"for short reads — but {pr['cdna_distance_nt']:,} nt apart in the mRNA ({pr['fold_shorter']}× closer). One amplicon of "
                     f"~{pr['amplicon_nt']:,} nt from cultured cells, with and without an NMD inhibitor as is standard in RNA diagnostics "
                     f"({c('40473777')}), shows whether the alleles are in trans and how much nonsense-allele mRNA "
                     f"survives NMD. The two sites are {pr['cdna_distance_nt']:,} nt apart within the amplicon, which is beyond one Sanger read, so the "
                     "single-molecule readout is long-read sequencing of the amplicon or Sanger sequencing of individual clones; "
                     f"RNA-based phasing of distant variants is established in principle ({c('27605262')}); parental testing "
                     f"remains the clinical standard ({c('25741868')}).")
            if pr.get("primers"):
                p0 = pr["primers"][0]
                L.append(f"  Primer3 pair, ready to order: forward `{p0['left']}` (exon {p0['exons_left']}, Tm {p0['tm_left']} °C), "
                         f"reverse `{p0['right']}` (exon {p0['exons_right']}, Tm {p0['tm_right']} °C); {p0['product_nt']:,}-nt product with "
                         f"the two sites at positions {p0['site_offsets'][0]} and {p0['site_offsets'][1]}. "
                         + ("One primer spans an exon–exon junction, so genomic DNA does not amplify. " if p0["junction_spanning"]
                            else f"Genomic DNA would give a {p0['genomic_span_bp']:,}-bp product, beyond a standard extension time. ")
                         + ("No variant of the child " if p0["child_variants_under_primers"] == 0 else "")
                         + ("and no common SNP (MAF ≥ 1 %) " if p0["common_snps_under_primers"] == 0 and p0["child_variants_under_primers"] == 0
                            else "No common SNP (MAF ≥ 1 %) " if p0["common_snps_under_primers"] == 0 else "")
                         + ("lies under either primer, so neither allele should drop out and bias the allelic ratio. "
                            if 0 in (p0["child_variants_under_primers"], p0["common_snps_under_primers"]) else "")
                         + "Reads carrying the stop allele with the reference missense base (and vice versa) "
                         f"prove trans; {len(pr['primers']) - 1} alternative pair(s) are in the notebook output.")
        peps = lv.get("peptides") or []
        if peps:
            lg = lv.get("peptide_logic") or {}
            L.append("* **Allele-resolved peptides.** If the alleles are in trans, a read-through product of the nonsense allele carries a "
                     "near-cognate amino acid at the stop residue"
                     + (" and the reference residue at the missense site, which no other product can carry because the truncated "
                        "product ends before it" if lg.get("missense_downstream_of_stop") else "")
                     + ". Each peptide below is digested from the protein that allele actually makes and differs in mass from the others, "
                     "so targeted mass spectrometry measures read-through directly, separately from the missense allele's protein:\n")
            L.append("| Protease | Form | Peptide | Monoisotopic mass (Da) | Made by |\n|---|---|---|---|---|\n"
                     + "\n".join(f"| {p['protease']} | {p['form']} | `{p['peptide']}` | {p['mass']:.3f} | {p['made_by']} |" for p in peps) + "\n")
        L.append("**Pre-registered predictions**\n")
        L.append(predictions_table(lv, pm, gene) + "\n")
        if gene == "BUB1B":
            L.append("**What would falsify each hypothesis.** Stated before the experiment so the result cannot be reinterpreted "
                     "afterwards:\n\n"
                     "* *No read-through peptide and no full-length protein in the gentamicin arm* → this allele is not "
                     "read-through-competent, and the stop-codon hypotheses (1, 2 and 4) are dead, whatever the mRNA does.\n"
                     "* *No read-through above the assay's limit of quantitation* — which must be fixed first, from a dilution series of "
                     "synthetic peptide spiked into untreated lysate, so that 'none detected' means a number rather than an "
                     "impression. *Read-through detectable with gentamicin but the ceiling it reaches is far below the level at which "
                     "premature chromatid separation improves* → the approach cannot work for this genotype at achievable "
                     "exposures; report the ceiling and stop.\n"
                     "* *Nonsense-allele mRNA does not rise when NMD is blocked* → the NMD premise is wrong and hypothesis 2 "
                     "loses its rationale.\n"
                     "* *The truncated fragment is stable but segregation worsens* → the dominant-negative risk above is real, "
                     "and relieving NMD is harmful rather than helpful.\n"
                     "* *No increase in BUBR1 half-life or abundance with niacin* → the NAD+/SIRT2 hypothesis (3) is dead for "
                     "this missense allele.\n"
                     "* *The amplicon shows the alleles in cis* → the compound-heterozygous model is wrong, and every "
                     "allele-level hypothesis here is void.\n")
            L.append("**Feasibility.** The whole panel runs on cultured skin fibroblasts or a lymphoblastoid line from one biopsy "
                     "or blood draw that a diagnostic laboratory can already obtain; the readouts (RT-PCR read on a long-read platform or "
                     "cloned and Sanger-sequenced, two immunoblots, targeted mass spectrometry, and standard cytogenetic scoring of premature "
                     "chromatid separation and micronuclei) are routine, need no new reagents beyond the primers printed above, "
                     "and would report in weeks rather than years. Nothing here requires the child to take anything.\n")

    # ---- 3. hypotheses ---------------------------------------------------------------------------------------------------
    if t2.get("curated"):
        L.append("## 3. Ranked hypotheses\n")
        # index first (one line each, readable at a glance), then one block per hypothesis: an 8-column
        # table of paragraphs is unreadable once it is printed to PDF
        L.append("| # | Lever | Hypothesis | Grade |\n|---|---|---|---|")
        for h in t2["curated"]:
            L.append(f"| {h['rank']} | {h['lever']} | {h['hypothesis']} | {h['grade']} |")
        L.append("")
        for h in t2["curated"]:
            used.update(h["pmids"])
            L.append(f"### {h['rank']}. {h['hypothesis']} — grade {h['grade']}\n")
            L.append(f"* **Lever.** {h['lever']}")
            L.append(f"* **Agent(s).** {h['agents']}")
            cap = lambda t: t[:1].upper() + t[1:]           # the source strings are written as clauses
            L.append(f"* **Rationale.** {cap(h['mechanism'])}")
            L.append(f"* **Main caveat.** {cap(h['caveat'])}")
            L.append(f"* **How it would be tested.** {cap(h['validation'])}")
            L.append("* **References.** " + ", ".join(c(p) for p in h["pmids"]) + "\n")
        L.append("## 4. Considered and set aside\n")
        L.append("| Option | Why it is not proposed |\n|---|---|")
        for x in t2.get("set_aside", []):
            L.append(f"| {x['item']} | {x['reason']}" + (f" (PMID {', '.join(x['pmids'])})" if x["pmids"] else "") + " |")
            used.update(x["pmids"])
        L.append("")

    # ---- 5. network pass -------------------------------------------------------------------------------------------------
    L.append("## 5. What the automated network pass found\n")
    if net.get("n_network_nodes") is None:
        L.append("The network pass did not complete in this run (see the run log).\n")
    else:
        L.append(f"{net.get('n_network_nodes')} nodes (the gene, its Open Targets interaction partners with score ≥ 0.7, and literature "
                 f"mechanism nodes); every drug / clinical candidate on them (Open Targets {net.get('open_targets_api')}), gated by approval → "
                 f"direction (drug action × cited node sign) → antimitotic / genotoxic mechanism. {gene} has {net.get('n_direct_drugs')} drugs "
                 f"acting on it directly (essential gene: {net.get('essential')}); {net.get('n_approved')} approved drug–target pairs on the "
                 "network; by verdict: " + ", ".join(f"{k}: {v}" for k, v in (net.get("verdicts") or {}).items()) + ". "
                 + ("**No approved drug acts on the network in a direction predicted to raise residual function**"
                    + (" — the classes that would (SIRT2 activation, partial APC/C-CDC20 inhibition) have no approved member" if gene == "BUB1B" else "")
                    + ". This honest negative is why the allele-level and literature layers exist.\n"
                    if not (net.get("verdicts") or {}).get("plausibly beneficial direction") else
                    "Drugs in the 'plausibly beneficial direction' class are listed in `track2_network_drugs.csv`.\n"))
    gen = t2.get("generality") or []
    if gen:
        L.append("The identical pass for the other genes of the indication panel:\n\n" + df_md(pd.DataFrame(gen).fillna(0)) + "\n")

    # ---- 6. cautions -----------------------------------------------------------------------------------------------------
    if t2.get("caution"):
        L.append("## 6. Caution list for the clinical team\n")
        L.append("| Medicine / class | Reason |\n|---|---|")
        for x in t2["caution"]:
            L.append(f"| {x['item']} | {x['reason']} (PMID {', '.join(x['pmids'])}) |")
            used.update(x["pmids"])
        L.append("")

    # ---- 7. impact -------------------------------------------------------------------------------------------------------
    L.append("## 7. Impact\n")
    rc = lv.get("reclassification")
    if rc:
        L.append(f"**For the diagnosis.** The weaker allele ({rc['allele']}) is classed "
                 f"{(rc.get('current_class') or 'as shown below').lower()} by automated criteria. The evidence that would "
                 f"move it, on the point scale of {c('32720330')}:\n")
        L.append("| Step | Points | Posterior | Class |\n|---|---|---|---|\n"
                 + "\n".join(f"| {r['step']} | {r['points']:g} | {r['posterior']:.3f} | {r['classification']} |" for r in rc["path"]) + "\n")
        L.append("Parental testing (or the cDNA amplicon above) supplies PM3, and the same cell experiment that tests the drugs supplies "
                 "functional evidence: one experiment moves the diagnosis and the therapy question forward together. A firm diagnosis "
                 "opens carrier testing for the parents and, if both carry one allele, a 25 % recurrence risk per pregnancy and the "
                 "option of prenatal or preimplantation testing.\n")
    if gene == "BUB1B":
        L.append(f"**Published MVA surveillance guidance, for the treating team's reference.** Consensus surveillance ({c('39264246')}): renal ultrasound every "
                 "3 months until age 7; clinical review for signs of rhabdomyosarcoma, Wilms tumour and haematological malignancy; avoid "
                 "ionising radiation; HPV vaccination. The nephrocalcinosis work-up (section 3) may uncover a treatable cause of part of "
                 "the failure to thrive.\n")
    at = lv.get("atlas")
    if at:
        bl = at.get("by_lever") or {}
        L.append(f"**For other children with the disorder.** The triage was repeated over all {at['n_alleles']} ClinVar P/LP alleles in "
                 f"{at['n_genes']} of the panel's genes that have any catalogued alleles: " + ", ".join(f"{k}: {v}" for k, v in bl.items()) + ". Nonsense alleles by stop "
                 "codon: " + (", ".join(f"{k.replace('T', 'U')} {v}" for k, v in (at.get("nonsense_by_stop") or {}).items()) or "none")
                 + f" (in {gene}: " + (", ".join(f"{k.replace('T', 'U')} {v}" for k, v in (at.get("top_gene_nonsense_by_stop") or {}).items()) or "none")
                 + "). A positive cell result for the nonsense lever would therefore matter beyond this child.\n")
        nb = at.get("nonsense_by_stop") or {}
        n_uga = nb.get("UGA") or nb.get("TGA") or 0
        n_rt = sum(v for k, v in (at.get("by_lever") or {}).items() if "read-through" in k)
        if n_uga:
            L.append(f"**The number that matters for reach.** Of the {at['n_alleles']} catalogued disease-causing alleles in "
                     f"this panel, {n_rt} are premature stops and {n_uga} of those are the same stop codon as this child's. "
                     "Those are alleles, not people, and one person carries two; but every one of them is an allele for which "
                     "this exact assay — the same primer logic, the same peptide design, the same readouts — would be built "
                     "the same way. That is the sense in which a result here is not an n-of-1."
                     + (f" {at['n_no_criteria']} of the {at['n_alleles']} have no assertion criteria in ClinVar; the review status is "
                        "in the atlas file, so a stricter reading can simply filter them out." if at.get("n_no_criteria") else "")
                     + "\n")
        L.append(df_md(pd.DataFrame(at.get("by_gene") or [])) + "\n")
    if gene == "BUB1B":
        L.append("**What would have to happen next, and who could do it.** Nothing in this report changes anything today; it is a set of "
                 "laboratory questions, and this is the shortest path from here to an answer:\n\n"
                 "1. *Confirm the genetics.* The treating clinical genetics service confirms both alleles and tests the parents, which "
                 "settles phase by the standard route and is worth doing whatever happens with the rest.\n"
                 "2. *Grow the cells.* Skin fibroblasts or a lymphoblastoid line from the child, with a parental line alongside as the "
                 "comparison. The challenge page says the MVA Society is funding focused research; whether banked patient cells already exist is for the family and the Society to say, and nothing here assumes it.\n"
                 "3. *Run the one amplicon.* Phase, NMD escape and the drug arms are all read out from the same cultured cells using the "
                 "primers printed in section 2 — the cheapest experiment here and the one that can falsify the most.\n"
                 "4. *Report the answer either way.* Because mosaic variegated aneuploidy affects so few people, a negative result that "
                 "retires a plausible idea is itself worth publishing; the atlas above names the other alleles to which the same answer "
                 "would apply.\n\n"
                 "Each step uses assays a diagnostic or research laboratory already runs, which is the point: this is a protocol, not a "
                 "programme.\n")
        L.append("**In plain language.** The child has two changes in the same gene, one inherited from each parent (which parental "
                 "testing would confirm). One change stops the gene's instructions early, so that copy makes little or no protein. The "
                 "other changes a single building block, so that copy probably makes protein that does not last as long or work as well. "
                 "The protein's job is to check that chromosomes are shared out correctly when a cell divides, which is why cells in this "
                 "condition end up with the wrong number of chromosomes. **No medicine is being recommended here, and nothing in this "
                 "report should change anyone's current treatment.** What we have done is ask whether any already-approved medicine acts "
                 "on either of those two specific problems, and then describe a laboratory test, done on cells grown from a small skin or "
                 "blood sample, that would show whether any of them does anything useful — before anyone considers whether it could help a "
                 "person. The same test would also help the laboratory decide, formally, whether the second change is disease-causing, "
                 "which matters for the family's own genetic counselling. Questions a family might reasonably raise with their own care "
                 "team are whether both parents have been tested, whether the recommended surveillance schedule is in place, and whether "
                 "stored cells could be made available to researchers.\n")

    # ---- 7b. does the triage get known answers right? --------------------------------------------------------------------
    val = S.get("lever_validation") or {}
    if val.get("n_tested"):
        L.append("## 8. Does the triage get published answers right?\n")
        L.append(f"A classifier is only worth anything if it reproduces answers that are already known. The same triage was run, "
                 f"unchanged, over {val['n_tested']} premature-stop alleles in {len(val.get('genes') or [])} genes outside this "
                 "disorder's panel — alleles chosen because the published literature already reports how they behave when "
                 f"read-through is attempted. It recovered the published stop codon for {val['n_codon_match']} of "
                 f"{val['n_tested']} ({100 * val.get('frac_codon_match', 0):.0f} %) directly from the reference sequence and the "
                 "gene model, and assigned the matching lever class in "
                 f"{val.get('n_lever_match', val['n_codon_match'])} of {val['n_tested']}.\n")
        if val.get("confusion"):
            L.append("| Lever assigned | UGA | UAG | UAA |\n|---|---|---|---|\n"
                     + "\n".join(f"| {r['lever']} | {r.get('UGA', 0)} | {r.get('UAG', 0)} | {r.get('UAA', 0)} |"
                                 for r in val["confusion"]) + "\n")
        L.append("Recovering a stop codon from the reference sequence and the gene model is close to a sequence lookup, so a full "
                 "score is the expected result rather than a finding about biology. What the check does establish is that the "
                 "pipeline reads the reference, the gene model and the numbering correctly across genes and strands, and that its "
                 "lever classes separate by stop codon as intended.\n")
        L.append("Two honest limits on what this shows. Every allele in the set is one where read-through has been reported to "
                 "work to some degree, so the check measures whether the triage finds the alleles it should, not whether it "
                 "rejects the ones it should — it tests sensitivity, not specificity. And one of the alleles contradicts the "
                 "ordering the triage uses as a prior: a UAA allele responded better than UGA alleles tested alongside it. That "
                 "is why the stop codon is treated here as a prior to be measured rather than a rule, and it is the clearest "
                 "argument for running the laboratory arm before believing any ranking, including this report's own.\n")
        L.append("The per-allele table is saved as `lever_validation.csv`, with the published source for every entry.\n")

    # ---- 8. scalability --------------------------------------------------------------------------------------------------
    L.append("## 9. Scalability\n")
    rt = RUN.get("Track 2: allele-level therapeutic triage", {})
    L.append("* The allele-level triage, the network pass and the report are produced by the notebook with no manual step, for whichever "
             "gene leads, from public resources only (MANE, GRCh38, ClinVar, ESM-2, AlphaFold DB, UniProt, Ensembl, Open Targets, Europe PMC)"
             + (f"; the triage took {rt['seconds'] / 60:.1f} min." if rt.get("seconds") else ".")
             + "\n* In demo mode the same code runs on a public genome with a different spiked-in recessive gene, which is how it is tested."
             "\n* The atlas above is the same computation applied to a whole gene panel; it extends to any gene list. It is saved as "
             "`lever_atlas.csv` (one row per public ClinVar pathogenic / likely-pathogenic allele, with the lever, the reason for it, the "
             "stop codon, the exon and the ClinVar review status), so another group can point the same triage at their own gene list and "
             "get a comparable table without rerunning the patient-specific parts.\n")
    L.append("**Reproducibility.** Code: " + CFG.get("REPO_URL", "see the repository link submitted with this report")
             + " (CC BY 4.0, as required for submissions to this hackathon). `python build_nb.py` assembles the percent-format "
             "sources in `src/` into the notebook; running that notebook end to end reproduces every number, table and figure in "
             "this report, including this sentence, because the report is written by the pipeline rather than by hand. Without the "
             "gated dataset the same notebook runs in demo mode on a public genome with a different recessive gene spiked in, which "
             "is how each stage is tested; the run that produced this report took "
             + (f"{sum(v.get('seconds', 0) for v in RUN.values() if isinstance(v, dict)) / 60:.0f} minutes " if RUN else "")
             + "on a 4-CPU machine with no GPU.\n")

    # ---- 9. limitations --------------------------------------------------------------------------------------------------
    L.append("## 10. Limitations\n")
    L.append("* No candidate has been tested in MVA; grades reflect indirect evidence. Amlexanox's cellular concentrations may not be "
             "reachable in plasma, its TBK1/IKKε inhibition could itself affect mitosis, and its oral form has been marketed in Japan, with its current status unverified.\n"
             "* Read-through efficiency in patients is not predicted by the stop-codon context; ESM-2 scores of the inserted amino acids are "
             "predictions, not measurements, and read-through products can be immunogenic.\n"
             "* Phase is inferred, not observed (see the Track 1 report); the allele-resolved readouts assume the alleles are in trans, which "
             "the cDNA amplicon tests first.\n"
             "* Network drug lists depend on database coverage and on a small hand-curated sign map (printed in the notebook).\n"
             f"* Generative AI: {CFG['LLM_DISCLOSURE']}\n")
    L.append("## References (resolved live from Europe PMC)\n")
    for p in sorted(used | {"15475955", "16411201"}, key=lambda x: ((pm.get(x) or {}).get("year", "9999"), x)):
        if pm.get(p):
            L.append(f"* {cite(p, pm)} — {clean_title(pm[p]['title'])}")
    L.append("\n*" + ACK + "*")
    return "\n".join(L)


def write_reports():
    if "report_pmids" not in S:
        try:
            S["report_pmids"] = {k: v for k, v in resolve_pmids(REPORT_PMIDS).items() if v}
        except Exception:       # noqa: BLE001
            S["report_pmids"] = {}
    paths = []
    for name, fn in ((f"{CFG['TEAM']}_track1_report.md", track1_report), (f"{CFG['TEAM']}_track2_report.md", track2_report)):
        try:
            (OUT / name).write_text(fn(), encoding="utf-8")
            paths.append(name)
        except Exception as e:      # noqa: BLE001
            log(f"report {name} failed: {type(e).__name__}: {str(e)[:200]}")
    summary = {k: v for k, v in S.items() if k not in ("report_pmids",)}
    summary["run"] = RUN
    (OUT / "run_summary.json").write_text(json.dumps(summary, indent=1, default=str), encoding="utf-8")
    return paths


with stage("write reports"):
    log(f"reports written: {write_reports()}")
    display(Markdown((OUT / f"{CFG['TEAM']}_track1_report.md").read_text(encoding="utf-8")[:6000]))

## 16 · Read-level evidence from the raw FASTQ (optional; the long stage)

Aligning 85 GB of reads genome-wide is not possible on this machine, and is not needed. Instead:

1. **Bait** — the leading gene ± 50 kb, the next genes ± 5 kb, and ~500 random 2-kb control windows (autosomes and, as a
   built-in single-copy control in a male, chromosome X).
2. **Unique-minimizer recruitment** — a `deacon` minimizer index of the bait, minus every minimizer that also occurs
   elsewhere in GRCh38, so reads from repeats (Alu, L1) are not pulled in. Each lane is downloaded, streamed through the
   recruiter (a pair is kept if either mate hits) and deleted.
3. **Re-alignment of the recruited reads to the whole genome** (`minimap2 -ax sr`, duplicates marked), so reads that
   belong elsewhere receive low mapping quality instead of piling up on the bait.
4. **Analyses** — independent allele counts (per strand, per lane) at the leading alleles; re-calling with a second
   caller and concordance with the provided VCF; read-pair links between consecutive heterozygous sites between the two
   alleles (empirical test of phaseability); depth along the gene relative to autosomal controls (chrX as positive
   control for a one-copy change); clipped-read and discordant-pair clusters (structural-variant screen).

The module stops starting new lanes after `CFG["READ_LEVEL_MAX_HOURS"]` of total run time; one lane (~10×) is already
enough for allele-level validation. In demo mode it runs on the first 2 M read pairs of a public HG002 run, as a
toolchain test only.

In [ ]:
READS = WORK / "reads"
HG002_FQ = "https://storage.googleapis.com/brain-genomics-public/research/sequencing/fastq/novaseq/wgs_pcr_free/30x/HG002.novaseq.pcr-free.30x.R{r}.fastq.gz"


def ngs_env():
    """deacon (+ samtools / minimap2 already on PATH) in its own micromamba environment."""
    env = WORK / "env_reads"
    if not (env / "bin" / "deacon").exists():
        if not MICROMAMBA.exists():
            fetch("https://github.com/mamba-org/micromamba-releases/releases/latest/download/micromamba-linux-64", MICROMAMBA, min_bytes=1_000_000)
            MICROMAMBA.chmod(0o755)
        sh(f"'{MICROMAMBA}' create -y -q -p '{env}' -c conda-forge -c bioconda deacon", name="mamba_deacon", timeout=1800)
    return env / "bin" / "deacon"


def make_bait(genes, fa, n_auto=400, n_x=100, seed=3):
    regs = []
    for i, g in enumerate(genes):
        if g in GENE_POS:
            c, s, e, _, _ = GENE_POS[g]
            f = 50_000 if i == 0 else 5_000
            regs.append((c, max(1, s - f), min(CHROM_LEN[c], e + f), f"target_{g}"))
    rng = np.random.default_rng(seed)
    autos = [c for c in PRIMARY[:22]]
    w = np.array([CHROM_LEN[c] for c in autos], dtype=float)
    ctrl = []
    while len(ctrl) < n_auto + n_x:
        if len(ctrl) < n_auto:
            c = autos[int(rng.choice(len(autos), p=w / w.sum()))]
            s = int(rng.integers(5_000_000, CHROM_LEN[c] - 5_000_000))
            tag = "ctrl_auto"
        else:
            c, s, tag = "chrX", int(rng.integers(3_000_000, 150_000_000)), "ctrl_X"
        if any(c == r[0] and s < r[2] and s + 2000 > r[1] for r in regs):
            continue
        seq = str(fa[c][s - 1:s + 1999])
        if seq.count("N") > 20 or len(seq) < 2000:
            continue
        ctrl.append((c, s, s + 1999, tag))
    return regs + ctrl


def write_masked_genome_script(bait, path):
    """Stream GRCh38 with the bait intervals replaced by N (input for subtracting non-unique minimizers)."""
    by = collections.defaultdict(list)
    for c, s, e, _ in bait:
        by[c].append((s, e))
    path.write_text(textwrap.dedent(f'''
        import sys
        from pyfaidx import Fasta
        bait = {dict(by)!r}
        fa = Fasta({RES["ref"]!r}, as_raw=True, sequence_always_upper=True)
        out = sys.stdout
        for name in fa.keys():
            seq = fa[name][:]
            for s, e in sorted(bait.get(name, [])):
                seq = seq[:s - 1] + "N" * (e - s + 1) + seq[e:]
            out.write(">" + name + "\\n")
            for i in range(0, len(seq), 1000000):
                out.write(seq[i:i + 1000000] + "\\n")
    '''))


def fastq_sizes():
    from huggingface_hub import HfApi
    files = [CFG["FASTQ_PATTERN"].format(lane=l, read=r) for l in CFG["FASTQ_LANES"] for r in (1, 2)]
    info = HfApi(token=HF_TOKEN).get_paths_info(CFG["HF_DATASET"], files, repo_type="dataset")
    return {f.path: f.size for f in info}


def recruit_lane(deacon, idx, lane, sizes, fq_dir, out_dir):
    """Download both mates of one lane (or stream them when scratch space is short), recruit, delete."""
    fns = [CFG["FASTQ_PATTERN"].format(lane=lane, read=r) for r in (1, 2)]
    o1, o2, js = out_dir / f"L{lane}.R1.fq.gz", out_dir / f"L{lane}.R2.fq.gz", out_dir / f"L{lane}.json"
    need = sum(sizes.get(f, 11e9) for f in fns) / 1e9
    stream = free_gb(WORK) < need + 8
    t0 = time.time()
    if not stream:
        from huggingface_hub import hf_hub_download
        with ThreadPoolExecutor(2) as ex:
            paths = list(ex.map(lambda f: hf_hub_download(CFG["HF_DATASET"], f, repo_type="dataset", local_dir=str(fq_dir), token=HF_TOKEN), fns))
        for p, f in zip(paths, fns):
            if f in sizes and Path(p).stat().st_size != sizes[f]:
                raise RuntimeError(f"size mismatch for {f}")
        t_dl = time.time() - t0
        sh(f"'{deacon}' filter --threads {NCPU} -a 2 -s '{js}' '{idx}' '{paths[0]}' '{paths[1]}' -o '{o1}' -O '{o2}'", name=f"deacon_L{lane}", timeout=4 * 3600)
        for p in paths:
            Path(p).unlink(missing_ok=True)
        shutil.rmtree(fq_dir / ".cache", ignore_errors=True)
    else:
        base = f"https://huggingface.co/datasets/{CFG['HF_DATASET']}/resolve/main/"
        p1, p2 = fq_dir / f"L{lane}_R1.fq.gz", fq_dir / f"L{lane}_R2.fq.gz"
        sh(f"""rm -f '{p1}' '{p2}'; mkfifo '{p1}' '{p2}'
               curl -sSfL --retry 3 -H "Authorization: Bearer $HF_TOKEN" '{base}{fns[0]}' > '{p1}' & c1=$!
               curl -sSfL --retry 3 -H "Authorization: Bearer $HF_TOKEN" '{base}{fns[1]}' > '{p2}' & c2=$!
               '{deacon}' filter --threads {NCPU} -a 2 -s '{js}' '{idx}' '{p1}' '{p2}' -o '{o1}' -O '{o2}'
               wait $c1; wait $c2; rm -f '{p1}' '{p2}'""", name=f"deacon_stream_L{lane}", timeout=4 * 3600)
        t_dl = float("nan")
    summ = json.loads(js.read_text()) if js.exists() else {}
    return dict(lane=lane, mode="stream" if stream else "download", minutes=round((time.time() - t0) / 60, 1),
                download_minutes=round(t_dl / 60, 1) if t_dl == t_dl else None,
                **{k: v for k, v in summ.items() if isinstance(v, (int, float)) and k.startswith(("seqs", "bp", "seconds", "time"))})


def demo_reads(deacon, idx, out_dir, n_pairs=2_000_000):
    fq = {}
    for r in (1, 2):
        fq[r] = READS / f"demo.R{r}.fq"
        sh(f"set +o pipefail; curl -sSfL '{HG002_FQ.format(r=r)}' | gzip -dc 2>/dev/null | head -n {4 * n_pairs} > '{fq[r]}'; true",
           name=f"demo_fq_R{r}", timeout=1800)
    js = out_dir / "L1.json"
    sh(f"'{deacon}' filter --threads {NCPU} -a 2 -s '{js}' '{idx}' '{fq[1]}' '{fq[2]}' -o '{out_dir / 'L1.R1.fq.gz'}' -O '{out_dir / 'L1.R2.fq.gz'}'",
       name="deacon_demo")
    for p in fq.values():
        p.unlink(missing_ok=True)
    summ = json.loads(js.read_text()) if js.exists() else {}
    return [dict(lane="demo (HG002, 2M pairs)", mode="stream",
                 **{k: v for k, v in summ.items() if isinstance(v, (int, float)) and k.startswith(("seqs", "bp", "seconds", "time"))})]


def allele_counts(bam, v, min_mapq=20, min_bq=20):
    """Per-strand and per-lane read counts supporting REF / ALT at an SNV (fragment-level: one count per read pair)."""
    import pysam
    out = dict(ref_fwd=0, ref_rev=0, alt_fwd=0, alt_rev=0, other=0, lanes=collections.Counter())
    seen = set()
    pos0 = int(v["pos"]) - 1
    with pysam.AlignmentFile(str(bam)) as f:
        for col in f.pileup(v["chrom"], pos0, pos0 + 1, truncate=True, min_mapping_quality=min_mapq, min_base_quality=min_bq,
                            stepper="nofilter", ignore_orphans=False):
            for pr in col.pileups:
                a = pr.alignment
                if pr.is_del or pr.is_refskip or a.is_duplicate or a.is_secondary or a.is_supplementary or a.query_name in seen:
                    continue
                seen.add(a.query_name)
                b = a.query_sequence[pr.query_position]
                strand = "rev" if a.is_reverse else "fwd"
                if b == v["ref"]:
                    out[f"ref_{strand}"] += 1
                elif b == v["alt"]:
                    out[f"alt_{strand}"] += 1
                    parts = a.query_name.split(":")
                    out["lanes"][parts[3] if len(parts) > 4 else "?"] += 1
                else:
                    out["other"] += 1
    out["lanes"] = dict(out["lanes"])
    return out


def fragment_links(bam, sites, min_mapq=20):
    """Number of read pairs that cover both members of each consecutive pair of heterozygous SNV sites."""
    import pysam
    cover = collections.defaultdict(set)
    with pysam.AlignmentFile(str(bam)) as f:
        for p in sites:
            for a in f.fetch(sites[0][0], p[1] - 1, p[1]):
                if a.mapping_quality >= min_mapq and not a.is_duplicate and not a.is_secondary and not a.is_supplementary:
                    if a.reference_start <= p[1] - 1 < a.reference_end:
                        cover[p[1]].add(a.query_name)
    return [dict(from_pos=a[1], to_pos=b[1], gap_bp=b[1] - a[1], linking_pairs=len(cover[a[1]] & cover[b[1]]))
            for a, b in zip(sites, sites[1:])]


def sv_screen(bam, chrom, start, end, min_mapq=20):
    import pysam
    clips, disc = collections.Counter(), collections.Counter()
    with pysam.AlignmentFile(str(bam)) as f:
        for a in f.fetch(chrom, start, end):
            if a.mapping_quality < min_mapq or a.is_duplicate or a.is_secondary or a.is_supplementary:
                continue
            cig = a.cigartuples or []
            if cig and cig[0][0] == 4 and cig[0][1] >= 20:
                clips[a.reference_start // 10 * 10] += 1
            if cig and cig[-1][0] == 4 and cig[-1][1] >= 20:
                clips[a.reference_end // 10 * 10] += 1
            if a.is_paired and not a.mate_is_unmapped and (a.reference_name != a.next_reference_name or abs(a.template_length) > 2000):
                disc[a.reference_start // 500 * 500] += 1
    return ([dict(pos=p, clipped_reads=n) for p, n in sorted(clips.items()) if n >= 5],
            [dict(bin_start=p, discordant_pairs=n) for p, n in sorted(disc.items()) if n >= 4])


with stage("read-level evidence from the raw FASTQ"):
    if not CFG["RUN_READ_LEVEL"]:
        raise RuntimeError("disabled in CFG")
    import pysam
    from pyfaidx import Fasta
    for d_ in (READS, READS / "fq", READS / "rec"):
        d_.mkdir(parents=True, exist_ok=True)
    # free scratch space: the genome-wide frequency archive is no longer needed
    for k in ("af_data",):
        if k in RES and Path(RES[k]).exists():
            Path(RES[k]).unlink()
    TOPH = HYPS[0]
    genes = list(dict.fromkeys([TOPH["gene"]] + [h["gene"] for h in HYPS[:20]]))[:3]
    deacon = ngs_env()
    fa = Fasta(RES["ref"], as_raw=True, sequence_always_upper=True)
    bait = make_bait(genes, fa)
    with open(READS / "bait.fa", "w") as f:
        for c, s, e, tag in bait:
            f.write(f">{tag}_{c}_{s}\n{str(fa[c][s - 1:e])}\n")
    pd.DataFrame(bait, columns=["chrom", "start", "end", "tag"]).to_csv(READS / "bait.tsv", sep="\t", index=False)
    write_masked_genome_script(bait, READS / "mask.py")
    sh(f"'{deacon}' index build '{READS / 'bait.fa'}' > '{READS / 'bait.idx'}'", name="deacon_index")
    sh(f"'{sys.executable}' '{READS / 'mask.py'}' | '{deacon}' index diff '{READS / 'bait.idx'}' - > '{READS / 'bait.uniq.idx'}'",
       name="deacon_diff", timeout=3600)
    info = sh(f"'{deacon}' index info '{READS / 'bait.uniq.idx'}' 2>&1 || true", name="deacon_info", capture=True)
    log(f"bait: {len(bait)} regions, {sum(e - s + 1 for _, s, e, _ in bait) / 1e6:.2f} Mb; unique-minimizer index: {info.strip()[:200]}")
    idx = READS / "bait.uniq.idx"
    lanes_done = []
    if DEMO:
        lanes_done = demo_reads(deacon, idx, READS / "rec")
    else:
        sizes = fastq_sizes()
        for lane in CFG["FASTQ_LANES"]:
            if elapsed_h() > CFG["READ_LEVEL_MAX_HOURS"]:
                log(f"time budget reached ({elapsed_h():.1f} h): no further lanes")
                break
            try:
                r = recruit_lane(deacon, idx, lane, sizes, READS / "fq", READS / "rec")
                lanes_done.append(r)
                log(f"lane {lane}: {r}")
            except Exception as e:      # noqa: BLE001
                log(f"lane {lane} failed ({str(e)[:200]}); continuing")
                for p in (READS / "fq").glob("*"):
                    p.unlink() if p.is_file() else shutil.rmtree(p, ignore_errors=True)
    if not list((READS / "rec").glob("L*.R1.fq.gz")):
        raise RuntimeError("no lane could be recruited")
    # ---- align the recruited reads to the whole genome --------------------------------------------------------------
    bam = READS / "mini.bam"
    sh(f"cat {READS / 'rec'}/L*.R1.fq.gz > {READS / 'all.R1.fq.gz'} && cat {READS / 'rec'}/L*.R2.fq.gz > {READS / 'all.R2.fq.gz'} && "
       f"minimap2 -ax sr -t {NCPU} -R '@RG\\tID:proband\\tSM:proband\\tPL:ILLUMINA' '{RES['ref']}' {READS / 'all.R1.fq.gz'} {READS / 'all.R2.fq.gz'} "
       f"| samtools fixmate -m -u - - | samtools sort -m 1500M -@ 2 -T '{WORK / 'tmp' / 'srt'}' -u - "
       f"| samtools markdup -@ 2 -d 2500 - '{bam}' && samtools index '{bam}'", name="align", timeout=3 * 3600)
    flag = sh(f"samtools flagstat '{bam}'", name="flagstat", capture=True)
    n_reads = int(flag.split()[0])
    # ---- depth: bait targets vs control windows ---------------------------------------------------------------------
    bed = READS / "bins.bed"
    rows = []
    for c, s, e, tag in bait:
        if tag.startswith("target"):
            for b in range(s, e, 500):
                rows.append((c, b - 1, min(b + 499, e), tag))
        else:
            rows.append((c, s - 1, e, tag))
    pd.DataFrame(rows).to_csv(bed, sep="\t", header=False, index=False)
    cov = pd.read_csv(io.StringIO(sh(f"samtools bedcov -Q 20 -j '{bed}' '{bam}'", name="bedcov", capture=True)), sep="\t", header=None,
                      names=["chrom", "start", "end", "tag", "sum"])
    cov["depth"] = cov["sum"] / (cov.end - cov.start)
    auto_med = float(cov[cov.tag == "ctrl_auto"].depth.median()) or float("nan")
    x_ratio = float(cov[cov.tag == "ctrl_X"].depth.median()) / auto_med if auto_med == auto_med and auto_med > 0 else float("nan")
    tg = cov[cov.tag == f"target_{TOPH['gene']}"].copy()
    tg["ratio"] = tg.depth / auto_med
    gs, ge = GENE_POS[TOPH["gene"]][1], GENE_POS[TOPH["gene"]][2]
    in_gene = tg[(tg.end > gs) & (tg.start < ge)]
    low = in_gene[in_gene.ratio < 0.65]
    runs = (low.start.diff().fillna(500) > 500).cumsum()
    del_runs = [dict(start=int(g.start.min()), end=int(g.end.max()), bins=len(g), mean_ratio=round(float(g.ratio.mean()), 2))
                for _, g in low.groupby(runs) if len(g) >= 3]
    clip_cl, disc_cl = sv_screen(bam, GENE_POS[TOPH["gene"]][0], gs, ge)
    # ---- leading alleles: independent counts and re-calling -----------------------------------------------------------
    allele_rows = []
    for v in TOPH["variants"]:
        if len(v["ref"]) == 1 and len(v["alt"]) == 1:
            ac = allele_counts(bam, v)
            n_ref, n_alt = ac["ref_fwd"] + ac["ref_rev"], ac["alt_fwd"] + ac["alt_rev"]
            from scipy.stats import binomtest
            allele_rows.append(dict(variant=describe_variant(v), ref_reads=n_ref, alt_reads=n_alt,
                                    alt_fraction=round(n_alt / max(n_ref + n_alt, 1), 3),
                                    alt_strands=f"+{ac['alt_fwd']}/-{ac['alt_rev']}", alt_by_lane=ac["lanes"],
                                    het_binom_p=round(binomtest(n_alt, max(n_ref + n_alt, 1), 0.5).pvalue, 3) if n_ref + n_alt else None,
                                    vcf_AD="/".join(fmt(int(x)) if isinstance(x, (int, float)) and x == x else fmt(x) for x in (v.get('ad_ref'), v.get('ad_alt')))))
    tgt_bed = READS / "targets.bed"
    pd.DataFrame([(c, s - 1, e) for c, s, e, t in bait if t.startswith("target")]).to_csv(tgt_bed, sep="\t", header=False, index=False)
    sh(f"bcftools mpileup -f '{RES['ref']}' -R '{tgt_bed}' -q 20 -Q 20 -a AD,DP -d 10000 -Ou '{bam}' "
       f"| bcftools call -mv -Oz -o '{READS / 'recall.vcf.gz'}' && bcftools index -f '{READS / 'recall.vcf.gz'}'", name="recall")
    recall = {(c[0], int(c[1]), c[2], c[3]): c[4] for c in
              (l.split("\t") for l in sh(f"bcftools query -f '%CHROM\\t%POS\\t%REF\\t%ALT[\\t%GT]\\n' '{READS / 'recall.vcf.gz'}'",
                                          name="recall_q", capture=True).splitlines() if l)}
    _names = {l.split("\t")[0] for l in sh(f"bcftools index -s '{S['vcf_raw']}'", name="raw_idx", capture=True).splitlines() if l.strip()}
    raw_chr = "chr1" in _names or "chrX" in _names
    raw_bed = READS / "targets.raw.bed"
    pd.DataFrame([(c if raw_chr else c[3:], s - 1, e) for c, s, e, t in bait if t.startswith("target")]).to_csv(
        raw_bed, sep="\t", header=False, index=False)
    orig = {(("" if raw_chr else "chr") + c[0], int(c[1]), c[2], c[3]): c[4] for c in
            (l.split("\t") for l in sh(f"bcftools query -R '{raw_bed}' -i 'GT=\"het\" && FILTER=\"PASS\" && TYPE=\"snp\"' "
                                        f"-f '%CHROM\\t%POS\\t%REF\\t%ALT[\\t%GT]\\n' '{S['vcf_raw']}'", name="orig_q",
                                        capture=True).splitlines() if l)}
    conc = sum(1 for k in orig if k in recall) / max(len(orig), 1)
    lead_recalled = {describe_variant(v): (vkey(v) in recall) for v in TOPH["variants"]}
    # ---- empirical phaseability between the two leading alleles ------------------------------------------------------
    links = []
    if TOPH["model"] == "compound_het":
        a, b = sorted(TOPH["variants"], key=lambda v: int(v["pos"]))
        between = sorted({(k[0], k[1]) for k, gt in recall.items() if k[0] == a["chrom"] and int(a["pos"]) <= k[1] <= int(b["pos"])
                          and gt.replace("|", "/") in ("0/1", "1/0") and len(k[2]) == 1 and len(k[3]) == 1}
                         | {(a["chrom"], int(a["pos"])), (b["chrom"], int(b["pos"]))}, key=lambda x: x[1])
        links = fragment_links(bam, between)
    S["reads"] = dict(lanes=lanes_done, mapped_reads=n_reads, autosomal_control_depth=round(auto_med, 1), chrX_to_autosome_depth=round(x_ratio, 3),
                      leading_alleles=allele_rows, recalled_leading=lead_recalled, vcf_het_concordance=round(conc, 3), n_vcf_hets_in_bait=len(orig),
                      low_depth_runs_in_gene=del_runs, clip_clusters=clip_cl[:10], discordant_clusters=disc_cl[:10], phase_links=links,
                      demo=DEMO)
    broken = [l for l in links if l["linking_pairs"] == 0]
    S["reads"]["summary"] = (
        ("*Demo mode: toolchain test on 2 M public read pairs; coverage too low for interpretation.* " if DEMO else "")
        + f"{len(lanes_done)} lane(s) streamed through unique-minimizer recruitment; {n_reads:,} recruited reads re-aligned genome-wide. "
        f"Control-window depth {auto_med:.1f}× on autosomes; chrX/autosome depth {x_ratio:.2f} (expected 0.5 in a male, which validates "
        f"the depth method for one-copy changes). "
        + " ".join(f"{r['variant']}: {r['alt_reads']}/{r['ref_reads'] + r['alt_reads']} alt reads (fraction {r['alt_fraction']}; strands "
                   f"{r['alt_strands']}; lanes {r['alt_by_lane']}; original VCF AD {r['vcf_AD']})." for r in allele_rows)
        + f" Independent re-calling (bcftools) recovered {100 * conc:.0f} % of the {len(orig)} PASS heterozygous SNVs of the provided VCF in the "
          f"targets, and the leading alleles: {lead_recalled}. Depth along {TOPH['gene']}: "
        + (f"{len(del_runs)} run(s) of ≥ 3 bins below 0.65× ({del_runs})" if del_runs else "no run of low-depth bins (no exon-scale deletion)")
        + f"; clipped-read clusters: {len(clip_cl)}; discordant-pair clusters: {len(disc_cl)}. "
        + (f"Between the two alleles the independent re-call finds {len(links) - 1} heterozygous SNV site(s) (the provided VCF lists "
           f"{fmt((S.get('phase') or {}).get('n_het_sites_between'))}); {len(broken)} of the {len(links)} consecutive site pairs are linked by no read pair "
           f"(largest gap {max((l['gap_bp'] for l in links), default=0):,} bp)"
           + (": the reads cannot phase the alleles." if broken else ": every consecutive pair is linked, so read-level chaining is possible.")
           if links else ""))
    log(S["reads"]["summary"])
    if allele_rows:
        md("**Leading alleles in the re-aligned reads**\n\n" + df_md(pd.DataFrame(allele_rows)))
    fig, ax = plt.subplots(figsize=(9, 2.8))
    ax.scatter((tg.start + 250) / 1e6, tg.ratio, s=8, color=VIZ["s1"], linewidths=0, label=f"{TOPH['gene']} locus, 500-bp bins")
    ax.axhline(1.0, color=VIZ["axis"], linewidth=0.8)
    ax.axhline(0.5, color=VIZ["s2"], linewidth=1.2, label=f"one-copy level (chrX control measured {x_ratio:.2f})")
    for v in TOPH["variants"]:
        ax.axvline(int(v["pos"]) / 1e6, color=VIZ["ink2"], linewidth=0.8)
    ax.axvspan(gs / 1e6, ge / 1e6, color=VIZ["grid"], alpha=0.5, linewidth=0)
    ax.set_ylim(0, max(2.0, float(tg.ratio.quantile(0.99)) if len(tg) else 2.0))
    ax.set_xlabel(f"{GENE_POS[TOPH['gene']][0]} position (Mb); shaded = gene body; vertical lines = leading alleles")
    ax.set_ylabel("depth / autosomal control")
    ax.set_title(f"Read depth across the {TOPH['gene']} locus from recruited reads")
    ax.legend(loc="upper right", ncol=2)
    save_fig(fig, "locus_depth")

## 17 · Final reports, data deletion and run log

The reports are regenerated with every result of the run. All scratch storage that held patient data (VCF, reads,
annotated variant tables, the re-aligned reads) is deleted; `/kaggle/working` keeps findings only.

In [ ]:
with stage("final reports"):
    log(f"reports written: {write_reports()}")

with stage("delete scratch data"):
    if DEMO:
        log("demo mode: scratch holds public data only; deleting anyway")
    os.chdir(OUT)
    shutil.rmtree(WORK, ignore_errors=True)
    S["scratch_deleted"] = not WORK.exists()
    log(f"scratch directory deleted: {S['scratch_deleted']}")

print("\n" + "=" * 100)
print(f"Mode: {S.get('mode')} · total run time {(time.time() - T0) / 60:.0f} min")
print("=" * 100)
for k, v in RUN.items():
    print(f"{v['status']:>8}  {v['seconds'] / 60:6.1f} min  {k}" + (f"  — {v['note'][:120]}" if v["note"] else ""))
print("\nFiles in the output directory (download these for the submission):")
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(OUT)}  ({p.stat().st_size / 1024:.0f} KB)")
if SUBMISSION_CSV.exists():
    print(f"\nTrack 1 submission file: {SUBMISSION_CSV.name}")
    print(SUBMISSION_CSV.read_text())
if DEMO:
    print("DEMO run finished. To analyse the challenge genome: Add-ons → Secrets → add HF_TOKEN (a token of a Hugging Face account "
          "that has been granted access to the gated dataset), tick it for this notebook, then Save Version → Save & Run All.")